install


xd

In [1]:
# ============================================================
# CELL 1 — Environment  [v2]
# Colab (Python 3.13) already ships numpy, pandas, scipy, sklearn,
# networkx, pyarrow, openpyxl, tqdm. Do NOT reinstall or pin them —
# the old pins had no 3.13 wheels and forced slow source builds.
# Only three packages are genuinely missing.
#
# >>> DO NOT CANCEL. inmoose may take 2-3 minutes to build. <
# ============================================================
import sys
print(f"Python {sys.version.split()[0]}\n")

!pip install GEOparse
!pip install mygene
!pip install inmoose

import importlib
NEED = {"GEOparse": "GEOparse", "mygene": "mygene", "inmoose": "inmoose",
        "networkx": "networkx", "pyarrow": "pyarrow", "openpyxl": "openpyxl",
        "tqdm": "tqdm", "sklearn": "scikit-learn", "scipy": "scipy",
        "numpy": "numpy", "pandas": "pandas"}

print("\n" + "="*50)
missing = []
for mod, pkg in NEED.items():
    try:
        m = importlib.import_module(mod)
        print(f"  OK   {mod:12s} {getattr(m, '__version__', '?')}")
    except Exception as e:
        print(f"  FAIL {mod:12s} {type(e).__name__}")
        missing.append(pkg)

if missing:
    raise RuntimeError(
        f"MISSING: {missing}\n"
        f"Run: !pip install {' '.join(missing)}\n"
        f"Then restart the runtime (Runtime > Restart session) and re-run "
        f"Cell 1 before anything else. Do NOT cancel the install.")
print("\nAll imports OK — proceed to Cell 2.")

Python 3.13.15

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.1/52.1 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 61.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.2/190.2 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 127.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.9/193.9 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.9/376.9 kB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 89.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4

In [3]:
# ============================================================
# CELL 2 — Imports, Drive, Paths, Config
# ============================================================
import os, re, json, gzip, time, hashlib, warnings, tarfile, shutil
from pathlib import Path
from collections import namedtuple, defaultdict, Counter

import numpy as np
import pandas as pd
import networkx as nx
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# ---- Single source of truth for every path ------------------

ROOT = Path("/content/drive/MyDrive/CoDAGAT_preprocessing_fix_v1")

PATHS = {
    "root":      ROOT,
    "raw":       ROOT / "raw",           # untouched downloads
    "raw_geo":   ROOT / "raw" / "geo",
    "raw_gtex":  ROOT / "raw" / "gtex",
    "raw_string":ROOT / "raw" / "string",
    "raw_nano":  ROOT / "raw" / "nanostring",
    "interim":   ROOT / "interim",       # stage outputs
    "processed": ROOT / "processed",     # final artifacts
    "reports":   ROOT / "reports",       # audit + integrity reports
    "config":    ROOT / "config",
    "manifest":  ROOT / "manifests",
}
for p in PATHS.values():
    p.mkdir(parents=True, exist_ok=True)

# ---- Global constants (locked by spec) ----------------------
RANDOM_SEED       = 42
SEEDS             = [42, 43, 44, 45, 46]
STRING_THRESHOLD  = 700
N_PANEL_GENES     = 72
GTEX_MIN_SD       = 0.1

np.random.seed(RANDOM_SEED)

# ---- Manifest helper: makes every stage idempotent ----------
def sha256_file(path, blocksize=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(blocksize), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_obj(obj):
    return hashlib.sha256(
        json.dumps(obj, sort_keys=True, default=str).encode()
    ).hexdigest()

def write_manifest(stage, inputs, params, outputs):
    """Record what produced what, so re-runs can be skipped and
    so every number in the paper is traceable to a config."""
    man = {
        "stage": stage,
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
        "inputs": inputs,
        "params": params,
        "outputs": {
            str(o): (sha256_file(o) if Path(o).exists() else None)
            for o in outputs
        },
    }
    fp = PATHS["manifest"] / f"manifest_{stage}.json"
    fp.write_text(json.dumps(man, indent=2))
    print(f"[manifest] wrote {fp.name}")
    return man

def stage_is_current(stage, outputs):
    """True if all outputs exist and manifest hashes still match."""
    fp = PATHS["manifest"] / f"manifest_{stage}.json"
    if not fp.exists():
        return False
    man = json.loads(fp.read_text())
    for o in outputs:
        o = Path(o)
        if not o.exists():
            return False
        if man["outputs"].get(str(o)) != sha256_file(o):
            return False
    return True

print(f"\nROOT = {ROOT}")
print(f"Free space: {shutil.disk_usage('/content').free / 1e9:.1f} GB on /content")
for k, v in PATHS.items():
    v.mkdir(parents=True, exist_ok=True)
    assert v.exists(), f"could not create {k}: {v} — is Drive mounted?"
print("all PATHS verified:", {k: str(v) for k, v in PATHS.items()})


Mounted at /content/drive

ROOT = /content/drive/MyDrive/CoDAGAT_preprocessing_fix_v1
Free space: 75.1 GB on /content
all PATHS verified: {'root': '/content/drive/MyDrive/CoDAGAT_preprocessing_fix_v1', 'raw': '/content/drive/MyDrive/CoDAGAT_preprocessing_fix_v1/raw', 'raw_geo': '/content/drive/MyDrive/CoDAGAT_preprocessing_fix_v1/raw/geo', 'raw_gtex': '/content/drive/MyDrive/CoDAGAT_preprocessing_fix_v1/raw/gtex', 'raw_string': '/content/drive/MyDrive/CoDAGAT_preprocessing_fix_v1/raw/string', 'raw_nano': '/content/drive/MyDrive/CoDAGAT_preprocessing_fix_v1/raw/nanostring', 'interim': '/content/drive/MyDrive/CoDAGAT_preprocessing_fix_v1/interim', 'processed': '/content/drive/MyDrive/CoDAGAT_preprocessing_fix_v1/processed', 'reports': '/content/drive/MyDrive/CoDAGAT_preprocessing_fix_v1/reports', 'config': '/content/drive/MyDrive/CoDAGAT_preprocessing_fix_v1/config', 'manifest': '/content/drive/MyDrive/CoDAGAT_preprocessing_fix_v1/manifests'}


In [4]:
# ============================================================
# CELL 3 — COHORT REGISTRY  (v2, post-tracker audit)
#   T1 = LODO primary   (train / calibrate / TEST)
#   T2 = training-only  (train only)
#   T3 = sensitivity    (train only, reported separately)
#   T4 = external       (never enters pipeline)
#   AUDIT / EXCLUDED / REJECTED = logged, not usable
# ============================================================

COHORTS = {
    # ---------------- CONFIRMED T1 ----------------
    "GSE135251": dict(tier="T1", platform="rnaseq", etiology="NAFLD", repo="GEO",
                      preservation="frozen", n_expected=206, status="confirmed"),
    "GSE84044":  dict(tier="T1", platform="microarray", etiology="HBV", gpl="GPL570", repo="GEO",
                      preservation="frozen", n_expected=124, status="confirmed"),
    "GSE162694": dict(tier="T1", platform="rnaseq", etiology="NAFLD", repo="GEO",
                      preservation="frozen", n_expected=112, status="confirmed"),
    "GSE130970": dict(tier="T1", platform="rnaseq", etiology="NAFLD", repo="GEO",
                      preservation="frozen", n_expected=78, status="confirmed"),
    "E-MTAB-6863": dict(tier="T1", platform="rnaseq", etiology="HCV_NASH_mixed",
                      repo="ArrayExpress", preservation="frozen", n_expected=69,
                      status="new_accepted",
                      note="F0=12 F1=15 F2=15 F3=14 F4=13. Exact staging. "
                           "Best F2 density available; adds HCV etiology."),
    "GSE89632":  dict(tier="T1", platform="microarray", etiology="NAFLD", gpl="GPL14951",
                      repo="GEO", preservation="frozen", n_expected=55, status="confirmed",
                      note="marginal: 10 positives under F2+"),
    "GSE48452":  dict(tier="T1", platform="microarray", etiology="NAFLD", gpl="GPL11532",
                      repo="GEO", preservation="frozen", n_expected=60, status="confirmed",
                      note="NOT viable as T1 for F2+ (4 positives); T1 for early-vs-adv only"),
    "GSE31803":  dict(tier="T1", platform="microarray", etiology="NAFLD", gpl="GPL570",
                      repo="GEO", preservation="frozen", n_expected=72, status="confirmed",
                      note="BOUNDARY-DEGENERATE: grouped F0-1 vs F3-4, no F1/F2 present"),

    # ---------------- T2 training-only ----------------
    "GSE276114": dict(tier="T2", platform="rnaseq", etiology="MASLD_mixed", repo="GEO",
                      preservation="frozen", n_expected=177, status="confirmed",
                      note="39 samples grouped_F0_F2 -> ambiguous across F2 boundary"),
    "GSE240729": dict(tier="T2", platform="rnaseq", etiology="MASLD", repo="GEO",
                      preservation="FFPE", n_expected=67, status="confirmed"),
    "E-MTAB-9815": dict(tier="T2", platform="rnaseq", etiology="NAFLD",
                      repo="ArrayExpress", preservation="frozen", n_expected=58,
                      status="new_accepted",
                      note="Groups: NAFL / NASH F0-2 / NASH F3-4. F0-2 bin straddles "
                           "the F2 boundary -> grouped_F0_F2, training only."),

    # ---------------- T3 sensitivity ----------------
    "GSE213621": dict(tier="T3", platform="rnaseq", etiology="MASLD", repo="GEO",
                      preservation="frozen", n_expected=368, status="RELABEL_REQUIRED",
                      note="CRITICAL: Phase 2 recovered only 107 F2. Published work using "
                           "this series reports n=368 with F0-F2 vs F3-F4 labels. "
                           "Staging exists under a metadata key the parser missed. "
                           "Recover before tiering: may promote to T1."),

    # ---------------- T4 external ----------------
    "GSE163211": dict(tier="T4", platform="nanostring", etiology="NAFLD_spectrum", repo="GEO",
                      preservation="frozen", n_expected=318, status="confirmed",
                      note="EXTERNAL ONLY. Coarse labels. Cannot validate F2+ boundary."),

    # ---------------- AUDIT ----------------
    "GSE193066": dict(tier="AUDIT", platform="rnaseq", etiology="NAFLD", repo="GEO",
                      n_expected=164, status="candidate",
                      note="106 baseline + 58 follow-up biopsies. LONGITUDINAL DUPLICATES: "
                           "keep one biopsy per patient. Staging unverified."),
    "GSE159088": dict(tier="AUDIT", platform="microarray", etiology="NAFLD", repo="GEO",
                      n_expected=80, status="candidate",
                      note="Fibrosis staging not visible in public sample fields. Low priority."),
    "GSE207310": dict(tier="AUDIT", platform="rnaseq", etiology="NAFLD", repo="GEO",
                      n_expected=30, status="candidate",
                      note="Kleiner grades in metadata; Phase 2 TAR extraction empty. "
                           "Retry per-GSM table fallback."),

    # ---------------- REJECTED (logged, never used) ----------------
    "GSE49541":  dict(tier="REJECTED", platform="microarray", gpl="GPL570", repo="GEO",
                      n_expected=72, status="rejected",
                      reason="SUSPECTED DUPLICATE of GSE31803: both 72 samples, GPL570, "
                             "NAFLD, 40 mild(F0-1) / 32 advanced(F3-4). Adding it would "
                             "place identical patients in train and test. "
                             "Verify by GSM overlap before any reconsideration."),
    "GSE167523": dict(tier="REJECTED", platform="rnaseq", etiology="NAFLD", repo="GEO",
                      n_expected=98, status="rejected",
                      reason="No F0-F4 staging. 48 mild NAFLD / 50 NASH = disease category, "
                             "not fibrosis stage. Stage must never be inferred from NAFL/NASH."),
    "GSE126848": dict(tier="REJECTED", repo="GEO", status="rejected",
                      reason="No fibrosis staging"),
    "GSE61260":  dict(tier="REJECTED", repo="GEO", status="rejected",
                      reason="0 staged samples recovered in Phase 2"),
    "GSE151158": dict(tier="REJECTED", platform="nanostring", repo="GEO", status="rejected",
                      reason="NASH activity panel, not F0-F4 staging"),
    "GSE139602": dict(tier="REJECTED", platform="rnaseq", repo="GEO", status="rejected",
                      reason="Cirrhosis/ACLF focused; no early-stage distribution"),
    "GSE202379": dict(tier="REJECTED", repo="GEO", status="rejected",
                      reason="single-nucleus RNA-seq; bulk-only criterion"),
    "GSE270584": dict(tier="REJECTED", repo="GEO", status="rejected",
                      reason="spatial/single-cell, non-human signal"),
    "GSE115193": dict(tier="REJECTED", repo="GEO", n_expected=3, status="rejected",
                      reason="only 3 samples"),
    "E-MTAB-12445": dict(tier="REJECTED", repo="ArrayExpress", status="rejected",
                      reason="NAFLD vs healthy; no fibrosis staging"),
    "GSE134146": dict(tier="REJECTED", repo="GEO", status="rejected",
                      reason="circRNA; not bulk staged transcriptomics"),
    "GSE94799":  dict(tier="REJECTED", repo="GEO", status="rejected",
                      reason="no confirmed fibrosis-stage labels"),
}

TIER_RULES = {          # can_train, can_calibrate, can_test
    "T1": (True, True, True),   "T2": (True, False, False),
    "T3": (True, False, False), "T4": (False, False, False),
    "AUDIT": (False, False, False),
    "REJECTED": (False, False, False),
}

REPO_OF = {g: m.get("repo", "GEO") for g, m in COHORTS.items()}

def tier_of(gse):            return COHORTS[gse]["tier"]
def cohorts_by_tier(*tiers): return [g for g, m in COHORTS.items() if m["tier"] in tiers]

(PATHS["config"] / "cohorts.json").write_text(json.dumps(COHORTS, indent=2))

print("REGISTRY v2")
for t in ["T1", "T2", "T3", "T4", "AUDIT", "REJECTED"]:
    ms = cohorts_by_tier(t)
    print(f"  {t:9s} n={len(ms):2d}  {ms}")

usable = [g for g in COHORTS if tier_of(g) in ("T1", "T2", "T3")]
print(f"\nProjected N_real (T1+T2+T3): "
      f"{sum(COHORTS[g].get('n_expected', 0) for g in usable)}")
print("  (assumes GSE213621 relabel recovers 368; if it stays 107, subtract 261)")
print(f"\nRepositories to handle: {sorted(set(REPO_OF[g] for g in usable))}")
print("!! Cell 5 needs an ArrayExpress fetcher for E-MTAB-6863 / E-MTAB-9815")

REGISTRY v2
  T1        n= 8  ['GSE135251', 'GSE84044', 'GSE162694', 'GSE130970', 'E-MTAB-6863', 'GSE89632', 'GSE48452', 'GSE31803']
  T2        n= 3  ['GSE276114', 'GSE240729', 'E-MTAB-9815']
  T3        n= 1  ['GSE213621']
  T4        n= 1  ['GSE163211']
  AUDIT     n= 3  ['GSE193066', 'GSE159088', 'GSE207310']
  REJECTED  n=12  ['GSE49541', 'GSE167523', 'GSE126848', 'GSE61260', 'GSE151158', 'GSE139602', 'GSE202379', 'GSE270584', 'GSE115193', 'E-MTAB-12445', 'GSE134146', 'GSE94799']

Projected N_real (T1+T2+T3): 1446
  (assumes GSE213621 relabel recovers 368; if it stays 107, subtract 261)

Repositories to handle: ['ArrayExpress', 'GEO']
!! Cell 5 needs an ArrayExpress fetcher for E-MTAB-6863 / E-MTAB-9815


In [5]:
# ============================================================
# CELL 4 — GUARDS
# The three non-negotiable rules, as assertions. No bypass flag.
# ============================================================

class GuardViolation(AssertionError):
    pass

SYN_PREFIX = "SYN_"

def assert_no_synthetic(df, split_name, fold_id="NA"):
    """RULE 2: calibration and test must be raw."""
    idx = pd.Index(df.index if df.index.name in (None, "sample_id") else df["sample_id"])
    bad = [s for s in idx.astype(str) if s.startswith(SYN_PREFIX)]
    if bad:
        raise GuardViolation(
            f"FOLD {fold_id}: {len(bad)} synthetic samples found in '{split_name}'. "
            f"Examples: {bad[:5]}. Augmentation must run AFTER the "
            f"proper-train/calibration split, on proper-train only."
        )
    return True

def assert_cohort_disjoint(train_df, cal_df, test_df, fold_id="NA"):
    """RULE 1: test cohort appears nowhere else."""
    tr, ca, te = set(train_df.cohort), set(cal_df.cohort), set(test_df.cohort)
    if len(te) != 1:
        raise GuardViolation(f"FOLD {fold_id}: test must be exactly ONE cohort, got {te}")
    if te & tr:
        raise GuardViolation(f"FOLD {fold_id}: test cohort {te & tr} also in proper-train")
    if te & ca:
        raise GuardViolation(f"FOLD {fold_id}: test cohort {te & ca} also in calibration")
    return True

def assert_no_subject_overlap(train_df, test_df, fold_id="NA"):
    """Silent leakage: GEO super-series and re-analyses share patients."""
    if "subject_id" in train_df.columns and "subject_id" in test_df.columns:
        a = set(train_df.subject_id.dropna())
        b = set(test_df.subject_id.dropna())
        if a & b:
            raise GuardViolation(
                f"FOLD {fold_id}: {len(a & b)} subjects appear in both train and test: "
                f"{list(a & b)[:5]}")
    return True

def assert_synthetic_provenance(syn_df):
    """RULE 3: synthesis is within-cohort and within-class."""
    req = {"source_cohort_a", "source_cohort_b", "source_stage_a", "source_stage_b",
           "cohort", "stage", "method"}
    missing = req - set(syn_df.columns)
    if missing:
        raise GuardViolation(f"synthetic frame missing provenance columns: {missing}")
    bad_c = syn_df[(syn_df.cohort != syn_df.source_cohort_a) |
                   (syn_df.cohort != syn_df.source_cohort_b)]
    if len(bad_c):
        raise GuardViolation(f"{len(bad_c)} synthetic samples cross cohort boundaries")
    bad_s = syn_df[(syn_df.stage != syn_df.source_stage_a) |
                   (syn_df.stage != syn_df.source_stage_b)]
    if len(bad_s):
        raise GuardViolation(f"{len(bad_s)} synthetic samples cross class boundaries")
    return True

def assert_tier_usage(df, split_name, fold_id="NA"):
    """T2/T3 never calibrate or test. T4/AUDIT/EXCLUDED never enter at all."""
    slot = {"proper_train": 0, "calibration": 1, "test": 2}[split_name]
    for coh in set(df.cohort):
        allowed = TIER_RULES[tier_of(coh)][slot]
        if not allowed:
            raise GuardViolation(
                f"FOLD {fold_id}: cohort {coh} (tier {tier_of(coh)}) "
                f"is not permitted in split '{split_name}'")
    return True

def assert_no_healthy_as_F0(labels_df):
    """Healthy liver is not stage F0. Conflating them leaks the deviation signal."""
    if "resolution" not in labels_df.columns:
        raise GuardViolation("labels frame has no 'resolution' column")
    bad = labels_df[(labels_df.resolution == "healthy") & (labels_df.stage.notna())]
    if len(bad):
        raise GuardViolation(f"{len(bad)} healthy samples were assigned a fibrosis stage")
    return True

def assert_tensor_integrity(arr, name="tensor"):
    if np.isnan(arr).any():
        raise GuardViolation(f"{name}: {int(np.isnan(arr).sum())} NaN values")
    if np.isinf(arr).any():
        raise GuardViolation(f"{name}: {int(np.isinf(arr).sum())} Inf values")
    return True


# ---------------- self-test ----------------
def _test_guards():
    ok = 0
    tr = pd.DataFrame({"cohort": ["GSE135251"]*3}, index=["a","b","c"])
    ca = pd.DataFrame({"cohort": ["GSE84044"]*2},  index=["d","e"])
    te = pd.DataFrame({"cohort": ["GSE130970"]*2}, index=["f","g"])
    assert assert_cohort_disjoint(tr, ca, te); ok += 1

    bad_cal = pd.DataFrame({"cohort": ["GSE84044"]*2}, index=["SYN_1","e"])
    try:
        assert_no_synthetic(bad_cal, "calibration"); print("  FAIL: synthetic not caught")
    except GuardViolation:
        ok += 1

    leaky = pd.DataFrame({"cohort": ["GSE130970"]*2}, index=["x","y"])
    try:
        assert_cohort_disjoint(leaky, ca, te); print("  FAIL: overlap not caught")
    except GuardViolation:
        ok += 1

    try:
        assert_tier_usage(pd.DataFrame({"cohort": ["GSE240729"]}), "test")
        print("  FAIL: T2-as-test not caught")
    except GuardViolation:
        ok += 1

    try:
        assert_tensor_integrity(np.array([1.0, np.nan])); print("  FAIL: NaN not caught")
    except GuardViolation:
        ok += 1

    print(f"guard self-test: {ok}/5 passed")

_test_guards()
def check_expression(df, name):
    if not isinstance(df, pd.DataFrame) or df.empty:
        raise ValueError(f"{name}: missing or empty matrix")

    if not df.index.is_unique or not df.columns.is_unique:
        raise ValueError(f"{name}: duplicate gene/sample IDs")

    if df.index.hasnans or df.columns.hasnans:
        raise ValueError(f"{name}: missing gene/sample IDs")

    if not np.isfinite(df.to_numpy(dtype=float)).all():
        raise ValueError(f"{name}: NaN or infinite expression values")

    return df

guard self-test: 5/5 passed


In [6]:
# ============================================================
# CELL 5 — DOWNLOAD
# ============================================================
try:
    import GEOparse
except ModuleNotFoundError:
    raise RuntimeError(
        "GEOparse missing. Run Cell 1 and let it FINISH (do not cancel). "
        "Then restart the runtime and re-run Cells 1, 2, 3, 4, then 5.")

import urllib.request, urllib.error
import GEOparse
import urllib.request, urllib.error

GEO_FTP = "https://ftp.ncbi.nlm.nih.gov/geo/series/{stub}nnn/{gse}/"

def _stub(gse):
    n = gse.replace("GSE", "")
    return "GSE" + (n[:-3] if len(n) > 3 else "")

def download_file(url, dest, retries=3, timeout=180):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return dest, "cached"
    dest.parent.mkdir(parents=True, exist_ok=True)
    for attempt in range(retries):
        try:
            urllib.request.urlretrieve(url, dest)
            if dest.stat().st_size == 0:
                raise IOError("zero-byte download")
            return dest, "downloaded"
        except Exception as e:
            if dest.exists():
                dest.unlink()
            if attempt == retries - 1:
                return None, f"failed: {type(e).__name__}: {e}"
            time.sleep(3 * (attempt + 1))
    return None, "failed"

def verify_gzip(path):
    """A truncated gzip that half-reads is a failure mode you already hit."""
    try:
        with gzip.open(path, "rb") as f:
            while f.read(1 << 22):
                pass
        return True
    except Exception:
        return False

def fetch_cohort(gse, pull_supp=True):
    outdir = PATHS["raw_geo"] / gse
    outdir.mkdir(parents=True, exist_ok=True)
    rec = {"gse": gse, "tier": tier_of(gse), "soft": None,
           "supp": [], "errors": []}

    # --- SOFT (metadata; always needed, even for EXCLUDED cohorts) ---
    try:
        gseobj = GEOparse.get_GEO(geo=gse, destdir=str(outdir), silent=True,
                                  include_data=False)
        rec["soft"] = "ok"
        rec["n_gsm"] = len(gseobj.gsms)
        rec["gpl"] = list(gseobj.gpls.keys())
    except Exception as e:
        rec["errors"].append(f"SOFT: {type(e).__name__}: {e}")
        return rec

    # --- supplementary files ---
    if pull_supp:
        base = GEO_FTP.format(stub=_stub(gse), gse=gse) + "suppl/"
        try:
            listing = urllib.request.urlopen(base, timeout=60).read().decode("utf-8", "ignore")
            fnames = sorted(set(re.findall(r'href="([^"]+)"', listing)))
            fnames = [f for f in fnames if not f.startswith(("/", "?"))]
        except Exception as e:
            rec["errors"].append(f"suppl listing: {e}")
            fnames = []

        for fn in fnames:
            # skip giant RAW tarballs unless the cohort needs them
            if fn.endswith("_RAW.tar") and gse not in ("GSE207310", "GSE163211"):
                rec["supp"].append({"file": fn, "status": "skipped_RAW"})
                continue
            dest, status = download_file(base + fn, outdir / fn)
            entry = {"file": fn, "status": status}
            if dest and fn.endswith(".gz"):
                entry["gzip_ok"] = verify_gzip(dest)
                if not entry["gzip_ok"]:
                    dest.unlink(missing_ok=True)
                    entry["status"] = "corrupt_removed"
            if dest and dest.exists():
                entry["bytes"] = dest.stat().st_size
            rec["supp"].append(entry)
    return rec


# ---------------- run ----------------
DOWNLOAD_LOG = {}
targets = [g for g, m in COHORTS.items()]      # EVERY cohort, no exclusions

for gse in tqdm(targets, desc="GEO"):
    DOWNLOAD_LOG[gse] = fetch_cohort(gse)

(PATHS["reports"] / "download_log.json").write_text(json.dumps(DOWNLOAD_LOG, indent=2))

print("\nDOWNLOAD SUMMARY")
print(f"{'cohort':12s} {'tier':9s} {'GSM':>5s}  {'supp':>4s}  errors")
for g, r in DOWNLOAD_LOG.items():
    n_ok = sum(1 for s in r.get("supp", []) if s.get("status") in ("downloaded", "cached"))
    print(f"{g:12s} {r['tier']:9s} {r.get('n_gsm', 0):5d}  {n_ok:4d}  "
          f"{'; '.join(r['errors'])[:60]}")

write_manifest("p01_download",
               inputs={"registry": sha256_obj(COHORTS)},
               params={"pull_supp": True},
               outputs=[PATHS["reports"] / "download_log.json"])

GEO:   0%|          | 0/28 [00:00<?, ?it/s]


DOWNLOAD SUMMARY
cohort       tier        GSM  supp  errors
GSE135251    T1          216     1  
GSE84044     T1          124     1  
GSE162694    T1          143     1  
GSE130970    T1           78     2  
E-MTAB-6863  T1            0     0  SOFT: UnknownGEOTypeException: E-M type is not known
GSE89632     T1           63     1  
GSE48452     T1           73     1  
GSE31803     T1          131     1  
GSE276114    T2          177     1  
GSE240729    T2           67     2  
E-MTAB-9815  T2            0     0  SOFT: UnknownGEOTypeException: E-M type is not known
GSE213621    T3          368     1  
GSE163211    T4          318     2  
GSE193066    AUDIT       164     2  
GSE159088    AUDIT        80     1  
GSE207310    AUDIT        30     2  
GSE49541     REJECTED     72     1  
GSE167523    REJECTED     98     3  
GSE126848    REJECTED     57     1  
GSE61260     REJECTED    134     3  
GSE151158    REJECTED     66     1  
GSE139602    REJECTED     39     1  
GSE202379    REJECTED

{'stage': 'p01_download',
 'timestamp': '2026-10-08T10:26:28',
 'inputs': {'registry': 'e9d9637d8c5c211a3dc40dc0745f54698d2c4aa398470d6a8e0e09fd84eeb449'},
 'params': {'pull_supp': True},
 'outputs': {'/content/drive/MyDrive/CoDAGAT_preprocessing_fix_v1/reports/download_log.json': 'c03c01baf8336686fc9695195f58100a36aad29c465573e41488560f5e8bfc47'}}

In [7]:
# RETRY: GSE240729 metadata download failed (GEO network hiccup). Touches only raw/geo/GSE240729.
r = None
for attempt in range(1, 6):
    r = fetch_cohort("GSE240729")
    print(f"attempt {attempt}: soft={r['soft']}  n_gsm={r.get('n_gsm')}  errors={r['errors']}")
    if r["soft"] == "ok":
        break
    time.sleep(15 * attempt)
DOWNLOAD_LOG["GSE240729"] = r
assert r["soft"] == "ok" and r.get("n_gsm") == 67, "still failing — wait a few minutes and rerun this cell"
print(sorted(p.name for p in (PATHS["raw_geo"] / "GSE240729").iterdir()))

attempt 1: soft=ok  n_gsm=67  errors=[]
['GSE240729_family.soft.gz', 'GSE240729_merged.counts.67samp.txt.gz', 'filelist.txt', 'https:']


In [8]:
# ============================================================
# CELL 6 — ARRAYEXPRESS / BIOSTUDIES FETCHER
# For E-MTAB-6863 (T1) and E-MTAB-9815 (T2).
# Different API from GEO: metadata lives in the SDRF file,
# expression in a processed-data file whose name varies per study.
# NOTE: endpoint layout changes occasionally. This tries several
# and prints exactly what it found — read the output before trusting it.
# ============================================================

BIOSTUDIES_API  = "https://www.ebi.ac.uk/biostudies/api/v1/studies/{acc}"
BIOSTUDIES_FILE = "https://www.ebi.ac.uk/biostudies/files/{acc}/{fname}"
AE_FILES_ALT    = "https://www.ebi.ac.uk/biostudies/api/v1/files/{acc}/{fname}"

def _http_json(url, timeout=90):
    req = urllib.request.Request(url, headers={"Accept": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode("utf-8", "ignore"))

def _walk_files(node, out):
    """BioStudies nests file entries in section/subsections; flatten them."""
    if isinstance(node, dict):
        if "path" in node and "size" in node:
            out.append(node)
        for v in node.values():
            _walk_files(v, out)
    elif isinstance(node, list):
        for v in node:
            _walk_files(v, out)
    return out

def fetch_arrayexpress(acc):
    outdir = PATHS["raw_geo"] / acc
    outdir.mkdir(parents=True, exist_ok=True)
    rec = {"acc": acc, "tier": tier_of(acc), "files": [], "errors": []}

    # ---- 1. study metadata + file listing ----
    try:
        meta = _http_json(BIOSTUDIES_API.format(acc=acc))
        (outdir / f"{acc}_biostudies.json").write_text(json.dumps(meta, indent=2))
    except Exception as e:
        rec["errors"].append(f"api: {type(e).__name__}: {e}")
        return rec

    entries = _walk_files(meta, [])
    names = sorted({e["path"] for e in entries})
    rec["listing"] = names
    print(f"\n[{acc}] {len(names)} files listed")
    for n in names[:40]:
        print("   ", n)
    if len(names) > 40:
        print(f"    ... and {len(names)-40} more")

    # ---- 2. always take SDRF + IDF (metadata) ----
    want = [n for n in names if n.lower().endswith((".sdrf.txt", ".idf.txt"))]

    # ---- 3. processed expression: name varies, so pattern-match ----
    expr_pat = re.compile(
        r"(count|cpm|fpkm|tpm|expression|processed|matrix|norm)", re.I)
    want += [n for n in names
             if expr_pat.search(n)
             and n.lower().endswith((".txt", ".tsv", ".csv", ".gz", ".zip"))
             and not n.lower().endswith((".sdrf.txt", ".idf.txt"))]
    want = sorted(set(want))

    for fname in want:
        got = None
        for tmpl in (BIOSTUDIES_FILE, AE_FILES_ALT):
            url = tmpl.format(acc=acc, fname=urllib.parse.quote(fname))
            dest, status = download_file(url, outdir / Path(fname).name)
            if dest:
                got = {"file": fname, "status": status,
                       "bytes": dest.stat().st_size, "url": url}
                if fname.endswith(".gz"):
                    got["gzip_ok"] = verify_gzip(dest)
                break
        rec["files"].append(got or {"file": fname, "status": "failed_all_endpoints"})

    return rec


import urllib.parse
AE_COHORTS = [g for g, m in COHORTS.items()
              if m.get("repo") == "ArrayExpress" and m["tier"] in ("T1", "T2", "T3")]

for acc in AE_COHORTS:
    DOWNLOAD_LOG[acc] = fetch_arrayexpress(acc)

(PATHS["reports"] / "download_log.json").write_text(json.dumps(DOWNLOAD_LOG, indent=2))

print("\nARRAYEXPRESS SUMMARY")
for acc in AE_COHORTS:
    r = DOWNLOAD_LOG[acc]
    ok = [f for f in r["files"] if f and f.get("status") in ("downloaded", "cached")]
    print(f"  {acc:14s} files_ok={len(ok):2d}  errors={r['errors']}")
    for f in ok:
        print(f"      {f['file']:55s} {f['bytes']/1e6:8.2f} MB")

print("\nACTION: confirm an SDRF file downloaded for each accession.")
print("        The SDRF is where per-sample fibrosis stage lives.")


[E-MTAB-6863] 74 files listed
    E-MTAB-6863.idf.txt
    E-MTAB-6863.sdrf.txt
    norm_counts_highfilt.txt
    norm_counts_lowfilt.txt
    norm_counts_medfilt.txt
    rawcounts_sample01.txt
    rawcounts_sample02.txt
    rawcounts_sample03.txt
    rawcounts_sample04.txt
    rawcounts_sample05.txt
    rawcounts_sample06.txt
    rawcounts_sample07.txt
    rawcounts_sample08.txt
    rawcounts_sample09.txt
    rawcounts_sample10.txt
    rawcounts_sample11.txt
    rawcounts_sample12.txt
    rawcounts_sample13.txt
    rawcounts_sample14.txt
    rawcounts_sample15.txt
    rawcounts_sample16.txt
    rawcounts_sample17.txt
    rawcounts_sample18.txt
    rawcounts_sample19.txt
    rawcounts_sample20.txt
    rawcounts_sample21.txt
    rawcounts_sample22.txt
    rawcounts_sample23.txt
    rawcounts_sample24.txt
    rawcounts_sample25.txt
    rawcounts_sample26.txt
    rawcounts_sample27.txt
    rawcounts_sample28.txt
    rawcounts_sample29.txt
    rawcounts_sample30.txt
    rawcounts_sample31.tx

In [9]:
# ============================================================
# CELL 7 — STAGE PARSER  (the Phase 2 bug lives here)
#
# Returns (stage, resolution, raw) — NEVER a bare stage.
# The bare regex \b([0-4])\b as a fallback is what fabricated
# exact stages for 111 grouped-label samples. It is not used.
# ============================================================

Label = namedtuple("Label", ["stage", "resolution", "raw"])

RES_EXACT       = "exact"
RES_G_F0F1      = "grouped_F0_F1"
RES_G_F3F4      = "grouped_F3_F4"
RES_G_F0F2      = "grouped_F0_F2"
RES_HEALTHY     = "healthy"
RES_UNKNOWN     = "unknown"

STAGE_KEYS = [
    "fibrosis stage", "fibrosis_stage", "fibrosis-stage", "fibrotic stage",
    "fibrosis score", "fibrosis_score", "fibrosis", "stage",
    "nas fibrosis", "nas_fibrosis", "nafld fibrosis stage",
    "scheuer score", "scheuer", "metavir", "metavir score",
    "kleiner fibrosis_grade", "kleiner fibrosis grade", "kleiner",
    "ishak fibrosis", "ishak", "knodell fibrosis", "brunt",
    "disease group", "group", "diagnosis", "histology", "condition",
    "fibrosis severity", "severity", "characteristics",
]

HEALTHY_WORDS = ("healthy", "normal liver", "normal control", "control",
                 "donor", "non-diseased", "nondiseased")

# --- ordered patterns; FIRST MATCH WINS -----------------------
GROUPED_PATTERNS = [
    # explicit ranges, with or without the letter F
    (re.compile(r"f?\s*0\s*[-–—to]+\s*f?\s*1\b", re.I), "F0", RES_G_F0F1),
    (re.compile(r"f?\s*3\s*[-–—to]+\s*f?\s*4\b", re.I), "F3", RES_G_F3F4),
    (re.compile(r"f?\s*0\s*[-–—to]+\s*f?\s*2\b", re.I), "F0", RES_G_F0F2),
    (re.compile(r"f?\s*1\s*[-–—to]+\s*f?\s*4\b", re.I), None, RES_UNKNOWN),   # straddles
    # word forms only when a bracketed range accompanies them
    (re.compile(r"mild.*?stage\s*0\s*[-–—]\s*1", re.I),     "F0", RES_G_F0F1),
    (re.compile(r"advanced.*?stage\s*3\s*[-–—]\s*4", re.I), "F3", RES_G_F3F4),
    (re.compile(r"mild.*?\bf?0\s*[-–—]\s*f?1\b", re.I),     "F0", RES_G_F0F1),
    (re.compile(r"advanced.*?\bf?3\s*[-–—]\s*f?4\b", re.I), "F3", RES_G_F3F4),
]

RE_F_EXACT     = re.compile(r"\bf\s*([0-4])\b", re.I)
RE_STAGE_EXACT = re.compile(r"(?:stage|score|grade)\s*[:=]?\s*([0-4])(?![\d.])", re.I)
RE_KLEINER_SUB = re.compile(r"\b([0-4])\s*([abc])\b", re.I)
RE_ISHAK       = re.compile(r"ishak[^0-9]{0,12}([0-6])", re.I)

ISHAK_MAP = {0: "F0", 1: "F1", 2: "F1", 3: "F2", 4: "F3", 5: "F4", 6: "F4"}


def parse_stage(raw_value, key_hint=""):
    """raw_value: the characteristics string. key_hint: its metadata key."""
    if raw_value is None:
        return Label(None, RES_UNKNOWN, "")
    raw = str(raw_value).strip()
    if not raw:
        return Label(None, RES_UNKNOWN, raw)
    low = f"{key_hint} {raw}".lower()

    # 1. healthy FIRST — healthy liver is never stage F0
    if any(w in low for w in HEALTHY_WORDS):
        if not re.search(r"\b(f[0-4]|stage\s*[0-4])\b", low):
            return Label(None, RES_HEALTHY, raw)

    # 2. grouped ranges BEFORE any single-digit extraction
    for pat, stage, res in GROUPED_PATTERNS:
        if pat.search(low):
            return Label(stage, res, raw)

    # 3. Ishak (different scale, must be caught before generic digits)
    m = RE_ISHAK.search(low)
    if m:
        return Label(ISHAK_MAP[int(m.group(1))], "exact_ishak_mapped", raw)

    # 4. Kleiner sub-stages 1A/1B/1C -> F1
    m = RE_KLEINER_SUB.search(raw)
    if m and ("kleiner" in low or "grade" in low or "fibrosis" in low):
        return Label(f"F{int(m.group(1))}", RES_EXACT, raw)

    # 5. explicit F0..F4
    m = RE_F_EXACT.search(raw)
    if m:
        return Label(f"F{int(m.group(1))}", RES_EXACT, raw)

    # 6. "stage: 2" / "score = 3" — requires the keyword, never a bare digit
    m = RE_STAGE_EXACT.search(low)
    if m:
        return Label(f"F{int(m.group(1))}", RES_EXACT, raw)

    # 7. a bare digit ONLY if the metadata key itself is a fibrosis key
    if any(k in key_hint.lower() for k in
           ("fibrosis", "fibrotic", "metavir", "scheuer", "kleiner", "ishak")):
        m = re.fullmatch(r"\s*([0-4])(?:\.0)?\s*", raw)
        if m:
            return Label(f"F{int(m.group(1))}", RES_EXACT, raw)

    # 8. NEVER guess. NAFL/NASH/steatosis are diagnoses, not stages.
    return Label(None, RES_UNKNOWN, raw)


USABLE = {
    # resolution -> (ok_for_F2plus_task, ok_for_ordinal, ok_for_early_vs_adv)
    RES_EXACT:            (True,  True,  True),
    "exact_ishak_mapped": (True,  True,  True),
    RES_G_F0F1:           (True,  False, True),
    RES_G_F3F4:           (True,  False, True),
    RES_G_F0F2:           (False, False, True),   # straddles the F2 boundary
    RES_HEALTHY:          (False, False, False),
    RES_UNKNOWN:          (False, False, False),
}


# ---------------- self-test on REAL strings from your Phase 2 run ----------------
def _test_parser():
    cases = [
        # (key, value, expected_stage, expected_resolution)
        ("Stage",                   "mild (fibrosis stage 0-1)",     "F0", RES_G_F0F1),
        ("Stage",                   "advanced (fibrosis stage 3-4)", "F3", RES_G_F3F4),
        ("disease group",           "F0-2",                          "F0", RES_G_F0F2),
        ("disease group",           "F3",                            "F3", RES_EXACT),
        ("disease group",           "F4",                            "F4", RES_EXACT),
        ("fibrotic stage",          "2",                             "F2", RES_EXACT),
        ("fibrosis stage",          "3",                             "F3", RES_EXACT),
        ("nas_fibrosis",            "0",                             "F0", RES_EXACT),
        ("kleiner fibrosis_grade",  "1A",                            "F1", RES_EXACT),
        ("kleiner fibrosis_grade",  "1B",                            "F1", RES_EXACT),
        ("Scheuer score",           "S2",                            None, RES_UNKNOWN),
        ("fibrosis stage",          "F2",                            "F2", RES_EXACT),
        ("diagnosis",               "NAFL",                          None, RES_UNKNOWN),
        ("diagnosis",               "NASH",                          None, RES_UNKNOWN),
        ("tissue",                  "healthy liver",                 None, RES_HEALTHY),
        ("condition",               "normal control",                None, RES_HEALTHY),
        ("group",                   "steatosis",                     None, RES_UNKNOWN),
        ("ishak fibrosis",          "ishak 5",                       "F4", "exact_ishak_mapped"),
        ("fibrosis",                "1-4",                           None, RES_UNKNOWN),
        ("age",                     "52",                            None, RES_UNKNOWN),  # bare digit
    ]
    fails = []
    for key, val, exp_s, exp_r in cases:
        got = parse_stage(val, key)
        if (got.stage, got.resolution) != (exp_s, exp_r):
            fails.append((key, val, exp_s, exp_r, got.stage, got.resolution))
    print(f"parser self-test: {len(cases)-len(fails)}/{len(cases)} passed")
    for f in fails:
        print(f"  FAIL key={f[0]!r} val={f[1]!r}  expected=({f[2]},{f[3]})  got=({f[4]},{f[5]})")
    return not fails

_test_parser()

parser self-test: 20/20 passed


True

In [10]:
# ============================================================
# CELL 8 — METADATA KEY DISCOVERY
# Dumps every characteristics key + value distribution per cohort.
# Purpose: find staging keys the parser doesn't know about yet.
# GSE213621 is the priority — published work reports n=368 with
# F0-F2 / F3-F4 labels; Phase 2 recovered only 107 F2.
# ============================================================

def harvest_characteristics(gse):
    """Return {key: Counter(values)} across all GSMs in a series."""
    softdir = PATHS["raw_geo"] / gse
    try:
        g = GEOparse.get_GEO(geo=gse, destdir=str(softdir), silent=True, include_data=False)
    except Exception as e:
        return None, f"{type(e).__name__}: {e}"

    keys = defaultdict(Counter)
    per_sample = {}
    for gsm_id, gsm in g.gsms.items():
        row = {}
        for field, vals in gsm.metadata.items():
            if not field.startswith(("characteristics", "title", "source_name",
                                     "description", "supplementary")):
                continue
            for v in vals:
                if ":" in v:
                    k, val = v.split(":", 1)
                    k, val = k.strip().lower(), val.strip()
                else:
                    k, val = field, v.strip()
                keys[k][val] += 1
                row[k] = val
        per_sample[gsm_id] = row
    return (keys, per_sample), None


DISCOVERY = {}
targets = cohorts_by_tier("T1", "T2", "T3", "T4", "AUDIT")
targets = [t for t in targets if REPO_OF[t] == "GEO"]

for gse in tqdm(targets, desc="metadata"):
    res, err = harvest_characteristics(gse)
    DISCOVERY[gse] = {"error": err} if err else {
        "keys": {k: dict(c.most_common(25)) for k, c in res[0].items()},
        "n_gsm": len(res[1]),
        "per_sample": res[1],
    }

# --- print, prioritising the cohort we need to recover ---
def show(gse, max_keys=40):
    d = DISCOVERY.get(gse, {})
    if d.get("error"):
        print(f"{gse}: ERROR {d['error']}"); return
    print(f"\n{'='*72}\n{gse}  (n_gsm={d['n_gsm']})")
    for k, vals in list(d["keys"].items())[:max_keys]:
        total = sum(vals.values())
        looks_staging = any(s in k for s in
            ("fibro", "stage", "metavir", "scheuer", "kleiner", "ishak",
             "grade", "group", "nas", "histo", "severity", "diagnos"))
        flag = " <<< POSSIBLE STAGING" if looks_staging else ""
        preview = ", ".join(f"{v}({n})" for v, n in list(vals.items())[:8])
        print(f"  [{total:4d}] {k:34s} {preview[:110]}{flag}")

show("GSE213621", max_keys=60)          # priority
for gse in targets:
    if gse != "GSE213621":
        show(gse)

with open(PATHS["reports"] / "metadata_keys.json", "w") as f:
    json.dump({g: {k: v for k, v in d.items() if k != "per_sample"}
               for g, d in DISCOVERY.items()}, f, indent=2)

print("\n" + "="*72)
print("ACTION: scan the <<< POSSIBLE STAGING lines above.")
print("Any key the parser does not already handle goes into STAGE_KEYS (Cell 7),")
print("and any new value format goes into the parser test cases BEFORE re-running.")

metadata:   0%|          | 0/14 [00:00<?, ?it/s]


GSE213621  (n_gsm=368)
  [  25] title                              human liver sample A1025223(1), human liver sample 200707904(1), human liver sample 200707919(1), human liver 
  [ 368] source_name_ch1                    liver cells(368)
  [ 368] cell type                          liver cells(368)
  [ 368] fibrotic stage                     F2(107), F0F1(97), F3F4(95), Control(69) <<< POSSIBLE STAGING
  [ 368] supplementary_file_1               NONE(368)

GSE135251  (n_gsm=216)
  [  25] title                              Liver patient 97(1), Liver patient 98(1), Liver patient 101(1), Liver patient 102(1), Liver patient 105(1), Li
  [ 216] source_name_ch1                    NAFLD_early_liver biopsy(138), NAFLD_moderate_liver biopsy(68), Control_control_liver biopsy(10)
  [ 216] nas score                          5(47), 4(38), 6(37), 3(26), 2(21), 7(18), 1(11), 0(10) <<< POSSIBLE STAGING
  [ 216] fibrosis stage                     2(54), 3(54), 1(48), 0(46), 4(14) <<< POSSIBLE STAGING


In [11]:
# ============================================================
# CELL 9 — BUILD LABEL TABLE
# One row per sample. Resolution class is mandatory.
# Nothing is dropped — unusable samples are retained and flagged.
# ============================================================

def build_labels_for_cohort(gse):
    d = DISCOVERY.get(gse)
    if not d or d.get("error"):
        return pd.DataFrame()
    meta = COHORTS[gse]
    rows = []
    for gsm_id, row in d["per_sample"].items():
        best = Label(None, RES_UNKNOWN, "")
        # try dedicated staging keys first, then everything else
        ordered = ([k for k in row if any(s in k for s in
                    ("fibro", "metavir", "scheuer", "kleiner", "ishak"))]
                   + [k for k in row if k not in
                      [j for j in row if any(s in j for s in
                       ("fibro", "metavir", "scheuer", "kleiner", "ishak"))]])
        for k in ordered:
            lab = parse_stage(row[k], k)
            if lab.resolution != RES_UNKNOWN:
                best = lab
                if lab.resolution in (RES_EXACT, "exact_ishak_mapped"):
                    break        # exact beats grouped; stop looking
        rows.append(dict(
            sample_id=gsm_id, cohort=gse,
            stage=best.stage, resolution=best.resolution, raw_string=best.raw,
            etiology=meta.get("etiology"), platform=meta.get("platform"),
            preservation=meta.get("preservation"), tier=meta["tier"],
            repo=meta.get("repo", "GEO"), subject_id=None,
        ))
    return pd.DataFrame(rows)


frames = [build_labels_for_cohort(g) for g in targets]
labels_raw = pd.concat([f for f in frames if len(f)], ignore_index=True)

# ---- task labels, derived strictly from resolution ----
def f2plus(r):
    if not USABLE[r.resolution][0] or r.stage is None: return np.nan
    return 1 if r.stage in ("F2", "F3", "F4") else 0

def early_adv(r):
    if not USABLE[r.resolution][2] or r.stage is None: return np.nan
    return 1 if r.stage in ("F3", "F4") else 0

def ordinal(r):
    if not USABLE[r.resolution][1] or r.stage is None: return np.nan
    return int(r.stage[1])

labels_raw["y_T1_significant"] = labels_raw.apply(f2plus, axis=1)
labels_raw["y_T3_early_adv"]   = labels_raw.apply(early_adv, axis=1)
labels_raw["y_T2_ordinal"]     = labels_raw.apply(ordinal, axis=1)

assert_no_healthy_as_F0(labels_raw)

out = PATHS["interim"] / "labels_raw.parquet"
labels_raw.to_parquet(out, index=False)

# ---------------- reports ----------------
print("RESOLUTION BREAKDOWN")
print(pd.crosstab(labels_raw.cohort, labels_raw.resolution, margins=True))

print("\nSTAGE x COHORT (exact + grouped)")
print(pd.crosstab(labels_raw.cohort, labels_raw.stage.fillna("NONE"), margins=True))

print("\nTASK USABILITY per cohort")
summ = labels_raw.groupby("cohort").agg(
    n=("sample_id", "size"),
    T1_usable=("y_T1_significant", lambda s: s.notna().sum()),
    T1_pos=("y_T1_significant", lambda s: (s == 1).sum()),
    T2_usable=("y_T2_ordinal", lambda s: s.notna().sum()),
    T3_usable=("y_T3_early_adv", lambda s: s.notna().sum()),
    T3_pos=("y_T3_early_adv", lambda s: (s == 1).sum()),
)
summ["T1_minority"] = summ[["T1_pos"]].join(
    (summ.T1_usable - summ.T1_pos).rename("T1_neg")).min(axis=1)
summ["T1_tier_suggestion"] = np.where(summ.T1_minority >= 15, "T1-eligible", "T2/T3 only")
print(summ.to_string())

print(f"\nN_real (any usable task): "
      f"{labels_raw[['y_T1_significant','y_T3_early_adv']].notna().any(axis=1).sum()}")
print(f"Unknown/unparsed: {(labels_raw.resolution == RES_UNKNOWN).sum()}")
print(f"Healthy (excluded from staging): {(labels_raw.resolution == RES_HEALTHY).sum()}")

write_manifest("p03_labels",
               inputs={"registry": sha256_obj(COHORTS)},
               params={"parser": "v2_resolution_classes"},
               outputs=[out])

RESOLUTION BREAKDOWN
resolution  exact  grouped_F0_F1  grouped_F0_F2  grouped_F3_F4  healthy  unknown   All
cohort                                                                                
GSE130970      78              0              0              0        0        0    78
GSE135251     216              0              0              0        0        0   216
GSE159088       0              0              0              0        0       80    80
GSE162694     143              0              0              0        0        0   143
GSE163211       0              0              0              0        0      318   318
GSE193066     164              0              0              0        0        0   164
GSE207310      30              0              0              0        0        0    30
GSE213621     107              0              0              0       69      192   368
GSE240729      67              0              0              0        0        0    67
GSE276114     138     

{'stage': 'p03_labels',
 'timestamp': '2026-10-08T10:30:49',
 'inputs': {'registry': 'e9d9637d8c5c211a3dc40dc0745f54698d2c4aa398470d6a8e0e09fd84eeb449'},
 'params': {'parser': 'v2_resolution_classes'},
 'outputs': {'/content/drive/MyDrive/CoDAGAT_preprocessing_fix_v1/interim/labels_raw.parquet': 'a4e8d113975ae05b7b56cfed99188dd999f6e5ca5bb964a191a87b8c40f47667'}}

In [12]:
# ============================================================
# CELL 10 — PARSER v3
# Fixes found in the Cell 8/9 audit:
#   (a) concatenated grouped forms: F0F1, F3F4, F0F2
#   (b) "Control"/"HC"/"Normal" as a VALUE -> healthy
#   (c) Scheuer S (fibrosis) must beat Scheuer G (inflammation)
#   (d) "nas score" / "activity score" must NOT satisfy the
#       generic (stage|score|grade)\s*[0-4] pattern
#   (e) cohort-level healthy override from a separate field
# ============================================================

# --- keys that are FIBROSIS, in priority order (first wins) ---
FIBROSIS_KEY_PRIORITY = [
    "scheuer score s",          # Scheuer S = fibrosis stage  (NOT "scheuer score g")
    "fibrosis stage", "fibrosis_stage", "fibrosis-stage",
    "fibrotic stage", "fibrosisscore", "fibrosis score",
    "fibrosis (stage)", "fibrosis",
    "kleiner fibrosis_grade", "kleiner fibrosis grade",
    "metavir", "metavir score", "ishak fibrosis", "ishak",
    "disease group", "stage",
]

# --- keys that look like staging but are NOT fibrosis ---
NON_FIBROSIS_KEYS = {
    "scheuer score g",                    # necroinflammatory grade
    "nas score", "nas", "nafld activity score", "nafld activity_score",
    "lobular inflammation grade", "lobular inflammation (severity)",
    "cytological ballooning grade", "ballooning (intensity)",
    "steatosis grade", "steatosis (%)", "inflammation", "fat",
    "aspartate transaminase (u/l)", "alanine transaminase (u/l)",
    "alkaline phosphatase (u/l)", "age", "age (y)", "age at biopsy",
    "bmi", "body mass index (kg/m2)", "batch", "washbatch",
    "steatosis, activity,_and_fibrosis_score",   # SAF composite, not F-stage
}

# --- values that mean "not a patient with staged fibrosis" ---
HEALTHY_VALUES = {
    "control", "controls", "hc", "healthy control", "normal",
    "normal liver", "normal liver histology", "healthy", "donor",
    "non-diseased", "nondiseased",
}
HEALTHY_FIELDS = ["disease", "diagnosis", "group", "source_name_ch1",
                  "fibrotic stage", "condition", "tissue"]

# --- concatenated + separated grouped ranges (checked BEFORE digits) ---
GROUPED_PATTERNS = [
    (re.compile(r"\bf?0\s*[-–—/to]*\s*f?1\b", re.I), "F0", RES_G_F0F1),
    (re.compile(r"\bf?3\s*[-–—/to]*\s*f?4\b", re.I), "F3", RES_G_F3F4),
    (re.compile(r"\bf?0\s*[-–—/to]*\s*f?2\b", re.I), "F0", RES_G_F0F2),
    (re.compile(r"\bf?1\s*[-–—/to]*\s*f?4\b", re.I), None, RES_UNKNOWN),  # straddles
    (re.compile(r"mild.*?stage\s*0\s*[-–—]\s*1", re.I),     "F0", RES_G_F0F1),
    (re.compile(r"advanced.*?stage\s*3\s*[-–—]\s*4", re.I), "F3", RES_G_F3F4),
]

RE_SCHEUER_GS = re.compile(r"\bg([0-4])\s*s([0-4])\b", re.I)   # "G1S2" -> S is fibrosis


def parse_stage(raw_value, key_hint=""):
    raw = "" if raw_value is None else str(raw_value).strip()
    key = str(key_hint).strip().lower()
    val = raw.lower()
    unknown = Label(None, RES_UNKNOWN, raw)

    if not raw or key in NON_FIBROSIS_KEYS:
        return unknown

    if val in HEALTHY_VALUES or any(
        re.fullmatch(rf"{re.escape(h)}s?", val)
        for h in HEALTHY_VALUES
    ):
        return Label(None, RES_HEALTHY, raw)

    if any(h in val for h in (
        "normal liver", "healthy control", "non-diseased"
    )):
        if re.search(r"\bf\s*\d|\bstage\s*\d", val):
            return unknown
        return Label(None, RES_HEALTHY, raw)

    # Handle the declared Ishak scale before generic numeric stages.
    if "ishak" in key or "ishak" in val:
        m = re.fullmatch(
            r"(?:ishak(?:\s+(?:fibrosis|score|stage))*"
            r"\s*[:=]?\s*)?([0-6])(?:\.0)?",
            val,
        )
        if m:
            return Label(
                ISHAK_MAP[int(m[1])], "exact_ishak_mapped", raw
            )
        return unknown

    m = re.fullmatch(r"g([0-4])\s*s([0-4])", val)
    if m:
        return Label(f"F{m[2]}", RES_EXACT, raw)

    v = val
    m = re.fullmatch(r"(?:mild|advanced)\s*\((.*)\)", v)
    if m:
        v = m[1].strip()

    v = re.sub(
        r"^(?:fibrosis\s+(?:stage|score)|fibrotic\s+stage|stage)"
        r"\s*[:=]?\s*",
        "",
        v,
    )

    # Match the complete range, never its first stage.
    m = re.fullmatch(
        r"f?\s*([0-4])\s*(?:[-–—/_]|to)\s*f?\s*([0-4])",
        v,
    )
    if m is None:
        m = re.fullmatch(r"f\s*([0-4])\s*f\s*([0-4])", v)

    if m:
        grouped = {
            (0, 1): ("F0", RES_G_F0F1),
            (0, 2): ("F0", RES_G_F0F2),
            (3, 4): ("F3", RES_G_F3F4),
        }
        stage, resolution = grouped.get(
            (int(m[1]), int(m[2])),
            (None, RES_UNKNOWN),
        )
        return Label(stage, resolution, raw)

    if "kleiner" in key or "fibro" in key:
        if re.fullmatch(r"1\s*[abc]", v):
            return Label("F1", RES_EXACT, raw)

    m = re.fullmatch(r"f\s*([0-4])(?:\.0)?", v)
    if m:
        return Label(f"F{m[1]}", RES_EXACT, raw)

    if (
        key in FIBROSIS_KEY_PRIORITY
        or "fibro" in key
        or key.startswith("scheuer score s")
    ):
        m = re.fullmatch(r"([0-4])(?:\.0)?", v)
        if m:
            return Label(f"F{m[1]}", RES_EXACT, raw)

    return unknown
for value, key, expected_stage, expected_resolution in [
    ("F1-F2", "fibrosis stage", None, RES_UNKNOWN),
    ("F2.5", "fibrosis stage", None, RES_UNKNOWN),
    ("F0-F1-F2", "fibrosis stage", None, RES_UNKNOWN),
    ("F0 to F2", "fibrosis stage", "F0", RES_G_F0F2),
    ("3", "ishak fibrosis", "F2", "exact_ishak_mapped"),
    ("34", "fibrosis stage", None, RES_UNKNOWN),
    ("F0 F1", "fibrosis stage", "F0", RES_G_F0F1),
]:
    got = parse_stage(value, key)
    assert (got.stage, got.resolution) == (
        expected_stage, expected_resolution
    ), (value, key, got)
def _test_parser_v3():
    cases = [
        ("fibrotic stage",         "F2",     "F2", RES_EXACT),
        ("fibrotic stage",         "F0F1",   "F0", RES_G_F0F1),   # was BROKEN
        ("fibrotic stage",         "F3F4",   "F3", RES_G_F3F4),   # was BROKEN
        ("fibrotic stage",         "Control", None, RES_HEALTHY), # was BROKEN
        ("scheuer score s",        "2",      "F2", RES_EXACT),
        ("scheuer score g",        "2",      None, RES_UNKNOWN),  # was BROKEN
        ("nas score",              "0",      None, RES_UNKNOWN),  # was BROKEN
        ("nafld activity score",   "5",      None, RES_UNKNOWN),
        ("fibrosis stage",         "normal liver histology", None, RES_HEALTHY),  # was BROKEN
        ("fibrosis stage",         "0",      "F0", RES_EXACT),
        ("disease group",          "F0-2",   "F0", RES_G_F0F2),
        ("disease group",          "F4",     "F4", RES_EXACT),
        ("stage",                  "mild (fibrosis stage 0-1)",     "F0", RES_G_F0F1),
        ("stage",                  "advanced (fibrosis stage 3-4)", "F3", RES_G_F3F4),
        ("stage",                  "mild",   None, RES_UNKNOWN),   # bare word, no range
        ("kleiner fibrosis_grade", "1A",     "F1", RES_EXACT),
        ("kleiner fibrosis_grade", "2",      "F2", RES_EXACT),
        ("fibrosisscore",          "F2",     "F2", RES_EXACT),
        ("fibrosis",               "3.5",    None, RES_UNKNOWN),
        ("diagnosis",              "HC",     None, RES_HEALTHY),
        ("diagnosis",              "NASH",   None, RES_UNKNOWN),
        ("group",                  "Healthy obese", None, RES_UNKNOWN),
        ("age",                    "52",     None, RES_UNKNOWN),
        ("nafld stage",            "NASH_F1_F4", None, RES_UNKNOWN),
    ]
    fails = [(k, v, es, er, g.stage, g.resolution)
             for k, v, es, er in cases
             for g in [parse_stage(v, k)]
             if (g.stage, g.resolution) != (es, er)]
    print(f"parser v3 self-test: {len(cases)-len(fails)}/{len(cases)} passed")
    for f in fails:
        print(f"  FAIL key={f[0]!r} val={f[1]!r} expected=({f[2]},{f[3]}) got=({f[4]},{f[5]})")
    return not fails

assert _test_parser_v3(), "Parser tests failed — DO NOT PROCEED."

parser v3 self-test: 24/24 passed


In [13]:
# ============================================================
# CELL 11 — REBUILD LABEL TABLE  (supersedes Cell 9)
# ============================================================

# Treat "Healthy obese" (GSE48452) as diseased-spectrum, not healthy.
# Flip to True only if you decide otherwise — and say so in the paper.
EXCLUDE_HEALTHY_OBESE = False

# Fields that override any fibrosis value with "this is a control subject"
COHORT_HEALTHY_OVERRIDE = {
    "GSE135251": ("disease",        {"control"}),
    "GSE89632":  ("diagnosis",      {"hc"}),
    "GSE48452":  ("group",          {"control"} | ({"healthy obese"} if EXCLUDE_HEALTHY_OBESE else set())),
    "GSE213621": ("fibrotic stage", {"control"}),
    "GSE162694": ("fibrosis stage", {"normal liver histology"}),
}

# Patient-level dedup: keep one biopsy per subject
DEDUP_RULES = {
    "GSE193066": ("biopsy", "1st biopsy"),   # drop 58 follow-up biopsies
}


def pick_stage(row_dict, gse):
    """Try fibrosis keys in declared priority order; exact beats grouped."""
    best = Label(None, RES_UNKNOWN, "")
    keys_present = list(row_dict.keys())
    ordered = ([k for k in FIBROSIS_KEY_PRIORITY if k in row_dict]
               + [k for k in keys_present
                  if k not in FIBROSIS_KEY_PRIORITY and k not in NON_FIBROSIS_KEYS])
    for k in ordered:
        lab = parse_stage(row_dict[k], k)
        if lab.resolution in (RES_EXACT, "exact_ishak_mapped"):
            return lab
        if lab.resolution != RES_UNKNOWN and best.resolution == RES_UNKNOWN:
            best = lab
    return best


def build_labels_v3(gse):
    d = DISCOVERY.get(gse)
    if not d or d.get("error"):
        return pd.DataFrame()
    meta = COHORTS[gse]
    ov_field, ov_values = COHORT_HEALTHY_OVERRIDE.get(gse, (None, set()))
    dd_field, dd_keep = DEDUP_RULES.get(gse, (None, None))

    rows = []
    for gsm_id, row in d["per_sample"].items():
        # --- dedup: drop repeat biopsies of the same patient ---
        if dd_field and str(row.get(dd_field, "")).strip().lower() != dd_keep.lower():
            rows.append(dict(sample_id=gsm_id, cohort=gse, stage=None,
                             resolution="duplicate_biopsy", raw_string=row.get(dd_field, ""),
                             etiology=meta.get("etiology"), platform=meta.get("platform"),
                             preservation=meta.get("preservation"), tier=meta["tier"],
                             repo=meta.get("repo", "GEO"), subject_id=gsm_id))
            continue

        # --- healthy override beats any fibrosis value ---
        if ov_field and str(row.get(ov_field, "")).strip().lower() in ov_values:
            lab = Label(None, RES_HEALTHY, str(row.get(ov_field, "")))
        else:
            lab = pick_stage(row, gse)

        rows.append(dict(
            sample_id=gsm_id, cohort=gse,
            stage=lab.stage, resolution=lab.resolution, raw_string=lab.raw,
            etiology=meta.get("etiology"), platform=meta.get("platform"),
            preservation=meta.get("preservation"), tier=meta["tier"],
            repo=meta.get("repo", "GEO"), subject_id=None,
        ))
    return pd.DataFrame(rows)


USABLE["duplicate_biopsy"] = (False, False, False)

labels_raw = pd.concat([build_labels_v3(g) for g in targets], ignore_index=True)
labels_raw = labels_raw[labels_raw.cohort.map(lambda c: tier_of(c) != "REJECTED")]

labels_raw["y_T1_significant"] = labels_raw.apply(f2plus, axis=1)
labels_raw["y_T3_early_adv"]   = labels_raw.apply(early_adv, axis=1)
labels_raw["y_T2_ordinal"]     = labels_raw.apply(ordinal, axis=1)

assert_no_healthy_as_F0(labels_raw)
labels_raw.to_parquet(PATHS["interim"] / "labels_raw.parquet", index=False)

# ---------------- reports ----------------
print("RESOLUTION BREAKDOWN (v3)")
print(pd.crosstab(labels_raw.cohort, labels_raw.resolution, margins=True))

print("\nSTAGE x COHORT")
print(pd.crosstab(labels_raw.cohort, labels_raw.stage.fillna("NONE"), margins=True))

summ = labels_raw.groupby("cohort").agg(
    n_gsm=("sample_id", "size"),
    healthy=("resolution", lambda s: (s == RES_HEALTHY).sum()),
    dup=("resolution", lambda s: (s == "duplicate_biopsy").sum()),
    unk=("resolution", lambda s: (s == RES_UNKNOWN).sum()),
    T1_usable=("y_T1_significant", lambda s: s.notna().sum()),
    T1_pos=("y_T1_significant", lambda s: (s == 1).sum()),
    T2_usable=("y_T2_ordinal", lambda s: s.notna().sum()),
    T3_usable=("y_T3_early_adv", lambda s: s.notna().sum()),
    T3_pos=("y_T3_early_adv", lambda s: (s == 1).sum()),
)
summ["T1_neg"] = summ.T1_usable - summ.T1_pos
summ["T1_minority"] = summ[["T1_pos", "T1_neg"]].min(axis=1)
summ["suggest"] = np.where(summ.T1_minority >= 15, "T1-eligible",
                   np.where(summ.T1_usable >= 20, "T2 (train-only)", "review"))
print("\nTASK USABILITY (v3)")
print(summ.to_string())

print("\nRETIER ACTIONS (registry says X, data says Y):")
for c, r in summ.iterrows():
    cur = tier_of(c)
    want = "T1" if r.suggest == "T1-eligible" else ("T2" if "T2" in r.suggest else "review")
    if cur in ("T1", "T2", "T3", "AUDIT") and cur != want and want != "review":
        print(f"  {c:12s} registry={cur:6s} -> data suggests {want}"
              f"  (usable={int(r.T1_usable)}, minority={int(r.T1_minority)})")

n_real = labels_raw[["y_T1_significant", "y_T3_early_adv"]].notna().any(axis=1).sum()
print(f"\nN_real (usable in at least one task): {n_real}")
print(f"  healthy excluded: {(labels_raw.resolution == RES_HEALTHY).sum()}")
print(f"  duplicate biopsies dropped: {(labels_raw.resolution == 'duplicate_biopsy').sum()}")

write_manifest("p03_labels", inputs={"registry": sha256_obj(COHORTS)},
               params={"parser": "v3", "exclude_healthy_obese": EXCLUDE_HEALTHY_OBESE},
               outputs=[PATHS["interim"] / "labels_raw.parquet"])

RESOLUTION BREAKDOWN (v3)
resolution  duplicate_biopsy  exact  grouped_F0_F1  grouped_F0_F2  grouped_F3_F4  healthy  unknown   All
cohort                                                                                                  
GSE130970                  0     78              0              0              0        0        0    78
GSE135251                  0    206              0              0              0       10        0   216
GSE159088                  0      0              0              0              0        0       80    80
GSE162694                  0    112              0              0              0       31        0   143
GSE163211                  0      0              0              0              0       76      242   318
GSE193066                 58    106              0              0              0        0        0   164
GSE207310                  0     30              0              0              0        0        0    30
GSE213621                  0 

{'stage': 'p03_labels',
 'timestamp': '2026-10-08T10:30:50',
 'inputs': {'registry': 'e9d9637d8c5c211a3dc40dc0745f54698d2c4aa398470d6a8e0e09fd84eeb449'},
 'params': {'parser': 'v3', 'exclude_healthy_obese': False},
 'outputs': {'/content/drive/MyDrive/CoDAGAT_preprocessing_fix_v1/interim/labels_raw.parquet': '6f25ba61c22d56c0a2238f01f5962da7db4fe5e24b28b4dc759686ea80f385b8'}}

In [14]:
# ============================================================
# CELL 12 — PARSER PATCH v3.1
# BUG: separator was optional, so bare "34" matched F3-F4 and
#      "01" matched F0-F1. Now require F-prefix OR a real separator.
# ALSO: T4 (NanoString) must never go through this parser.
# ============================================================

GROUPED_PATTERNS = [
    # F-prefixed: F0F1, F0-F1, F0_F1, f0 f1
    (re.compile(r"\bf\s*0\s*[-–—/_ ]*\s*f?\s*1\b", re.I), "F0", RES_G_F0F1),
    (re.compile(r"\bf\s*3\s*[-–—/_ ]*\s*f?\s*4\b", re.I), "F3", RES_G_F3F4),
    (re.compile(r"\bf\s*0\s*[-–—/_ ]*\s*f?\s*2\b", re.I), "F0", RES_G_F0F2),
    (re.compile(r"\bf\s*1\s*[-–—/_ ]*\s*f?\s*4\b", re.I), None, RES_UNKNOWN),
    # bare digits: EXPLICIT separator required
    (re.compile(r"\b0\s*[-–—/]\s*1\b"), "F0", RES_G_F0F1),
    (re.compile(r"\b3\s*[-–—/]\s*4\b"), "F3", RES_G_F3F4),
    (re.compile(r"\b0\s*[-–—/]\s*2\b"), "F0", RES_G_F0F2),
    (re.compile(r"\b1\s*[-–—/]\s*4\b"), None, RES_UNKNOWN),
    (re.compile(r"mild.*?stage\s*0\s*[-–—]\s*1", re.I),     "F0", RES_G_F0F1),
    (re.compile(r"advanced.*?stage\s*3\s*[-–—]\s*4", re.I), "F3", RES_G_F3F4),
]

def _test_patch():
    cases = [("34", None), ("age 34", None), ("01", None), ("1.34", None),
             ("52", None), ("F0F1", "F0"), ("F3F4", "F3"), ("F0-2", "F0"),
             ("f0-f1", "F0"), ("0-1", "F0"), ("3-4", "F3"),
             ("mild (fibrosis stage 0-1)", "F0"),
             ("advanced (fibrosis stage 3-4)", "F3"), ("NASH_F1_F4", None)]
    bad = []
    for v, exp in cases:
        got = next((s for p, s, r in GROUPED_PATTERNS if p.search(v.lower())), None)
        if got != exp:
            bad.append((v, exp, got))
    print(f"patch test: {len(cases)-len(bad)}/{len(cases)} passed")
    for b in bad:
        print("  FAIL", b)
    return not bad

assert _test_patch(), "Patch failed — STOP."
assert _test_parser_v3(), "Full parser suite failed — STOP."

# ---- T4 never enters the staging parser ----
PARSE_TARGETS = [g for g in targets if tier_of(g) != "T4"]
print(f"parsing {len(PARSE_TARGETS)} cohorts (T4 excluded: "
      f"{[g for g in targets if tier_of(g)=='T4']})")

labels_raw = pd.concat([build_labels_v3(g) for g in PARSE_TARGETS], ignore_index=True)
labels_raw = labels_raw[labels_raw.cohort.map(lambda c: tier_of(c) != "REJECTED")]
labels_raw["y_T1_significant"] = labels_raw.apply(f2plus, axis=1)
labels_raw["y_T3_early_adv"]   = labels_raw.apply(early_adv, axis=1)
labels_raw["y_T2_ordinal"]     = labels_raw.apply(ordinal, axis=1)
assert_no_healthy_as_F0(labels_raw)
labels_raw.to_parquet(PATHS["interim"] / "labels_raw.parquet", index=False)

print("\nSTAGE x COHORT (v3.1)")
print(pd.crosstab(labels_raw.cohort, labels_raw.stage.fillna("NONE"), margins=True))

summ = labels_raw.groupby("cohort").agg(
    n_gsm=("sample_id","size"),
    healthy=("resolution", lambda s:(s==RES_HEALTHY).sum()),
    dup=("resolution", lambda s:(s=="duplicate_biopsy").sum()),
    unk=("resolution", lambda s:(s==RES_UNKNOWN).sum()),
    T1_usable=("y_T1_significant", lambda s:s.notna().sum()),
    T1_pos=("y_T1_significant", lambda s:(s==1).sum()),
    T2_usable=("y_T2_ordinal", lambda s:s.notna().sum()),
    T3_usable=("y_T3_early_adv", lambda s:s.notna().sum()),
    T3_pos=("y_T3_early_adv", lambda s:(s==1).sum()))
summ["T1_neg"] = summ.T1_usable - summ.T1_pos
summ["T1_minority"] = summ[["T1_pos","T1_neg"]].min(axis=1)
print("\nTASK USABILITY (v3.1)")
print(summ.to_string())

print("\nSANITY vs published Table I:")
for c, exp in [("GSE84044","43/20/33/18/10"), ("GSE135251","38/47/53/54/14"),
               ("GSE162694","35/30/27/8/12"), ("GSE130970","25/28/9/14/2")]:
    d = labels_raw[labels_raw.cohort==c].stage.value_counts()
    got = "/".join(str(int(d.get(f"F{i}",0))) for i in range(5))
    print(f"  {c:12s} expect {exp:16s} got {got:16s} {'OK' if got==exp else '<<< MISMATCH'}")

patch test: 14/14 passed
parser v3 self-test: 24/24 passed
parsing 13 cohorts (T4 excluded: ['GSE163211'])

STAGE x COHORT (v3.1)
stage       F0   F1   F2   F3   F4  NONE   All
cohort                                        
GSE130970   25   28    9   14    2     0    78
GSE135251   38   47   53   54   14    10   216
GSE159088    0    0    0    0    0    80    80
GSE162694   35   30   27    8   12    31   143
GSE193066    6   37   37   26    0    58   164
GSE207310    7   15    7    1    0     0    30
GSE213621   97    0  107   95    0    69   368
GSE240729    9   17   25   10    6     0    67
GSE276114   39    0    0   24  114     0   177
GSE31803    40    0    0   32    0    59   131
GSE48452    37   16    0    2    1    17    73
GSE84044    43   20   33   18   10     0   124
GSE89632    21    8    2    4    4    24    63
All        397  218  300  288  163   348  1714

TASK USABILITY (v3.1)
           n_gsm  healthy  dup  unk  T1_usable  T1_pos  T2_usable  T3_usable  T3_pos  T1_neg  T

In [15]:
# ============================================================
# CELL 13 — TIER PROMOTION + F2 AUDIT + LABEL-SHIFT TABLE
# Tier is now set by DATA, not by the hand-written guess.
# Rule: minority class >= 15 under F2+ -> T1. Else T2.
# ============================================================

MIN_MINORITY_T1 = 15

PROMOTIONS = {
    "GSE213621": ("T3", "T1", "192 samples recovered (F0F1/F3F4); 97 neg / 202 pos"),
    "GSE193066": ("AUDIT","T1", "106 baseline biopsies; 43 neg / 63 pos. NOTE: PLS high-risk enriched"),
    "GSE240729": ("T2", "T1", "26 neg / 41 pos — binary-evaluable"),
    "GSE207310": ("AUDIT","T2", "only 8 positives"),
    "GSE48452":  ("T1", "T2", "3 positives under F2+ — not binary-evaluable"),
    "GSE89632":  ("T1", "T2", "10 positives under F2+ — below threshold"),
    "GSE159088": ("AUDIT","REJECTED", "no fibrosis staging in metadata (80/80 unknown)"),
}

for gse,(old,new,why) in PROMOTIONS.items():
    assert COHORTS[gse]["tier"] == old, f"{gse}: expected {old}, found {COHORTS[gse]['tier']}"
    COHORTS[gse]["tier"] = new
    COHORTS[gse]["retier_reason"] = why
    print(f"  {gse:12s} {old:6s} -> {new:8s}  {why}")

labels_raw["tier"] = labels_raw.cohort.map(tier_of)
labels_raw = labels_raw[labels_raw.tier != "REJECTED"]
labels_raw.to_parquet(PATHS["interim"] / "labels_raw.parquet", index=False)
(PATHS["config"] / "cohorts.json").write_text(json.dumps(COHORTS, indent=2))

# ---------------- F2+ audit ----------------
aud = (labels_raw.dropna(subset=["y_T1_significant"])
       .groupby(["tier","cohort"])["y_T1_significant"]
       .agg(n="size", pos="sum").reset_index())
aud["neg"] = aud.n - aud.pos
aud["pos_pct"] = 100*aud.pos/aud.n
aud = aud.sort_values(["tier","n"], ascending=[True,False])
print("\nF2+ AUDIT\n", aud.to_string(index=False))

# ---------------- per-fold label shift (JS) ----------------
def js_div(p,q):
    p,q = np.asarray(p,float), np.asarray(q,float); m=(p+q)/2
    kl = lambda a,b: np.sum([x*np.log2(x/y) for x,y in zip(a,b) if x>0])
    return 0.5*kl(p,m)+0.5*kl(q,m)

pool = labels_raw.dropna(subset=["y_T1_significant"])
shift = []
for f in aud.loc[aud.tier=="T1","cohort"]:
    te = pool[pool.cohort==f].y_T1_significant
    tr = pool[pool.cohort!=f].y_T1_significant
    mt = [1-te.mean(), te.mean()]; mr = [1-tr.mean(), tr.mean()]
    shift.append(dict(fold=f, n=len(te), test_pos=mt[1], train_pos=mr[1],
                      JS=js_div(mt,mr)))
shift = pd.DataFrame(shift).sort_values("JS", ascending=False)
print("\nLABEL SHIFT PER FOLD (independent variable of the dose-response experiment)")
print(shift.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
shift.to_csv(PATHS["reports"]/"label_shift_per_fold.csv", index=False)

# ---------------- cohort-label confounding ----------------
from sklearn.metrics import normalized_mutual_info_score as nmi
print(f"\nNMI(cohort, F2+ label) = "
      f"{nmi(pool.cohort, pool.y_T1_significant.astype(int)):.3f}")
print(f"JS range = {shift.JS.min():.4f} to {shift.JS.max():.4f} "
      f"({shift.JS.max()/shift.JS.min():.0f}x)")

print(f"\nT1 folds: {len(shift)}   T1 samples: {int(shift.n.sum())}")
print(f"N_real (usable in >=1 task): "
      f"{labels_raw[['y_T1_significant','y_T3_early_adv']].notna().any(axis=1).sum()}")

write_manifest("p03b_tiers", inputs={"labels": "labels_raw.parquet"},
               params={"min_minority_T1": MIN_MINORITY_T1, "promotions": PROMOTIONS},
               outputs=[PATHS["interim"]/"labels_raw.parquet",
                        PATHS["reports"]/"label_shift_per_fold.csv"])

  GSE213621    T3     -> T1        192 samples recovered (F0F1/F3F4); 97 neg / 202 pos
  GSE193066    AUDIT  -> T1        106 baseline biopsies; 43 neg / 63 pos. NOTE: PLS high-risk enriched
  GSE240729    T2     -> T1        26 neg / 41 pos — binary-evaluable
  GSE207310    AUDIT  -> T2        only 8 positives
  GSE48452     T1     -> T2        3 positives under F2+ — not binary-evaluable
  GSE89632     T1     -> T2        10 positives under F2+ — below threshold
  GSE159088    AUDIT  -> REJECTED  no fibrosis staging in metadata (80/80 unknown)

F2+ AUDIT
 tier    cohort   n   pos  neg    pos_pct
  T1 GSE213621 299 202.0 97.0  67.558528
  T1 GSE135251 206 121.0 85.0  58.737864
  T1  GSE84044 124  61.0 63.0  49.193548
  T1 GSE162694 112  47.0 65.0  41.964286
  T1 GSE193066 106  63.0 43.0  59.433962
  T1 GSE130970  78  25.0 53.0  32.051282
  T1  GSE31803  72  32.0 40.0  44.444444
  T1 GSE240729  67  41.0 26.0  61.194030
  T2 GSE276114 138 138.0  0.0 100.000000
  T2  GSE48452  56   3.0 5

{'stage': 'p03b_tiers',
 'timestamp': '2026-10-08T10:30:50',
 'inputs': {'labels': 'labels_raw.parquet'},
 'params': {'min_minority_T1': 15,
  'promotions': {'GSE213621': ('T3',
    'T1',
    '192 samples recovered (F0F1/F3F4); 97 neg / 202 pos'),
   'GSE193066': ('AUDIT',
    'T1',
    '106 baseline biopsies; 43 neg / 63 pos. NOTE: PLS high-risk enriched'),
   'GSE240729': ('T2', 'T1', '26 neg / 41 pos — binary-evaluable'),
   'GSE207310': ('AUDIT', 'T2', 'only 8 positives'),
   'GSE48452': ('T1', 'T2', '3 positives under F2+ — not binary-evaluable'),
   'GSE89632': ('T1', 'T2', '10 positives under F2+ — below threshold'),
   'GSE159088': ('AUDIT',
    'REJECTED',
    'no fibrosis staging in metadata (80/80 unknown)')}},
 'outputs': {'/content/drive/MyDrive/CoDAGAT_preprocessing_fix_v1/interim/labels_raw.parquet': 'c4a39bd91cc02de8991110cfe2d3d68046a5461bac463d8ac493deea6e6cfbc0',
  '/content/drive/MyDrive/CoDAGAT_preprocessing_fix_v1/reports/label_shift_per_fold.csv': 'aa361607e0719d

In [16]:
# ============================================================
# CELL 14 — ARRAYEXPRESS SDRF -> LABELS
# SDRF is a TSV, one row per assay. Fibrosis stage sits in a
# "Characteristics[...]" or "Factor Value[...]" column whose exact
# name varies per study. We discover, then parse with the SAME
# parser used for GEO — no separate code path, no separate bugs.
# ============================================================

def read_sdrf(acc):
    d = PATHS["raw_geo"] / acc
    cands = sorted(d.glob("*.sdrf.txt")) + sorted(d.glob("*sdrf*"))
    if not cands:
        print(f"  {acc}: NO SDRF FOUND in {d}")
        return None
    df = pd.read_csv(cands[0], sep="\t", dtype=str)
    print(f"  {acc}: {cands[0].name}  shape={df.shape}")
    return df

def sdrf_key(col):
    """'Characteristics[fibrosis stage]' -> 'fibrosis stage'"""
    m = re.search(r"\[(.+?)\]", col)
    return (m.group(1) if m else col).strip().lower()

AE_LABEL_FRAMES = {}
for acc in [g for g, m in COHORTS.items()
            if m.get("repo") == "ArrayExpress" and m["tier"] in ("T1","T2","T3")]:
    sdrf = read_sdrf(acc)
    if sdrf is None:
        continue

    # ---- show every candidate column so you can eyeball it ----
    cand = [c for c in sdrf.columns
            if c.lower().startswith(("characteristics", "factor value"))]
    print(f"\n  [{acc}] candidate annotation columns:")
    for c in cand:
        vals = sdrf[c].dropna().value_counts()
        flag = " <<< POSSIBLE STAGING" if any(
            s in sdrf_key(c) for s in ("fibro","stage","metavir","ishak",
                                        "kleiner","scheuer","grade","disease")) else ""
        print(f"      {sdrf_key(c):32s} {dict(list(vals.items())[:6])}{flag}")

    # ---- sample identifier column ----
    id_col = next((c for c in ["Source Name","Assay Name","Comment[ENA_RUN]",
                               "Scan Name","Extract Name"] if c in sdrf.columns), None)
    if id_col is None:
        print(f"  {acc}: no usable ID column, columns={list(sdrf.columns)[:12]}")
        continue

    rows = []
    meta = COHORTS[acc]
    for _, r in sdrf.iterrows():
        row_dict = {sdrf_key(c): r[c] for c in cand if pd.notna(r[c])}
        lab = pick_stage(row_dict, acc)
        rows.append(dict(
            sample_id=str(r[id_col]).strip(), cohort=acc,
            stage=lab.stage, resolution=lab.resolution, raw_string=lab.raw,
            etiology=meta.get("etiology"), platform=meta.get("platform"),
            preservation=meta.get("preservation"), tier=meta["tier"],
            repo="ArrayExpress", subject_id=None))
    f = pd.DataFrame(rows).drop_duplicates(subset="sample_id")
    AE_LABEL_FRAMES[acc] = f
    print(f"  {acc}: {len(f)} unique assays  ->  "
          f"{f.resolution.value_counts().to_dict()}")
    print(f"      stages: {f.stage.value_counts().to_dict()}")

# ---- merge into labels_raw ----
if AE_LABEL_FRAMES:
    ae = pd.concat(AE_LABEL_FRAMES.values(), ignore_index=True)
    ae["y_T1_significant"] = ae.apply(f2plus, axis=1)
    ae["y_T3_early_adv"]   = ae.apply(early_adv, axis=1)
    ae["y_T2_ordinal"]     = ae.apply(ordinal, axis=1)

    labels_raw = pd.concat(
        [labels_raw[~labels_raw.cohort.isin(AE_LABEL_FRAMES)], ae],
        ignore_index=True)
    assert_no_healthy_as_F0(labels_raw)
    labels_raw.to_parquet(PATHS["interim"] / "labels_raw.parquet", index=False)

    print("\nSTAGE x COHORT (with ArrayExpress)")
    print(pd.crosstab(labels_raw.cohort, labels_raw.stage.fillna("NONE"), margins=True))
    print(f"\nN_real: "
          f"{labels_raw[['y_T1_significant','y_T3_early_adv']].notna().any(axis=1).sum()}")
    print("\nEXPECT E-MTAB-6863 ~ F0=12 F1=15 F2=15 F3=14 F4=13")
else:
    print("\nNo ArrayExpress SDRF parsed — check Cell 6 downloaded the .sdrf.txt files.")

  E-MTAB-6863: E-MTAB-6863.sdrf.txt  shape=(69, 68)

  [E-MTAB-6863] candidate annotation columns:
      organism                         {'Homo sapiens': 69}
      age                              {'45': 5, '53': 4, '49': 3, '51': 3, '50': 3, '29': 3}
      sex                              {'male': 50, 'female': 19}
      organism part                    {'liver': 69}
      disease                          {'liver disease': 55, 'non-alcoholic fatty liver disease': 12, 'alcoholic liver disease': 2} <<< POSSIBLE STAGING
      disease staging                  {'early': 42, 'advanced': 27} <<< POSSIBLE STAGING
      infect                           {'Hepatitis C virus genotype 1': 28, 'Hepatitis C virus genotype 3': 21, 'none': 14, 'Hepacivirus C': 3, 'Hepatitis C virus genotype 2': 2, 'Hepatitis C virus genotype 4': 1}
      clinical information             {'steatosis': 43, 'no steatosis': 26}
      fibrosis stage                   {'METAVIR fibrosis score 2': 15, 'METAVIR fibrosis scor

In [17]:
# ============================================================
# CELL 14b — LOADERS, GPL MAPPING, AND SAMPLE-ID MATCHING
# Run after Cells 2 and 5, before Cell 15.
# _clean_symbol() is defined in Cell 15 before _gpl_map() is used.
# ============================================================
%pip install -q GEOparse

import GEOparse
print("GEOparse imported successfully")
import re
from pathlib import Path
from collections import defaultdict
from urllib.parse import urlsplit

import GEOparse
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

_GPL_CACHE = globals().get("_GPL_CACHE", {})

# Create the cache directory without deleting existing files.
cache = PATHS["interim"] / "expr_cache"
cache.mkdir(parents=True, exist_ok=True)


# ---------- GPL annotation tables ----------

def _gpl_tables(gse):
    """Return all non-empty GPL tables for a series."""
    key = f"{gse}::all"

    if key in _GPL_CACHE:
        return _GPL_CACHE[key]

    out = []

    try:
        g = GEOparse.get_GEO(
            geo=gse,
            destdir=str(PATHS["raw_geo"] / gse),
            silent=True,
            include_data=False,
        )

        for gpl_id, gpl in g.gpls.items():
            table = getattr(gpl, "table", None)

            if (
                table is not None
                and len(table.columns) > 0
                and len(table) > 0
            ):
                out.append((gpl_id, table))

    except Exception as exc:
        print(f"      GPL load failed {gse}: {exc}")
        return []

    # Do not cache failed/empty retrievals.
    if out:
        _GPL_CACHE[key] = out

    return out


def _gpl_map(ids, gse):
    tables = _gpl_tables(gse)

    if not tables:
        return {}

    ids_s = set(map(str, ids))
    prefer = [
        "gene symbol", "gene_symbol", "symbol", "genesymbol",
        "ilmn_gene", "gene", "gene_assignment", "gene assignment",
    ]

    best_map = {}
    best_cov = 0.0
    best_desc = None

    for gpl_id, table in tables:
        id_col = "ID" if "ID" in table.columns else table.columns[0]
        cols = {str(c).strip().lower(): c for c in table.columns}

        for candidate in prefer:
            if candidate not in cols:
                continue

            mapping = {
                str(k): _clean_symbol(v)
                for k, v in zip(table[id_col], table[cols[candidate]])
            }
            mapping = {k: v for k, v in mapping.items() if v}

            coverage = len(ids_s & set(mapping)) / max(len(ids_s), 1)

            if coverage > best_cov:
                best_map = mapping
                best_cov = coverage
                best_desc = f"{gpl_id}:{cols[candidate]}"

    if best_desc:
        print(
            f"      GPL {best_desc} -> {len(best_map)} probes, "
            f"coverage {best_cov:.1%}"
        )

    return best_map


# ---------- Sample-ID matching ----------

def _tokens(value):
    return {
        token
        for token in re.split(r"[^A-Za-z0-9]+", str(value))
        if len(token) >= 4
    }


def build_gsm_token_index(gse):
    """Retain exact metadata aliases only when they identify one GSM."""
    idx = defaultdict(set)
    exact_candidates = defaultdict(set)

    try:
        g = GEOparse.get_GEO(
            geo=gse,
            destdir=str(PATHS["raw_geo"] / gse),
            silent=True,
            include_data=False,
        )
    except Exception as exc:
        print(f"      Sample metadata failed {gse}: {exc}")
        return idx, {}

    for gsm_id, gsm in g.gsms.items():
        for key in (
            "title", "description", "supplementary_file_1",
            "supplementary_file", "source_name_ch1",
        ):
            values = gsm.metadata.get(key, [])

            if isinstance(values, str):
                values = [values]

            for value in values:
                value = str(value).strip()

                if not value:
                    continue

                for alias in (
                    value,
                    Path(value).name,
                    Path(value).stem,
                ):
                    exact_candidates[alias].add(gsm_id)

                for token in _tokens(value):
                    idx[token].add(gsm_id)

    exact = {
        alias: next(iter(sample_ids))
        for alias, sample_ids in exact_candidates.items()
        if len(sample_ids) == 1
    }

    return idx, exact


def reconcile_ids(expr_cols, want_ids, gse):
    # Preserve original column names as mapping keys.
    original = list(expr_cols)
    wanted = [str(value).strip() for value in want_ids]
    want = set(wanted)

    if not want:
        return {}, "FAILED: no requested sample IDs"

    if len(wanted) != len(want):
        raise ValueError(f"{gse}: repeated requested sample IDs")

    if not pd.Index(original).is_unique:
        raise ValueError(f"{gse}: duplicate expression column names")

    cleaned = {column: str(column).strip() for column in original}

    def acceptable(mapping):
        targets = list(mapping.values())

        if len(targets) != len(set(targets)):
            raise ValueError(
                f"{gse}: multiple expression columns map to one GSM"
            )

        return len(targets) >= 0.9 * len(want)

    mapping = {
        column: value
        for column, value in cleaned.items()
        if value in want
    }

    if acceptable(mapping):
        return mapping, "direct_gsm"

    mapping = {}

    for column, value in cleaned.items():
        hits = set(re.findall(r"GSM\d+", value)) & want

        if len(hits) == 1:
            mapping[column] = next(iter(hits))

    if acceptable(mapping):
        return mapping, "embedded_gsm"

    idx, exact = build_gsm_token_index(gse)

    mapping = {
        column: exact[value]
        for column, value in cleaned.items()
        if value in exact and exact[value] in want
    }

    if acceptable(mapping):
        return mapping, "exact_metadata_string"

    mapping = {}
    ambiguous = 0

    for column, value in cleaned.items():
        hits = set()

        for token in _tokens(value):
            if token in idx and len(idx[token]) == 1:
                hits.update(idx[token])

        hits &= want

        if len(hits) == 1:
            mapping[column] = next(iter(hits))
        elif len(hits) > 1:
            ambiguous += 1

    if acceptable(mapping):
        print(
            f"      token match: {len(mapping)}/{len(want)} "
            f"({ambiguous} ambiguous)"
        )
        return mapping, "token_match_VERIFIED"

    if len(original) == len(wanted):
        return {}, (
            "FAILED_ID_MATCH: equal counts do not prove sample order"
        )

    return {}, f"FAILED(cols={len(original)}, want={len(want)})"


def apply_sample_mapping(df, cmap, cohort):
    if not df.columns.is_unique:
        raise ValueError(
            f"{cohort}: duplicate source expression column names"
        )

    if not cmap:
        raise ValueError(f"{cohort}: no sample mapping available")

    target_ids = pd.Index(list(cmap.values()))

    if target_ids.hasnans or not target_ids.is_unique:
        raise ValueError(
            f"{cohort}: missing or repeated target sample IDs"
        )

    missing = [column for column in cmap if column not in df.columns]

    if missing:
        raise ValueError(
            f"{cohort}: mapping references absent columns: {missing[:5]}"
        )

    out = df.loc[:, list(cmap)].rename(columns=cmap)

    if not out.columns.equals(target_ids):
        raise ValueError(f"{cohort}: mapped sample order is incorrect")

    return out


# ---------- Supplementary matrix loader ----------

def load_supp_matrix(gse):
    directory = PATHS["raw_geo"] / gse

    if not directory.exists():
        return None, "no_dir"

    files = [
        path
        for path in directory.iterdir()
        if path.is_file()
        and path.suffix in (".gz", ".txt", ".tsv", ".csv")
        and not path.name.endswith(
            ("soft.gz", "family.soft.gz", ".sdrf.txt", ".idf.txt")
        )
    ]

    for path in sorted(files, key=lambda p: -p.stat().st_size):
        for sep in ["\t", ","]:
            try:
                df = pd.read_csv(
                    path,
                    sep=sep,
                    index_col=0,
                    low_memory=False,
                    compression="gzip" if path.name.endswith(".gz") else None,
                )
                numeric = df.select_dtypes(include=[np.number])

                if numeric.shape[0] > 5000 and numeric.shape[1] >= 10:
                    return numeric, path.name

            except Exception:
                continue

    return None, "no_supp_matrix"


# ---------- GEO sample table loader ----------

def load_series_matrix(gse):
    try:
        g = GEOparse.get_GEO(
            geo=gse,
            destdir=str(PATHS["raw_geo"] / gse),
            silent=True,
        )

        df = g.pivot_samples("VALUE").apply(
            pd.to_numeric, errors="coerce"
        )
        return df, "series_matrix"

    except Exception as exc:
        return None, f"series_matrix_failed: {exc}"


# ---------- GCT loader ----------

def load_gct(gse):
    directory = PATHS["raw_geo"] / gse

    for path in sorted(
        directory.glob("*.gct*"),
        key=lambda p: -p.stat().st_size,
    ):
        try:
            df = pd.read_csv(
                path,
                sep="\t",
                skiprows=2,
                low_memory=False,
            )

            name = "Name" if "Name" in df.columns else df.columns[0]
            out = df.set_index(name).drop(
                columns=["Description"],
                errors="ignore",
            )

            return (
                out.apply(pd.to_numeric, errors="coerce"),
                f"gct:{path.name}",
            )

        except Exception as exc:
            print(f"      GCT parse failed {path.name}: {exc}")

    return None, "no_gct"


# ---------- Per-GSM supplementary file loader ----------

def load_per_gsm(gse, max_workers=8):
    try:
        g = GEOparse.get_GEO(
            geo=gse,
            destdir=str(PATHS["raw_geo"] / gse),
            silent=True,
            include_data=False,
        )
    except Exception as exc:
        return None, f"per_gsm_soft_failed: {exc}"

    urls = {}

    for gsm_id, gsm in g.gsms.items():
        for key in ("supplementary_file_1", "supplementary_file"):
            values = gsm.metadata.get(key, [])

            if isinstance(values, str):
                values = [values]

            for value in values:
                value = str(value).strip()

                if not value or value.upper() == "NONE":
                    continue

                if value.startswith("//"):
                    value = "https:" + value

                value = value.replace("ftp://", "https://")

                if re.search(
                    r"\.(txt|tsv|csv|counts)(\.gz)?$",
                    urlsplit(value).path,
                    re.I,
                ):
                    urls[gsm_id] = value
                    break

    if len(urls) < 5:
        return None, f"per_gsm_no_files (found {len(urls)})"

    outdir = PATHS["raw_geo"] / gse / "per_gsm"
    outdir.mkdir(parents=True, exist_ok=True)

    series = {}
    failed = 0

    for gsm_id, url in tqdm(
        urls.items(),
        desc=f"  {gse} per-GSM",
        leave=False,
    ):
        filename = Path(urlsplit(url).path).name
        dest = outdir / gsm_id / filename
        dest.parent.mkdir(parents=True, exist_ok=True)

        path, status = download_file(
            url,
            dest,
            retries=2,
            timeout=90,
        )

        if path is None:
            failed += 1
            continue

        try:
            compression = "gzip" if path.name.endswith(".gz") else None
            table = None

            for sep in ["\t", ","]:
                candidate = pd.read_csv(
                    path,
                    sep=sep,
                    header=None,
                    comment="#",
                    compression=compression,
                )

                if candidate.shape[1] >= 2:
                    table = candidate
                    break

            if table is None:
                raise ValueError("No two-column expression table found")

            if not pd.api.types.is_numeric_dtype(table.iloc[:, 1]):
                table = pd.read_csv(
                    path,
                    sep=sep,
                    comment="#",
                    compression=compression,
                )

            table = table.iloc[:, :2].copy()
            table.columns = ["gene", gsm_id]

            table = table[
                ~table["gene"].astype(str).str.startswith(
                    ("__", "N_", "no_feature")
                )
            ]

            if table["gene"].isna().any() or table["gene"].duplicated().any():
                raise ValueError("Missing or duplicate gene identifiers")

            series[gsm_id] = pd.to_numeric(
                table.set_index("gene")[gsm_id],
                errors="raise",
            )

        except Exception as exc:
            failed += 1
            print(f"      {gsm_id}: parse failed: {exc}")

    if len(series) < 5:
        return None, f"per_gsm_parse_failed ({failed} failures)"

    if failed:
        print(f"      {gse}: {failed} per-GSM files failed")

    return pd.DataFrame(series), f"per_gsm({len(series)} files)"


# ---------- Orchestrated loader ----------

def load_expression(gse):
    for loader in (
        load_supp_matrix,
        load_gct,
        load_series_matrix,
        load_per_gsm,
    ):
        df, source = loader(gse)

        if (
            df is not None
            and df.shape[0] > 3000
            and df.shape[1] >= 10
        ):
            return df, source

    return None, "all_loaders_failed"


print("Cell 14b loaded: loaders, GPL mapping, and sample-ID helpers ready.")

GEOparse imported successfully
Cell 14b loaded: loaders, GPL mapping, and sample-ID helpers ready.


In [25]:
# ============================================================
# CELL 15 — EXPRESSION LOADING  (v3)
# ============================================================
try:
    import mygene
except ModuleNotFoundError:
    !pip install -q mygene
    import mygene

import logging
assert "PATHS" in globals(), "PATHS undefined — run Cell 2 first"
assert "labels_raw" in globals(), "labels_raw undefined — run Cells 7-13 first"
assert "load_supp_matrix" in globals(), "loaders undefined — run Cell 14b first"
assert "_GPL_CACHE" in globals() or True

MG = mygene.MyGeneInfo()
CACHE = PATHS["interim"] / "expr_cache"; CACHE.mkdir(parents=True, exist_ok=True)
logging.getLogger("biothings.client").setLevel(logging.WARNING)
_GPL_CACHE = globals().get("_GPL_CACHE", {})


def _safe_mg(ids, scope):
    try:
        res = MG.querymany(list(ids), scopes=scope, fields="symbol",
                           species="human", as_dataframe=True, df_index=False,
                           verbose=False)
    except Exception as e:
        print(f"      mygene({scope}) failed: {e}")
        return {}
    if not isinstance(res, pd.DataFrame) or "symbol" not in res.columns:
        return {}
    res = res[res.symbol.notna()].drop_duplicates("query")
    return dict(zip(res["query"].astype(str), res["symbol"].astype(str)))


def _gpl_table(gse):
    if gse in _GPL_CACHE:
        return _GPL_CACHE[gse]
    try:
        g = GEOparse.get_GEO(geo=gse, destdir=str(PATHS["raw_geo"]/gse),
                             silent=True, include_data=False)
        tbl = list(g.gpls.values())[0].table
    except Exception as e:
        print(f"      GPL load failed for {gse}: {e}")
        tbl = None
    _GPL_CACHE[gse] = tbl
    return tbl


def _clean_symbol(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    if s.upper() in ("---", "NA", "NAN", ""):
        return None
    if "///" in s:
        parts = [p.strip() for p in s.split("///") if p.strip()]
        s = parts[0] if parts else ""
    elif "//" in s:
        parts = [p.strip() for p in s.split("//") if p.strip()]
        toks = [p for p in parts if re.fullmatch(r"[A-Za-z0-9\-\.@_]+", p)]
        s = toks[1] if len(toks) > 1 else (toks[0] if toks else "")
    elif "|" in s:
        s = s.split("|")[0].strip()
    s = s.strip()
    return s if s and s.upper() not in ("---", "NA", "NAN") else None





def map_ids_to_symbols(ids, gse):
    ids = [str(i).strip() for i in ids]
    base = [i.split(".")[0] for i in ids]
    strategies = {}

    sym_like = np.mean([bool(re.fullmatch(r"[A-Z][A-Z0-9\-\.@]{1,14}", i))
                        for i in ids[:500]])
    if sym_like > 0.5:
        strategies["already_symbol"] = ({i: i for i in ids}, sym_like)

    if not re.match(r"^ENSG\d+", ids[0] or ""):
        m = _gpl_map(ids, gse)
        if m:
            strategies["gpl"] = (m, len(set(ids) & set(m)) / len(ids))

    if np.mean([bool(re.match(r"^ENSG\d+", i)) for i in ids[:500]]) > 0.3:
        m = _safe_mg(set(base), "ensembl.gene")
        if m:
            strategies["ensembl"] = ({i: m[b] for i, b in zip(ids, base) if b in m},
                                     len(m) / len(set(base)))

    if np.mean([bool(re.fullmatch(r"\d+", i)) for i in ids[:500]]) > 0.5 \
       and strategies.get("gpl", ({}, 0))[1] < 0.5:
        m = _safe_mg(set(ids), "entrezgene")
        if m:
            strategies["entrez"] = (m, len(m) / len(set(ids)))

    if not strategies:
        return [None] * len(ids), "unmapped(0%)"

    how, (mapping, cov) = max(strategies.items(), key=lambda kv: kv[1][1])
    print(f"      symbol strategies: "
          f"{ {k: f'{v[1]:.1%}' for k, v in strategies.items()} }  -> {how}")
    return [mapping.get(i) for i in ids], f"{how}({cov:.0%})"


# ---------------- run, with per-cohort cache ----------------
EXPR, LOAD_LOG = {}, {}
geo_targets = [g for g in labels_raw.cohort.unique()
               if COHORTS[g].get("repo", "GEO") == "GEO"
               and tier_of(g) in ("T1", "T2", "T3")]
print(f"targets ({len(geo_targets)}): {sorted(geo_targets)}\n")

FORCE_RELOAD = []

for gse in geo_targets:
    cf, lf = CACHE / f"{gse}.parquet", CACHE / f"{gse}.json"
    if cf.exists() and lf.exists() and gse not in FORCE_RELOAD:
        EXPR[gse] = pd.read_parquet(cf)
        LOAD_LOG[gse] = json.loads(lf.read_text())
        print(f"[cache] {gse}: {EXPR[gse].shape}")
        continue

    print(f"\n[load] {gse}")
    log = {"gse": gse}
    try:
        df, src = load_supp_matrix(gse)
        if df is None:
            df, src = load_series_matrix(gse)
        if df is None:
            log.update(status="FAILED", source=src)
            print(f"      FAILED: {src}")
            LOAD_LOG[gse] = log
            continue
        log["source"], log["raw_shape"] = src, list(df.shape)

        want = labels_raw.loc[labels_raw.cohort == gse, "sample_id"].tolist()
        cmap, method = reconcile_ids(df.columns, want, gse)
        log["id_method"], log["matched"] = method, len(cmap)
        if not cmap:
            log["status"] = "FAILED_ID_MATCH"
            print("      FAILED: no sample-ID match")
            LOAD_LOG[gse] = log
            continue
        df = apply_sample_mapping(df, cmap, gse)

        syms, how = map_ids_to_symbols(df.index, gse)
        log["symbol_method"] = how
        df.index = syms
        df = df[df.index.notna()]
        if df.shape[0] < 3000:
            log.update(status="FAILED_SYMBOL_MAP", genes=int(df.shape[0]))
            print(f"      FAILED: only {df.shape[0]} genes after mapping")
            LOAD_LOG[gse] = log
            continue
        df = df.groupby(level=0).median()

        pos = df.values[np.isfinite(df.values) & (df.values > 0)]
        med = float(np.median(pos)) if pos.size else 0.0
        log["median_before"] = round(med, 3)
        if med > 50:
            df = np.log2(df.clip(lower=0) + 1); log["log2_applied"] = True
        else:
            log["log2_applied"] = False

        log.update(status="OK", genes=int(df.shape[0]), samples=int(df.shape[1]),
                   median_after=round(float(np.nanmedian(df.values)), 3))
        EXPR[gse] = df
        df.to_parquet(cf)
        lf.write_text(json.dumps(log, indent=2))
        print(f"      OK: {df.shape[0]} genes x {df.shape[1]} samples")

    except NameError as e:
        raise RuntimeError(
            f"{gse}: loader function missing ({e}). Run Cell 14b FIRST. "
            f"This exact error, silently logged, produced the 4-cohort run.")
    except Exception as e:
        import traceback; traceback.print_exc()
        log.update(status=f"EXCEPTION: {type(e).__name__}: {e}")

    LOAD_LOG[gse] = log

# ---------------- summary ----------------
print("\n" + "=" * 100)
print(f"{'cohort':12s} {'status':20s} {'genes':>6s} {'smp':>4s} {'want':>4s} "
      f"{'median':>7s} {'log2':>5s}  {'id_method':22s} symbol_method")
for gse in geo_targets:
    l = LOAD_LOG.get(gse, {})
    want = (labels_raw.cohort == gse).sum()
    print(f"{gse:12s} {str(l.get('status','?'))[:20]:20s} {l.get('genes',0):6d} "
          f"{l.get('samples',0):4d} {want:4d} {l.get('median_after',0):7.2f} "
          f"{str(l.get('log2_applied','')):5s}  {str(l.get('id_method',''))[:22]:22s} "
          f"{l.get('symbol_method','')}")

(PATHS["reports"]/"expression_load_log.json").write_text(json.dumps(LOAD_LOG, indent=2))

ok  = [g for g, l in LOAD_LOG.items() if l.get("status") == "OK"]
bad = [g for g in geo_targets if g not in ok]
nameerr = [g for g, l in LOAD_LOG.items() if "NameError" in str(l.get("status", ""))]

print(f"\nOK {len(ok)}/{len(geo_targets)}: {sorted(ok)}")
print(f"needs repair by Cells 17/19/24: {sorted(bad)}")
assert not nameerr, f"NameError for {nameerr} — Cell 14b did not run"
assert len(ok) >= 5, (
    f"only {len(ok)} cohorts loaded — expected ~6 at this stage. "
    f"Check that Cell 14b ran and the cache was cleared.")
print("\nProceed to Cell 17 (retry) — it repairs the rest.")

targets (12): ['GSE130970', 'GSE135251', 'GSE162694', 'GSE193066', 'GSE207310', 'GSE213621', 'GSE240729', 'GSE276114', 'GSE31803', 'GSE48452', 'GSE84044', 'GSE89632']


[load] GSE135251
      FAILED: series_matrix_failed: 'ID_REF'

[load] GSE84044
      GPL GPL570:Gene Symbol -> 45782 probes, coverage 83.7%
      symbol strategies: {'gpl': '83.7%'}  -> gpl
      OK: 22880 genes x 124 samples

[load] GSE162694
      token match: 143/143 (0 ambiguous)
      symbol strategies: {'already_symbol': '100.0%', 'ensembl': '78.4%'}  -> already_symbol
      OK: 31683 genes x 143 samples

[load] GSE130970
      symbol strategies: {'entrez': '99.2%'}  -> entrez
      OK: 19426 genes x 78 samples

[load] GSE89632
      GPL GPL14951:Symbol -> 29377 probes, coverage 100.0%
      symbol strategies: {'gpl': '100.0%'}  -> gpl
      OK: 20819 genes x 63 samples

[load] GSE48452
      GPL GPL11532:gene_assignment -> 22148 probes, coverage 66.5%
      symbol strategies: {'gpl': '66.5%'}  -> gpl
      OK: 20

In [26]:
# BACKUP — copies the current pipeline state; overwrites nothing
import shutil, time
BK = PATHS["root"].parent / f"CoDAGAT_backup_{time.strftime('%Y%m%d_%H%M')}"
for name in ["interim", "processed", "config", "reports", "manifests", "handoff"]:
    src = PATHS["root"] / name
    if src.exists():
        shutil.copytree(src, BK / name)
        print("copied", name)
for f in PATHS["root"].glob("*.zip"):
    shutil.copy(f, BK / f.name); print("copied", f.name)
print("BACKUP ->", BK)

copied interim
copied processed
copied config
copied reports
copied manifests
BACKUP -> /content/drive/MyDrive/CoDAGAT_backup_20261008_1042


In [27]:
print(labels_raw.cohort.value_counts().sort_index().to_string())
print("DISCOVERY:", {k: v for k, v in DISCOVERY.get("GSE240729", {"missing": True}).items()
                     if k != "per_sample" and k != "keys"})
print("DOWNLOAD :", DOWNLOAD_LOG.get("GSE240729", "not in log"))

cohort
E-MTAB-6863     69
E-MTAB-9815     58
GSE130970       78
GSE135251      216
GSE162694      143
GSE193066      164
GSE207310       30
GSE213621      368
GSE240729       67
GSE276114      177
GSE31803       131
GSE48452        73
GSE84044       124
GSE89632        63
DISCOVERY: {'n_gsm': 67}
DOWNLOAD : {'gse': 'GSE240729', 'tier': 'T2', 'soft': 'ok', 'supp': [{'file': 'GSE240729_RAW.tar', 'status': 'skipped_RAW'}, {'file': 'GSE240729_merged.counts.67samp.txt.gz', 'status': 'cached', 'gzip_ok': True, 'bytes': 2843775}, {'file': 'filelist.txt', 'status': 'cached', 'bytes': 5319}, {'file': 'https://www.hhs.gov/vulnerability-disclosure-policy/index.html', 'status': 'failed: HTTPError: HTTP Error 404: Not Found'}], 'errors': [], 'n_gsm': 67, 'gpl': ['GPL24676']}


In [28]:
# ============================================================
# CELL 16 — VERIFY POSITIONAL ID ASSIGNMENT
# GSE162694 titles embed the stage (nash1_F0, nash101_F3, ..._N).
# If positional matching is correct, title-stage must agree with
# the parsed metadata stage. Disagreement = the mapping is wrong.
# ============================================================

def verify_positional(gse):
    g = GEOparse.get_GEO(geo=gse, destdir=str(PATHS["raw_geo"]/gse),
                         silent=True, include_data=False)
    rows = []
    for gsm_id, gsm in g.gsms.items():
        title = " ".join(gsm.metadata.get("title", []))
        m = re.search(r"_F([0-4])\b", title) or re.search(r"_(N)\b", title)
        rows.append(dict(sample_id=gsm_id, title=title,
                         title_stage=("healthy" if (m and m.group(1) == "N")
                                      else (f"F{m.group(1)}" if m else None))))
    t = pd.DataFrame(rows)
    lab = labels_raw[labels_raw.cohort == gse][["sample_id","stage","resolution"]]
    j = t.merge(lab, on="sample_id", how="inner")
    j["agree"] = ((j.title_stage == j.stage) |
                  ((j.title_stage == "healthy") & (j.resolution == RES_HEALTHY)))
    checkable = j[j.title_stage.notna()]
    rate = checkable.agree.mean() if len(checkable) else np.nan
    print(f"{gse}: {len(checkable)} checkable, agreement = {rate:.1%}")
    if len(checkable) and rate < 0.99:
        print(j[~j.agree].head(15).to_string(index=False))
    return rate, j

for gse in [g for g, l in LOAD_LOG.items()
            if "positional" in str(l.get("id_method", ""))]:
    rate, detail = verify_positional(gse)
    if not np.isnan(rate) and rate < 0.99:
        print(f"  !! {gse}: POSITIONAL MAPPING IS WRONG. Do not use this cohort "
              f"until the column order is resolved.")
    else:
        print(f"  OK {gse}: title-encoded stages confirm the positional order. "
              f"Record this verification in the paper.")

In [29]:
# ============================================================
# CELL 17 — EXPRESSION LOADER v3
# Fixes:
#  (a) empty/absent GPL table -> IndexError
#  (b) SuperSeries: pick the GPL that matches the expression samples
#  (c) per-GSM supplementary counts files (GSE135251, GSE207310)
#  (d) GCT format (GSE193066)
#  (e) token-based ID reconciliation -> removes positional guessing
# ============================================================
from urllib.parse import urlsplit





# ---------- re-run only what failed ----------
RETRY = [g for g, l in LOAD_LOG.items()
         if l.get("status") != "OK" or "positional" in str(l.get("id_method", ""))
         or "8%" in str(l.get("symbol_method", ""))]
print("RETRY:", RETRY)

for gse in RETRY:
    print(f"\n[reload] {gse}")
    log = {"gse": gse}
    try:
        df, src = load_expression(gse)
        if df is None:
            LOAD_LOG[gse] = {**log, "status": "FAILED", "source": src}
            print(f"      FAILED: {src}"); continue
        log["source"], log["raw_shape"] = src, list(df.shape)

        want = labels_raw.loc[labels_raw.cohort == gse, "sample_id"].tolist()
        cmap, method = reconcile_ids(df.columns, want, gse)
        log["id_method"], log["matched"] = method, len(cmap)
        if not cmap:
            LOAD_LOG[gse] = {**log, "status": "FAILED_ID_MATCH"}
            print(f"      FAILED: {method}"); continue
        df = apply_sample_mapping(df, cmap, gse)

        syms, how = map_ids_to_symbols(df.index, gse)
        log["symbol_method"] = how
        df.index = syms
        df = df[df.index.notna()]
        if df.shape[0] < 3000:
            LOAD_LOG[gse] = {**log, "status": "FAILED_SYMBOL_MAP",
                             "genes": int(df.shape[0])}
            print(f"      FAILED: {df.shape[0]} genes after mapping"); continue
        df = df.groupby(level=0).median()

        pos = df.values[np.isfinite(df.values) & (df.values > 0)]
        med = float(np.median(pos)) if pos.size else 0.0
        log["median_before"] = round(med, 3)
        if med > 50:
            df = np.log2(df.clip(lower=0) + 1); log["log2_applied"] = True
        else:
            log["log2_applied"] = False
        log.update(status="OK", genes=int(df.shape[0]), samples=int(df.shape[1]),
                   median_after=round(float(np.nanmedian(df.values)), 3))
        EXPR[gse] = df
        df.to_parquet(CACHE / f"{gse}.parquet")
        (CACHE / f"{gse}.json").write_text(json.dumps(log, indent=2))
        print(f"      OK: {df.shape[0]} genes x {df.shape[1]} samples, "
              f"median={log['median_after']}, ids={method}")
    except Exception as e:
        import traceback; traceback.print_exc()
        log.update(status=f"EXCEPTION: {type(e).__name__}: {e}")
    LOAD_LOG[gse] = log

print("\n" + "="*104)
print(f"{'cohort':12s} {'status':18s} {'genes':>6s} {'smp':>4s} {'want':>4s} "
      f"{'median':>7s}  {'id_method':24s} symbol_method")
for gse in geo_targets:
    l = LOAD_LOG.get(gse, {})
    print(f"{gse:12s} {str(l.get('status','?'))[:18]:18s} {l.get('genes',0):6d} "
          f"{l.get('samples',0):4d} {(labels_raw.cohort==gse).sum():4d} "
          f"{l.get('median_after',0):7.2f}  {str(l.get('id_method',''))[:24]:24s} "
          f"{l.get('symbol_method','')}")

(PATHS["reports"]/"expression_load_log.json").write_text(json.dumps(LOAD_LOG, indent=2))
ok = [g for g, l in LOAD_LOG.items() if l.get("status") == "OK"]
print(f"\nOK {len(ok)}/{len(geo_targets)}   "
      f"still failing: {[g for g in geo_targets if g not in ok]}")
print(f"still positional: "
      f"{[g for g in ok if 'positional' in str(LOAD_LOG[g].get('id_method',''))]}")

RETRY: ['GSE135251', 'GSE31803', 'GSE240729', 'GSE193066', 'GSE207310']

[reload] GSE135251


  GSE135251 per-GSM:   0%|          | 0/216 [00:00<?, ?it/s]

      symbol strategies: {'already_symbol': '100.0%', 'ensembl': '73.3%'}  -> already_symbol
      OK: 64253 genes x 216 samples, median=0.0, ids=direct_gsm

[reload] GSE31803
      GPL GPL570:Gene Symbol -> 45782 probes, coverage 8.5%
      symbol strategies: {'gpl': '8.5%'}  -> gpl
      OK: 22880 genes x 131 samples, median=2.585, ids=direct_gsm

[reload] GSE240729
      FAILED: FAILED_ID_MATCH: equal counts do not prove sample order

[reload] GSE193066
      FAILED: FAILED(cols=107, want=164)

[reload] GSE207310


  GSE207310 per-GSM:   0%|          | 0/30 [00:00<?, ?it/s]

      symbol strategies: {'already_symbol': '99.8%', 'ensembl': '74.8%'}  -> already_symbol
      OK: 60672 genes x 30 samples, median=0.0, ids=direct_gsm

cohort       status              genes  smp want  median  id_method                symbol_method
GSE135251    OK                  64253  216  216    0.00  direct_gsm               already_symbol(100%)
GSE84044     OK                  22880  124  124    4.55  direct_gsm               gpl(84%)
GSE162694    OK                  31683  143  143    5.81  token_match_VERIFIED     already_symbol(100%)
GSE130970    OK                  19426   78   78    7.79  exact_metadata_string    entrez(99%)
GSE89632     OK                  20819   63   63   10.15  direct_gsm               gpl(100%)
GSE48452     OK                  20578   73   73    5.53  direct_gsm               gpl(67%)
GSE31803     OK                  22880  131  131    2.58  direct_gsm               gpl(8%)
GSE276114    OK                  19983  177  177    7.27  exact_metadata_str

In [30]:
# ============================================================
# CELL 18 — ID VERIFICATION  (v2)
# Cell 16 returned "OK" when there was nothing to check.
# NaN is now UNVERIFIED, not a pass.
# Three independent checks; a cohort must pass at least one.
# ============================================================

def verify_ids(gse):
    checks = {}

    # (1) stage encoded in the sample title
    try:
        g = GEOparse.get_GEO(geo=gse, destdir=str(PATHS["raw_geo"]/gse),
                             silent=True, include_data=False)
        rows = []
        for gsm_id, gsm in g.gsms.items():
            title = " ".join(gsm.metadata.get("title", []) +
                             gsm.metadata.get("description", []))
            m = re.search(r"[_\- ]F([0-4])\b", title)
            n = re.search(r"[_\- ](N|normal)\b", title, re.I)
            rows.append(dict(sample_id=gsm_id,
                             title_stage=f"F{m.group(1)}" if m else
                                         ("healthy" if n else None)))
        t = pd.DataFrame(rows).merge(
            labels_raw[labels_raw.cohort == gse][["sample_id","stage","resolution"]],
            on="sample_id")
        ck = t[t.title_stage.notna()]
        if len(ck) >= 10:
            agree = ((ck.title_stage == ck.stage) |
                     ((ck.title_stage == "healthy") &
                      (ck.resolution == RES_HEALTHY))).mean()
            checks["title_stage"] = (agree, len(ck))
    except Exception as e:
        checks["title_stage_error"] = str(e)

    # (2) biological sanity: COL1A1 must rise with stage
    if gse in EXPR:
        df = EXPR[gse]
        lab = labels_raw[(labels_raw.cohort == gse) & labels_raw.stage.notna()]
        marker = next((m for m in ("COL1A1","COL1A2","COL3A1","TIMP1","ACTA2")
                       if m in df.index), None)
        if marker is not None and len(lab) >= 20:
            s = df.loc[marker].reindex(lab.sample_id.values)
            stg = lab.stage.str[1].astype(float).values
            ok = np.isfinite(s.values) & np.isfinite(stg)
            if ok.sum() >= 20:
                from scipy.stats import spearmanr
                r, p = spearmanr(s.values[ok], stg[ok])
                checks["marker_corr"] = (marker, round(float(r),3), float(p), int(ok.sum()))

    # (3) sex marker vs recorded sex (XIST / RPS4Y1)
    if gse in EXPR:
        df = EXPR[gse]
        d = DISCOVERY.get(gse, {}).get("per_sample", {})
        sexkey = next((k for k in ("sex","gender") if any(k in r for r in d.values())), None)
        gene = next((g for g in ("XIST","RPS4Y1","DDX3Y","KDM5D") if g in df.index), None)
        if sexkey and gene:
            rows = [(s, str(r[sexkey]).lower()[:1]) for s, r in d.items() if sexkey in r]
            sub = pd.DataFrame(rows, columns=["sample_id","sex"])
            sub = sub[sub.sample_id.isin(df.columns)]
            if len(sub) >= 20 and sub.sex.nunique() == 2:
                v = df.loc[gene, sub.sample_id.values].astype(float)
                grp = sub.assign(v=v.values).groupby("sex").v.mean()
                sep = abs(grp.iloc[0] - grp.iloc[1]) / (df.loc[gene].std() + 1e-9)
                checks["sex_marker"] = (gene, round(float(sep),2), dict(grp.round(2)))
    return checks


print(f"{'cohort':12s} {'id_method':24s} verification")
for gse in [g for g, l in LOAD_LOG.items() if l.get("status") == "OK"]:
    method = str(LOAD_LOG[gse].get("id_method",""))
    c = verify_ids(gse)
    bits = []
    verified = method in {
    "direct_gsm",
    "embedded_gsm",
    "exact_metadata_string",
    "token_match_VERIFIED",
    "normalized_match_VERIFIED",
}
    if "title_stage" in c:
        a, n = c["title_stage"]
        bits.append(f"title_stage={a:.0%}(n={n})")

    if "marker_corr" in c:
        m, r, p, n = c["marker_corr"]
        bits.append(f"{m}~stage rho={r} p={p:.1e}")

    if "sex_marker" in c:
        g_, sep, grp = c["sex_marker"]
        bits.append(f"{g_} sex-sep={sep}")

    status = "VERIFIED" if verified else ">>> UNVERIFIED — DO NOT USE IN T1"
    print(f"{gse:12s} {method[:24]:24s} {status}")
    for b in bits:
        print(f"{'':12s} {'':24s}   {b}")

cohort       id_method                verification
GSE135251    direct_gsm               VERIFIED
GSE84044     direct_gsm               VERIFIED
                                        COL1A1~stage rho=0.327 p=2.1e-04
                                        XIST sex-sep=2.06
GSE162694    token_match_VERIFIED     VERIFIED
                                        title_stage=100%(n=143)
GSE130970    exact_metadata_string    VERIFIED
                                        COL1A1~stage rho=0.492 p=4.9e-06
                                        RPS4Y1 sex-sep=1.88
GSE89632     direct_gsm               VERIFIED
                                        COL1A1~stage rho=0.401 p=1.1e-02
                                        RPS4Y1 sex-sep=1.92
GSE48452     direct_gsm               VERIFIED
                                        title_stage=0%(n=18)
GSE31803     direct_gsm               VERIFIED
                                        COL1A1~stage rho=0.486 p=1.5e-05
GSE276114    exact_metada

In [31]:
# CELL 18b — SOURCE PROCESSING REVIEW
# Run this before accepting Cell 19's normalization choices.

review_rows = []

for gse in geo_targets:
    rec = LOAD_LOG.get(gse, {})

    review_rows.append({
        "cohort": gse,
        "status": rec.get("status"),
        "source": rec.get("source"),
        "raw_shape": str(rec.get("raw_shape")),
        "id_method": rec.get("id_method"),
        "median_before": rec.get("median_before"),
        "log2_applied": rec.get("log2_applied"),
        "symbol_method": rec.get("symbol_method"),
    })

print(pd.DataFrame(review_rows).to_string(index=False))

for gse in geo_targets:
    print(f"\n--- {gse}: source processing metadata ---")

    obj = GEOparse.get_GEO(
        geo=gse,
        destdir=str(PATHS["raw_geo"] / gse),
        silent=True,
        include_data=False,
    )

    processing = set()

    for gsm in obj.gsms.values():
        values = gsm.metadata.get("data_processing", [])
        if isinstance(values, str):
            values = [values]

        processing.update(
            str(value).strip()
            for value in values
            if str(value).strip()
        )

    if processing:
        for description in sorted(processing):
            print(description)
    else:
        print("No GSM data_processing metadata found.")

   cohort          status                                                            source     raw_shape                                               id_method  median_before log2_applied        symbol_method
GSE135251              OK                                                per_gsm(216 files)  [64253, 216]                                              direct_gsm         58.000         True already_symbol(100%)
 GSE84044              OK                                                     series_matrix  [54675, 124]                                              direct_gsm          4.551        False             gpl(84%)
GSE162694              OK                                       GSE162694_raw_counts.csv.gz  [31683, 143]                                    token_match_VERIFIED         60.000         True already_symbol(100%)
GSE130970              OK GSE130970_all_sample_salmon_tximport_counts_entrez_gene_ID.csv.gz   [19585, 78]                                   exact_metadata_s

In [32]:
# ============================================================
# CELL 19 — SCALE DETECTION + TARGETED COHORT FIXES
#   (a) raw counts -> CPM -> log2   (GSE135251, GSE207310)
#   (b) GSE31803: drop methylation samples, diagnose index
#   (c) GSE240729: normalized full-string ID match
#   (d) GSE193066: prefer the 164-sample GCT, match baseline only
# ============================================================

def detect_scale(df):
    """Return one of: raw_counts | log_scale | linear_intensity | tpm_fpkm."""
    v = df.values
    fin = v[np.isfinite(v)]
    if fin.size == 0:
        return "empty", {}
    pos = fin[fin > 0]
    stats = dict(
        max=float(np.nanmax(fin)),
        median_pos=float(np.median(pos)) if pos.size else 0.0,
        p99=float(np.percentile(fin, 99)),
        frac_zero=float((fin == 0).mean()),
        frac_integer=float(np.mean(np.isclose(pos[:200000] % 1, 0))) if pos.size else 0.0,
    )
    if stats["frac_integer"] > 0.95 and stats["max"] > 1000:
        return "raw_counts", stats
    if stats["max"] > 100 and stats["frac_integer"] < 0.95:
        return "tpm_fpkm", stats
    if stats["max"] <= 30:
        return "log_scale", stats
    return "linear_intensity", stats


def normalize_scale(df, gse):
    kind, s = detect_scale(df)
    print(f"      scale={kind}  max={s.get('max',0):.1f} "
          f"median_pos={s.get('median_pos',0):.2f} "
          f"zeros={s.get('frac_zero',0):.1%} int={s.get('frac_integer',0):.0%}")
    if kind == "raw_counts":
        lib = df.sum(axis=0).replace(0, np.nan)
        df = np.log2(df.div(lib, axis=1) * 1e6 + 1)          # CPM -> log2
        note = "CPM+log2"
    elif kind == "tpm_fpkm":
        df = np.log2(df.clip(lower=0) + 1)
        note = "log2"
    elif kind == "linear_intensity":
        df = np.log2(df.clip(lower=0) + 1)
        note = "log2"
    else:
        note = "none(already log)"
    # low-expression filter: keep genes detected in >=20% of samples
    keep = (df > 0).mean(axis=1) >= 0.20
    print(f"      {note}; gene filter {int(keep.sum())}/{len(keep)} kept; "
          f"new median={float(np.nanmedian(df[keep].values)):.2f}")
    return df[keep], f"{kind}->{note}"


# ---------- (c) normalized-string ID matching ----------
def _norm_key(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower())

def reconcile_normalized(expr_cols, want_ids, gse):
    idx, exact = build_gsm_token_index(gse)
    norm2gsm = defaultdict(set)
    for raw, gsm in exact.items():
        k = _norm_key(raw)
        if len(k) >= 6:
            norm2gsm[k].add(gsm)
            norm2gsm[re.sub(r"\.(txt|tsv|csv|gz|counts)+$", "", k)].add(gsm)
    want = set(map(str, want_ids))
    out = {}
    for c in expr_cols:
        k = _norm_key(c)
        cands = set()
        if k in norm2gsm:
            cands = set(norm2gsm[k])
        else:                                  # substring containment
            for nk, gs in norm2gsm.items():
                if len(k) >= 8 and (k in nk or nk in k):
                    cands |= gs
        cands &= want
        if len(cands) == 1:
            out[str(c)] = cands.pop()
    return out


# =========================== APPLY ===========================

# ---- (a) rescale everything that needs it ----
print("SCALE NORMALISATION")
for gse in list(EXPR.keys()):
    print(f"  {gse}")
    EXPR[gse], note = normalize_scale(EXPR[gse], gse)
    LOAD_LOG[gse]["scale_fix"] = note
    LOAD_LOG[gse]["genes"] = int(EXPR[gse].shape[0])
    LOAD_LOG[gse]["median_after"] = round(float(np.nanmedian(EXPR[gse].values)), 3)
    EXPR[gse].to_parquet(CACHE / f"{gse}.parquet")

# ---- (b) GSE31803: diagnose index, drop methylation ----
print("\nGSE31803 DIAGNOSIS")
if "GSE31803" in EXPR:
    d = EXPR["GSE31803"]
    print("  index sample:", list(d.index[:8]))
    keep = labels_raw[(labels_raw.cohort == "GSE31803") &
                      (labels_raw.resolution != RES_UNKNOWN)].sample_id.tolist()
    have = [c for c in d.columns if c in keep]
    print(f"  expression samples with usable stage: {len(have)}/{d.shape[1]}")
    if 60 <= len(have) <= 80:
        EXPR["GSE31803"] = d[have]
        LOAD_LOG["GSE31803"]["samples"] = len(have)
        LOAD_LOG["GSE31803"]["note"] = "methylation samples dropped"
        EXPR["GSE31803"].to_parquet(CACHE / "GSE31803.parquet")
        print(f"  -> kept {len(have)} expression samples")
    else:
        print("  !! unexpected count — inspect before proceeding")

# ---- (c) GSE240729 ----
print("\nGSE240729 ID RE-MATCH")
if "GSE240729" in EXPR:
    raw, src = load_expression("GSE240729")
    if raw is not None:
        want = labels_raw.loc[labels_raw.cohort == "GSE240729", "sample_id"].tolist()
        cmap = reconcile_normalized(raw.columns, want, "GSE240729")
        print(f"  normalized match: {len(cmap)}/{len(want)}")
        print(f"  sample cols: {list(raw.columns[:4])}")
        if len(cmap) >= 0.9 * len(want):
            df = apply_sample_mapping(raw, cmap, "GSE240729")
            syms, how = map_ids_to_symbols(df.index, "GSE240729")
            df.index = syms; df = df[df.index.notna()].groupby(level=0).median()
            df, note = normalize_scale(df, "GSE240729")
            EXPR["GSE240729"] = df
            LOAD_LOG["GSE240729"].update(
                id_method="normalized_match_VERIFIED", matched=len(cmap),
                genes=int(df.shape[0]), samples=int(df.shape[1]), scale_fix=note,
                median_after=round(float(np.nanmedian(df.values)), 3))
            df.to_parquet(CACHE / "GSE240729.parquet")
            print("  -> RESOLVED, no longer positional")
        else:
            print("  -> still unmatched; stays positional_UNVERIFIED")

# ---- (d) GSE193066 ----
print("\nGSE193066 RETRY")
gdir = PATHS["raw_geo"] / "GSE193066"
gcts = sorted(gdir.glob("*.gct*"), key=lambda f: -f.stat().st_size)
print("  GCT files:", [f.name for f in gcts])
baseline = labels_raw[(labels_raw.cohort == "GSE193066") &
                      (labels_raw.resolution != "duplicate_biopsy")].sample_id.tolist()
print(f"  baseline samples wanted: {len(baseline)}")

for f in gcts:
    try:
        df = pd.read_csv(f, sep="\t", skiprows=2, low_memory=False)
        name = "Name" if "Name" in df.columns else df.columns[0]
        df = df.set_index(name).drop(columns=["Description"], errors="ignore")
        df = df.apply(pd.to_numeric, errors="coerce").dropna(how="all")
        print(f"  {f.name}: {df.shape}, cols {list(df.columns[:3])}")
        cmap, method = reconcile_ids(df.columns, baseline, "GSE193066")
        if len(cmap) < 0.9 * len(baseline):
            cmap = reconcile_normalized(df.columns, baseline, "GSE193066")
            method = "normalized_match_VERIFIED"
        print(f"    matched {len(cmap)}/{len(baseline)} via {method}")
        if len(cmap) >= 0.9 * len(baseline):
            d = apply_sample_mapping(df, cmap, "GSE193066")
            syms, how = map_ids_to_symbols(d.index, "GSE193066")
            d.index = syms; d = d[d.index.notna()].groupby(level=0).median()
            d, note = normalize_scale(d, "GSE193066")
            EXPR["GSE193066"] = d
            LOAD_LOG["GSE193066"] = dict(
                gse="GSE193066", status="OK", source=f"gct:{f.name}",
                id_method=method, matched=len(cmap), symbol_method=how,
                scale_fix=note, genes=int(d.shape[0]), samples=int(d.shape[1]),
                median_after=round(float(np.nanmedian(d.values)), 3))
            d.to_parquet(CACHE / "GSE193066.parquet")
            print("    -> RESOLVED")
            break
    except Exception as e:
        print(f"  {f.name}: {type(e).__name__}: {e}")

# ---------------- final summary ----------------
print("\n" + "="*112)
print(f"{'cohort':12s} {'status':10s} {'genes':>6s} {'smp':>4s} {'want':>4s} "
      f"{'median':>7s}  {'scale_fix':22s} {'id_method':26s}")
for gse in geo_targets:
    l = LOAD_LOG.get(gse, {})
    want = labels_raw[(labels_raw.cohort == gse) &
                      (~labels_raw.resolution.isin([RES_UNKNOWN, "duplicate_biopsy"]))].shape[0]
    print(f"{gse:12s} {str(l.get('status','?'))[:10]:10s} {l.get('genes',0):6d} "
          f"{l.get('samples',0):4d} {want:4d} {l.get('median_after',0):7.2f}  "
          f"{str(l.get('scale_fix',''))[:22]:22s} {str(l.get('id_method',''))[:26]:26s}")

(PATHS["reports"]/"expression_load_log.json").write_text(json.dumps(LOAD_LOG, indent=2))
ok = [g for g, l in LOAD_LOG.items() if l.get("status") == "OK"]
print(f"\nOK {len(ok)}/{len(geo_targets)}")
print(f"still positional: "
      f"{[g for g in ok if 'positional' in str(LOAD_LOG[g].get('id_method',''))]}")
meds = [l["median_after"] for l in LOAD_LOG.values()
        if l.get("status") == "OK" and l.get("median_after")]
print(f"median range across cohorts: {min(meds):.2f} to {max(meds):.2f}")
print("  ^ this spread is the platform-scale problem. It is EXPECTED and is exactly")
print("    why the GTEx deviation feature must be rank-based, not z-scored.")

SCALE NORMALISATION
  GSE84044
      scale=log_scale  max=13.8 median_pos=4.55 zeros=0.0% int=0%
      none(already log); gene filter 22880/22880 kept; new median=4.55
  GSE162694
      scale=log_scale  max=24.5 median_pos=5.93 zeros=2.2% int=2%
      none(already log); gene filter 31683/31683 kept; new median=5.81
  GSE130970
      scale=log_scale  max=22.4 median_pos=8.18 zeros=10.1% int=0%
      none(already log); gene filter 18061/19426 kept; new median=8.07
  GSE89632
      scale=log_scale  max=15.8 median_pos=10.15 zeros=0.0% int=0%
      none(already log); gene filter 20819/20819 kept; new median=10.15
  GSE48452
      scale=log_scale  max=13.9 median_pos=5.53 zeros=0.0% int=0%
      none(already log); gene filter 20578/20578 kept; new median=5.53
  GSE31803
      scale=log_scale  max=15.4 median_pos=2.58 zeros=0.0% int=0%
      none(already log); gene filter 22880/22880 kept; new median=2.58
  GSE276114
      scale=log_scale  max=22.2 median_pos=8.01 zeros=16.6% int=2%
      no

In [33]:
# ============================================================
# CELL 20 — SAMPLE-LEVEL QC
#  (a) library-size / depth normalisation check per cohort
#  (b) outlier samples by median correlation to cohort centroid
#  (c) marker sanity (COL1A1 / ALB) per cohort
#  Nothing is dropped automatically — this reports, you decide.
# ============================================================
from scipy.stats import spearmanr

def qc_cohort(gse, df):
    v = df.values
    col_med = np.nanmedian(v, axis=0)
    col_sum = np.nansum(v, axis=0)
    q = dict(
        n_genes=int(df.shape[0]), n_samples=int(df.shape[1]),
        median_of_medians=round(float(np.nanmedian(col_med)), 3),
        median_iqr=round(float(np.nanpercentile(col_med, 75) -
                               np.nanpercentile(col_med, 25)), 3),
        median_cv=round(float(np.nanstd(col_med) / (abs(np.nanmean(col_med)) + 1e-9)), 3),
        sum_cv=round(float(np.nanstd(col_sum) / (abs(np.nanmean(col_sum)) + 1e-9)), 3),
        frac_zero=round(float(np.mean(v == 0)), 3),
    )
    # depth-normalised data has near-constant per-sample totals
    q["depth_normalised"] = bool(q["sum_cv"] < 0.10)

    # correlation of each sample to the cohort centroid
    sub = df.sample(min(3000, df.shape[0]), random_state=42)
    c = np.corrcoef(np.nan_to_num(sub.values.T))
    med_corr = np.median(c, axis=1)
    q["min_sample_corr"] = round(float(med_corr.min()), 3)
    q["n_outliers_r<0.7"] = int((med_corr < 0.7).sum())
    q["outlier_ids"] = list(df.columns[med_corr < 0.7][:8])
    for g in ("COL1A1", "ALB"):
        q[f"{g}_present"] = bool(g in df.index)
    return q


QC = {}
print(f"{'cohort':12s} {'genes':>6s} {'smp':>4s} {'medOfMed':>8s} {'medIQR':>7s} "
      f"{'sumCV':>6s} {'depthN':>6s} {'zero%':>6s} {'minR':>6s} {'out':>4s}")
for gse, df in EXPR.items():
    q = qc_cohort(gse, df); QC[gse] = q
    print(f"{gse:12s} {q['n_genes']:6d} {q['n_samples']:4d} "
          f"{q['median_of_medians']:8.2f} {q['median_iqr']:7.2f} "
          f"{q['sum_cv']:6.3f} {str(q['depth_normalised']):>6s} "
          f"{q['frac_zero']*100:5.1f}% {q['min_sample_corr']:6.3f} "
          f"{q['n_outliers_r<0.7']:4d}")

print("\nCOHORTS NEEDING DEPTH NORMALISATION (sum_cv >= 0.10):")
need = [g for g, q in QC.items() if not q["depth_normalised"]]
for g in need:
    print(f"  {g}: sum_cv={QC[g]['sum_cv']:.3f}  median_iqr={QC[g]['median_iqr']:.2f}")
print("  ->", need or "none")

# ---- quantile-normalise within cohort where depth varies ----
# Rank-preserving, within-cohort only, no labels, no cross-cohort info.
def quantile_normalize(df):
    ranks = df.rank(axis=0, method="average")
    ref = np.nanmean(np.sort(df.values, axis=0), axis=1)
    out = np.empty_like(df.values, dtype=float)
    n = df.shape[0]
    for j in range(df.shape[1]):
        idx = np.clip((ranks.values[:, j] - 1).astype(int), 0, n - 1)
        out[:, j] = ref[idx]
    return pd.DataFrame(out, index=df.index, columns=df.columns)

APPLY_QN = True
if APPLY_QN and need:
    for g in need:
        EXPR[g] = quantile_normalize(EXPR[g])
        LOAD_LOG[g]["within_cohort_qn"] = True
        EXPR[g].to_parquet(CACHE / f"{g}.parquet")
        q = qc_cohort(g, EXPR[g])
        print(f"  {g}: sum_cv {QC[g]['sum_cv']:.3f} -> {q['sum_cv']:.3f}")
        QC[g] = q

print("\nOUTLIER SAMPLES (median correlation to centroid < 0.7):")
for g, q in QC.items():
    if q["n_outliers_r<0.7"]:
        print(f"  {g}: {q['n_outliers_r<0.7']} -> {q['outlier_ids']}")
print("  Phase 2 performed no outlier removal. Keeping them is defensible;")
print("  report the count either way. Do NOT drop silently.")

(PATHS["reports"]/"qc_samples.json").write_text(json.dumps(QC, indent=2, default=str))

cohort        genes  smp medOfMed  medIQR  sumCV depthN  zero%   minR  out
GSE84044      22880  124     4.55    0.03  0.004   True   0.0%  0.951    0
GSE162694     31683  143     5.81    0.30  0.038   True   2.2%  0.871    0
GSE130970     18061   78     8.06    0.24  0.026   True   3.6%  0.927    0
GSE89632      20819   63    10.14    0.02  0.001   True   0.0%  0.727    0
GSE48452      20578   73     5.53    0.07  0.004   True   0.0%  0.949    0
GSE31803      22880   72     2.58    0.01  0.006   True   0.0%  0.966    0
GSE276114     17633  177     7.89    0.57  0.072   True   6.1%  0.816    0
GSE213621     19640  368     7.44    0.07  0.014   True   3.6%  0.805    0
GSE135251     25655  216     4.86    0.40  0.043   True  15.8%  0.895    0
GSE207310     32967   30     0.71    0.18  0.043   True  20.7%  0.918    0
GSE193066     16727  106     7.11    0.07  0.018   True   6.7%  0.837    0

COHORTS NEEDING DEPTH NORMALISATION (sum_cv >= 0.10):
  -> none

OUTLIER SAMPLES (median correlatio

3884

In [34]:
# ============================================================
# CELL 22 — IDENTIFIER DIAGNOSIS + REMAP
# BUG: the "already_symbol" regex matched ENSG00000123456
#      (15 chars fits [A-Z][A-Z0-9\-\.@]{1,14}), so Ensembl-indexed
#      cohorts skipped mapping and shared 0 genes with symbol cohorts.
# ============================================================

def classify_index(idx, n=500):
    s = [str(i) for i in list(idx)[:n]]
    f = lambda p: float(np.mean([bool(re.match(p, x)) for x in s]))
    return dict(
        ensembl   = f(r"^ENS[GTP]\d"),
        refseq    = f(r"^(NM_|NR_|XM_|XR_|NP_)\d"),
        numeric   = f(r"^\d+$"),
        symbol    = float(np.mean([_looks_like_symbol(x) for x in s])),
        has_dot   = f(r"^[^.]+\.\d+$"),
    )

def _looks_like_symbol(i):
    i = str(i)
    if re.match(r"^ENS[GTP]\d", i):                 return False   # <-- the fix
    if re.match(r"^(NM_|NR_|XM_|XR_|NP_)\d", i):    return False
    if re.fullmatch(r"\d+(\.\d+)?", i):             return False
    if len(i) > 15:                                 return False
    return bool(re.fullmatch(r"[A-Za-z][A-Za-z0-9\-\.@_]{0,14}", i))


print("INDEX TYPE PER COHORT")
print(f"{'cohort':12s} {'n':>6s} {'ensembl':>8s} {'refseq':>7s} {'numeric':>8s} "
      f"{'symbol':>7s}  first 4 index values")
IDX_TYPE = {}
for gse, df in EXPR.items():
    c = classify_index(df.index)
    kind = max(("ensembl","refseq","numeric","symbol"), key=lambda k: c[k])
    IDX_TYPE[gse] = kind if c[kind] > 0.5 else "mixed"
    print(f"{gse:12s} {df.shape[0]:6d} {c['ensembl']:8.1%} {c['refseq']:7.1%} "
          f"{c['numeric']:8.1%} {c['symbol']:7.1%}  {list(df.index[:4])}")
print("\nclassified:", IDX_TYPE)

# ---------- pairwise overlap BEFORE the fix ----------
def overlap_matrix(expr, title):
    ks = list(expr.keys())
    M = pd.DataFrame(0, index=ks, columns=ks)
    for a in ks:
        for b in ks:
            M.loc[a, b] = len(set(expr[a].index) & set(expr[b].index))
    print(f"\nPAIRWISE GENE OVERLAP — {title}")
    print(M.to_string())
    return M
overlap_matrix(EXPR, "BEFORE remap")

# ---------- remap anything that is not symbols ----------
NEEDS_REMAP = [g for g, k in IDX_TYPE.items() if k != "symbol"]
print(f"\nREMAPPING: {NEEDS_REMAP}")

def force_symbol_map(df, gse, kind):
    ids  = [str(i) for i in df.index]
    base = [i.split(".")[0] for i in ids]
    if kind == "ensembl":
        m = _safe_mg(sorted(set(base)), "ensembl.gene")
        mapped = [m.get(b) for b in base]
    elif kind == "refseq":
        m = _safe_mg(sorted(set(base)), "refseq")
        mapped = [m.get(b) for b in base]
    elif kind == "numeric":
        m = _gpl_map(ids, gse)
        if len(set(ids) & set(m)) / len(ids) < 0.3:
            m = _safe_mg(sorted(set(ids)), "entrezgene")
        mapped = [m.get(i) for i in ids]
    else:                                    # mixed: try each, keep best
        best, cov = None, 0
        for sc, key in (("ensembl.gene", base), ("entrezgene", ids), ("symbol", ids)):
            m = _safe_mg(sorted(set(key)), sc)
            c = sum(1 for k in key if k in m) / len(key)
            if c > cov:
                best, cov = (m, key), c
        mapped = [best[0].get(k) for k in best[1]] if best else [None]*len(ids)
    out = df.copy(); out.index = mapped
    out = out[out.index.notna()]
    hit = out.shape[0] / max(df.shape[0], 1)
    return out.groupby(level=0).median(), hit

for gse in NEEDS_REMAP:
    before = EXPR[gse].shape
    fixed, hit = force_symbol_map(EXPR[gse], gse, IDX_TYPE[gse])
    print(f"  {gse:12s} {IDX_TYPE[gse]:8s} {before[0]:6d} -> {fixed.shape[0]:6d} "
          f"symbols ({hit:.0%} mapped)")
    if fixed.shape[0] >= 8000:
        EXPR[gse] = fixed
        LOAD_LOG[gse]["symbol_method"] = f"remapped_from_{IDX_TYPE[gse]}({hit:.0%})"
        LOAD_LOG[gse]["genes"] = int(fixed.shape[0])
        fixed.to_parquet(CACHE / f"{gse}.parquet")
    else:
        print(f"    !! only {fixed.shape[0]} symbols — NOT applied, investigate")

M = overlap_matrix(EXPR, "AFTER remap")
print("\nsanity: every off-diagonal cell should now be >8000")
low = [(a, b, int(M.loc[a, b])) for a in M.index for b in M.columns
       if a != b and M.loc[a, b] < 8000]
print("LOW-OVERLAP PAIRS:", low or "none")

INDEX TYPE PER COHORT
cohort            n  ensembl  refseq  numeric  symbol  first 4 index values
GSE84044      22880     0.0%    0.0%     0.0%  100.0%  ['A1BG', 'A1BG-AS1', 'A1CF', 'A2M']
GSE162694     31683   100.0%    0.0%     0.0%    0.0%  ['ENSG00000000003', 'ENSG00000000005', 'ENSG00000000419', 'ENSG00000000457']
GSE130970     18061     0.0%    0.0%     0.0%  100.0%  ['A1BG', 'A1CF', 'A2M', 'A2ML1']
GSE89632      20819     0.0%    0.0%     0.0%   99.8%  ['7A5', 'A1BG', 'A1CF', 'A26C3']
GSE48452      20578     3.0%    0.0%     0.0%   25.2%  ['13q12.3', '4q21.23', 'AAK1', 'AB001736 // IGLJ3 // immunoglobulin lambda joining 3 // 22q11.2 // 28831']
GSE31803      22880     0.0%    0.0%     0.0%  100.0%  ['A1BG', 'A1BG-AS1', 'A1CF', 'A2M']
GSE276114     17633     0.0%    0.0%     0.0%  100.0%  ['A1BG', 'A1CF', 'A2M', 'A2ML1']
GSE213621     19640     0.0%    0.0%     0.0%  100.0%  ['A1BG', 'A1BG-AS1', 'A1CF', 'A2M']
GSE135251     25655   100.0%    0.0%     0.0%    0.0%  ['ENSG0000000000

In [35]:
# ============================================================
# CELL 24 — GSE48452 SYMBOL REPAIR
# GPL11532 gene_assignment format:
#   "acc // SYMBOL // description // cytoband // entrez /// acc // SYMBOL2 // ..."
# Two-level split: '///' between assignments, '//' between fields.
# The symbol is field index 1.
# ============================================================

def parse_gene_assignment(v):
    """Return the first valid HGNC-looking symbol, or None."""
    if pd.isna(v):
        return None
    s = str(v).strip()
    if s in ("---", "", "NA", "nan"):
        return None
    for block in s.split("///"):                     # each assignment
        fields = [f.strip() for f in block.split("//")]
        if len(fields) >= 2:
            sym = fields[1]
            if re.fullmatch(r"[A-Za-z][A-Za-z0-9\-\.@_]{0,14}", sym) \
               and not re.fullmatch(r"\d+[pq][\d\.]+", sym):   # reject cytobands
                return sym
        # single-token fallback
        if len(fields) == 1 and re.fullmatch(r"[A-Z][A-Z0-9\-]{1,14}", fields[0]):
            return fields[0]
    return None


print("GSE48452 REPAIR")
tables = _gpl_tables("GSE48452")
print("  GPL tables:", [(g, t.shape, list(t.columns)[:6]) for g, t in tables])

repaired = None
for gpl_id, tbl in tables:
    if "gene_assignment" not in [c.lower() for c in tbl.columns]:
        continue
    col = next(c for c in tbl.columns if c.lower() == "gene_assignment")
    id_col = "ID" if "ID" in tbl.columns else tbl.columns[0]
    print(f"  using {gpl_id}:{col}, {len(tbl)} probes")
    for ex in tbl[col].dropna().head(3):
        print(f"    raw: {str(ex)[:110]}")
        print(f"    ->   {parse_gene_assignment(ex)}")

    probe2sym = {str(k): parse_gene_assignment(v)
                 for k, v in zip(tbl[id_col], tbl[col])}
    probe2sym = {k: v for k, v in probe2sym.items() if v}
    print(f"  {len(probe2sym)} probes mapped to symbols "
          f"({len(set(probe2sym.values()))} unique)")

    # reload raw so we start from probe IDs, not the corrupted index
    raw, src = load_expression("GSE48452")
    want = labels_raw.loc[labels_raw.cohort == "GSE48452", "sample_id"].tolist()
    cmap, method = reconcile_ids(raw.columns, want, "GSE48452")
    d = apply_sample_mapping(raw, cmap, "GSE48452")
    print(f"  raw index sample: {list(d.index[:4])}")
    cov = len(set(map(str, d.index)) & set(probe2sym)) / len(d.index)
    print(f"  probe coverage: {cov:.1%}")
    if cov < 0.3:
        print("  !! probe IDs do not match the GPL table — inspect above")
        break

    d.index = [probe2sym.get(str(i)) for i in d.index]
    d = d[d.index.notna()].groupby(level=0).median()
    d, note = normalize_scale(d, "GSE48452")
    repaired = d
    print(f"  -> {d.shape[0]} symbols x {d.shape[1]} samples")
    break

if repaired is not None and repaired.shape[0] >= 8000:
    EXPR["GSE48452"] = repaired
    LOAD_LOG["GSE48452"].update(symbol_method="gene_assignment_parsed",
                                genes=int(repaired.shape[0]),
                                samples=int(repaired.shape[1]))
    repaired.to_parquet(CACHE / "GSE48452.parquet")
    others = [g for g in EXPR if g != "GSE48452"]
    ov = {g: len(set(repaired.index) & set(EXPR[g].index)) for g in others}
    print(f"\n  overlap with other cohorts: min={min(ov.values())}, "
          f"median={int(np.median(list(ov.values())))}")
    print("  ", {k: v for k, v in sorted(ov.items(), key=lambda x: x[1])[:4]})
else:
    print("\n  REPAIR FAILED — GSE48452 is T2 (training-only) so the pipeline can")
    print("  proceed without it. Set DROP_GSE48452=True in the next cell if so.")

GSE48452 REPAIR
  GPL tables: [('GPL11532', (33297, 12), ['ID', 'GB_LIST', 'SPOT_ID', 'seqname', 'RANGE_GB', 'RANGE_STRAND'])]
  using GPL11532:gene_assignment, 33297 probes
    raw: ---
    ->   None
    raw: ---
    ->   None
    raw: NM_001005240 // OR4F17 // olfactory receptor, family 4, subfamily F, member 17 // 19p13.3 // 81099 /// NM_0010
    ->   OR4F17
  22146 probes mapped to symbols (20252 unique)
  raw index sample: [7892501, 7892502, 7892503, 7892504]
  probe coverage: 66.5%
      scale=log_scale  max=13.7 median_pos=5.54 zeros=0.0% int=0%
      none(already log); gene filter 20252/20252 kept; new median=5.54
  -> 20252 symbols x 73 samples

  overlap with other cohorts: min=13193, median=15054
   {'GSE213621': 13193, 'GSE135251': 14170, 'GSE162694': 14275, 'GSE207310': 14766}


In [50]:
# ============================================================
# CELL 24b — RECOVER GSE240729
# Run after Cell 24 and before Cell 25.
# ============================================================

gse = "GSE240729"
raw, source = load_expression(gse)

if raw is None:
    raise RuntimeError(f"{gse}: expression matrix could not be loaded")

want = labels_raw.loc[
    labels_raw.cohort.eq(gse), "sample_id"
].astype(str).tolist()

cmap = reconcile_normalized(raw.columns, want, gse)

if (
    len(want) != 67
    or len(cmap) != len(want)
    or set(cmap.values()) != set(want)
):
    raise ValueError(
        f"{gse}: expected a complete, unique mapping of 67 samples; "
        f"got {len(cmap)} matches"
    )

counts = apply_sample_mapping(raw, cmap, gse)
values = counts.to_numpy(dtype=float)

if not np.isfinite(values).all():
    raise ValueError(f"{gse}: raw counts contain NaN or infinity")

if (values < 0).any():
    raise ValueError(f"{gse}: raw counts contain negative values")

integer_fraction = float(
    np.isclose(values, np.rint(values), atol=1e-8).mean()
)

if integer_fraction < 0.99:
    raise ValueError(
        f"{gse}: source does not look like raw integer counts "
        f"({integer_fraction:.2%} integer-like); inspect its units"
    )

library_size = counts.sum(axis=0)

if (library_size <= 0).any():
    raise ValueError(f"{gse}: a sample has zero total counts")

# Normalize raw counts before aggregating gene identifiers.
log_cpm = np.log2(
    counts.div(library_size, axis=1) * 1_000_000 + 1
)

index_stats = classify_index(log_cpm.index)
kind = max(
    ("ensembl", "refseq", "numeric", "symbol"),
    key=lambda name: index_stats[name],
)

if index_stats[kind] <= 0.5:
    kind = "mixed"

if kind == "symbol":
    mapped = log_cpm.copy()
    mapped.index = mapped.index.astype(str).str.strip()
    mapped = mapped.groupby(level=0).median()
    mapping_method = "existing_gene_symbols"
else:
    mapped, hit = force_symbol_map(log_cpm, gse, kind)
    mapping_method = f"remapped_from_{kind}({hit:.1%})"

mapped = mapped.loc[
    (mapped > 0).mean(axis=1) >= 0.20
].copy()

check_expression(mapped, f"{gse}: normalized expression")

if mapped.shape[0] < 8000:
    raise ValueError(
        f"{gse}: only {mapped.shape[0]} mapped genes; inspect mapping"
    )

if set(mapped.columns) != set(want):
    raise ValueError(f"{gse}: normalized sample IDs do not match labels")

EXPR[gse] = mapped
IDX_TYPE[gse] = "symbol"

LOAD_LOG[gse] = {
    "gse": gse,
    "status": "OK",
    "source": source,
    "raw_shape": list(raw.shape),
    "id_method": "normalized_match_VERIFIED",
    "matched": len(cmap),
    "symbol_method": mapping_method,
    "scale_fix": "raw_counts->CPM+log2",
    "genes": int(mapped.shape[0]),
    "samples": int(mapped.shape[1]),
}

mapped.to_parquet(CACHE / f"{gse}.parquet")
(CACHE / f"{gse}.json").write_text(
    json.dumps(LOAD_LOG[gse], indent=2)
)

print(
    f"{gse} recovered: {mapped.shape[0]} genes x "
    f"{mapped.shape[1]} samples; {mapping_method}"
)

GSE240729 recovered: 25372 genes x 67 samples; remapped_from_ensembl(78.2%)


In [51]:
# ============================================================
# CELL 25 — INTEGRATION v2.1
# Change: a cohort that costs the intersection more than
# MAX_COHORT_COST genes is a DATA BUG, not a modelling choice.
# The pipeline halts and names it rather than silently degrading.
# ============================================================
from inmoose.pycombat import pycombat_norm

MAX_COHORT_COST = 3000
DROP_COHORTS    = []          # add a GSE only after a repair attempt fails

def intersection_cost(cohorts):
    cnt = Counter()
    for g in cohorts:
        cnt.update(set(EXPR[g].index))
    base = sum(1 for v in cnt.values() if v == len(cohorts))
    costs = {}
    for g in cohorts:
        others = [x for x in cohorts if x != g]
        c2 = Counter()
        for x in others:
            c2.update(set(EXPR[x].index))
        costs[g] = sum(1 for v in c2.values() if v == len(others)) - base
    return base, costs

def integrate(name, cohorts, run_combat=True):
    cohorts = [g for g in cohorts if g not in DROP_COHORTS]

    if not cohorts:
        raise ValueError(f"{name}: no cohorts available")

    if len(cohorts) != len(set(cohorts)):
        raise ValueError(f"{name}: repeated cohort entries")

    for g in cohorts:
        if g not in EXPR:
            raise KeyError(f"{name}: expression missing for {g}")
        check_expression(EXPR[g], g)

    print(
        f"\n[{name}] {len(cohorts)} cohorts, "
        f"{sum(EXPR[g].shape[1] for g in cohorts)} samples"
    )

    base, costs = intersection_cost(cohorts)
    print(f"  intersection: {base} genes")

    for g, cost in sorted(costs.items(), key=lambda item: -item[1]):
        flag = "  <<< BLOCKING" if cost > MAX_COHORT_COST else ""
        print(f"    {g:12s} costs {cost:6d} genes{flag}")

    blockers = [
        g for g, cost in costs.items()
        if cost > MAX_COHORT_COST
    ]
    if blockers:
        raise RuntimeError(
            f"{name}: {blockers} each cost >{MAX_COHORT_COST} genes. "
            "Inspect identifier mapping before proceeding."
        )

    genes = sorted(
        set.intersection(*(set(EXPR[g].index) for g in cohorts))
    )
    if not genes:
        raise ValueError(f"{name}: no shared genes")

    mats = []
    batch = []

    for g in cohorts:
        d = EXPR[g].loc[genes]
        mats.append(d)
        batch.extend([g] * d.shape[1])

    X = pd.concat(mats, axis=1)
    batch = np.asarray(batch)

    check_expression(X, f"{name}: concatenated expression")

    var_ok = np.ones(X.shape[0], dtype=bool)

    for b in np.unique(batch):
        block = X.loc[:, batch == b]

        if block.shape[1] < 2:
            raise ValueError(
                f"{name}: batch {b} has fewer than two samples"
            )

        var_ok &= block.var(axis=1).to_numpy() > 1e-8

    print(
        f"  variance filter: {int(var_ok.sum())}/{len(var_ok)} kept"
    )

    X = X.loc[var_ok].copy()
    check_expression(X, f"{name}: variance-filtered expression")

    if run_combat and len(np.unique(batch)) > 1:
        try:
            Xc = pycombat_norm(X, batch)
        except Exception as exc:
            raise RuntimeError(
                f"{name}: ComBat failed. "
                "Corrected output was not produced."
            ) from exc

        if isinstance(Xc, pd.DataFrame):
            if (
                not Xc.index.equals(X.index)
                or not Xc.columns.equals(X.columns)
            ):
                raise ValueError(
                    f"{name}: ComBat changed gene/sample labels or order"
                )
            corrected = Xc.copy()
        else:
            values = np.asarray(Xc)

            if values.shape != X.shape:
                raise ValueError(
                    f"{name}: ComBat returned {values.shape}; "
                    f"expected {X.shape}"
                )

            corrected = pd.DataFrame(
                values,
                index=X.index,
                columns=X.columns,
            )

        check_expression(corrected, f"{name}: ComBat output")
        X = corrected

    return X, batch

PLATFORM_GROUP = {}
for gse in EXPR:
    key = "microarray" if COHORTS[gse].get("platform") == "microarray" else "rnaseq"
    PLATFORM_GROUP.setdefault(key, []).append(gse)

RNASEQ_X, _ = integrate("rnaseq",     PLATFORM_GROUP["rnaseq"])
MICRO_X,  _ = integrate("microarray", PLATFORM_GROUP["microarray"])

COMMON = sorted(set(RNASEQ_X.index) & set(MICRO_X.index))
print(f"\nCOMMON GENE SPACE: {len(COMMON)}   "
      f"(rnaseq {RNASEQ_X.shape[0]}, microarray {MICRO_X.shape[0]})")
print(f"  Phase 2 reported 11,148 — a similar magnitude is expected.")
assert len(COMMON) > 5000, f"only {len(COMMON)} common genes — inspect"

BULK = pd.concat([RNASEQ_X.loc[COMMON], MICRO_X.loc[COMMON]], axis=1)
check_expression(BULK, "BULK")

lab = labels_raw.set_index("sample_id")
usable = set(lab.index[~lab.resolution.isin([RES_UNKNOWN, "duplicate_biopsy"])])
BULK = BULK[[c for c in BULK.columns if c in usable]]
meta = lab.loc[BULK.columns, ["cohort","stage","resolution","tier","etiology",
                              "platform","preservation","y_T1_significant",
                              "y_T2_ordinal","y_T3_early_adv"]].copy()

print(f"\nFINAL BULK: {BULK.shape[0]} genes x {BULK.shape[1]} samples")
print(meta.groupby(["tier","cohort"]).size().to_string())
print("\nper-cohort median after integration "
      "(differences are expected; per-cohort z-scoring comes next):")
for g in sorted(meta.cohort.unique()):
    cols = meta.index[meta.cohort == g]
    print(f"  {g:12s} {float(np.nanmedian(BULK[cols].values)):7.2f}  n={len(cols)}")

print("\nfibrosis marker presence:")
for m in ("COL1A1","COL1A2","COL3A1","TGFB1","ACTA2","TIMP1","SERPINE1",
          "TIMP2","MMP2","PDGFRB","LUM","THBS2","ALB"):
    print(f"  {m:10s} {'present' if m in BULK.index else '<<< MISSING'}")

check_expression(BULK, "BULK before saving")

if not meta.index.equals(BULK.columns):
    raise ValueError("BULK columns and metadata rows are misaligned")
BULK.to_parquet(PATHS["interim"]/"bulk_expression_log2.parquet")
meta.to_parquet(PATHS["interim"]/"bulk_meta.parquet")

t1 = meta[meta.tier == "T1"]
print(f"\nT1: {len(t1)} samples / {t1.cohort.nunique()} folds")
print(f"T2: {int((meta.tier=='T2').sum())}    N_real: {BULK.shape[1]}")
print("\nF2+ balance per T1 fold:")
for g in sorted(t1.cohort.unique()):
    s = t1[t1.cohort == g].y_T1_significant.dropna()
    print(f"  {g:12s} n={len(s):4d}  pos={int(s.sum()):4d}  "
          f"neg={int((s==0).sum()):4d}  pos%={100*s.mean():5.1f}")

write_manifest("p05_integrate",
               inputs={"expr_cohorts": sorted(EXPR.keys()), "index_types": IDX_TYPE},
               params={"combat": True, "platform_split": True, "z_scored": False,
                       "common_genes": len(COMMON), "dropped": DROP_COHORTS},
               outputs=[PATHS["interim"]/"bulk_expression_log2.parquet",
                        PATHS["interim"]/"bulk_meta.parquet"])


[rnaseq] 8 cohorts, 1185 samples
  intersection: 12122 genes
    GSE213621    costs   1148 genes
    GSE193066    costs    851 genes
    GSE240729    costs    460 genes
    GSE162694    costs    224 genes
    GSE135251    costs     91 genes
    GSE130970    costs     40 genes
    GSE276114    costs     22 genes
    GSE207310    costs     12 genes
  variance filter: 12122/12122 kept

[microarray] 4 cohorts, 332 samples
  intersection: 15757 genes
    GSE89632     costs    978 genes
    GSE48452     costs    256 genes
    GSE31803     costs      0 genes
    GSE84044     costs      0 genes
  variance filter: 15213/15757 kept

COMMON GENE SPACE: 10732   (rnaseq 12122, microarray 15213)
  Phase 2 reported 11,148 — a similar magnitude is expected.

FINAL BULK: 10732 genes x 1514 samples
tier  cohort   
T1    GSE130970     78
      GSE135251    216
      GSE162694    143
      GSE193066    106
      GSE213621    368
      GSE240729     67
      GSE31803      72
      GSE84044     124
T2    G

{'stage': 'p05_integrate',
 'timestamp': '2026-10-08T11:13:24',
 'inputs': {'expr_cohorts': ['GSE130970',
   'GSE135251',
   'GSE162694',
   'GSE193066',
   'GSE207310',
   'GSE213621',
   'GSE240729',
   'GSE276114',
   'GSE31803',
   'GSE48452',
   'GSE84044',
   'GSE89632'],
  'index_types': {'GSE84044': 'symbol',
   'GSE162694': 'ensembl',
   'GSE130970': 'symbol',
   'GSE89632': 'symbol',
   'GSE48452': 'mixed',
   'GSE31803': 'symbol',
   'GSE276114': 'symbol',
   'GSE213621': 'symbol',
   'GSE135251': 'ensembl',
   'GSE207310': 'ensembl',
   'GSE193066': 'symbol',
   'GSE240729': 'symbol'}},
 'params': {'combat': True,
  'platform_split': True,
  'z_scored': False,
  'common_genes': 10732,
  'dropped': []},
 'outputs': {'/content/drive/MyDrive/CoDAGAT_preprocessing_fix_v1/interim/bulk_expression_log2.parquet': '27a5c689115ff4ae4602b528cb83a13db3749d0deba03709df7e26e3bce2426e',
  '/content/drive/MyDrive/CoDAGAT_preprocessing_fix_v1/interim/bulk_meta.parquet': 'b70f9914d009349580f

In [52]:
# ============================================================
# CELL 26 — POST-INTEGRATION DIAGNOSTICS + DUAL SAVE   [v2]
# Fixes vs v1:
#   - no-ComBat arm now genuinely bypasses ComBat (v1 saved a copy
#     of the corrected matrix, so the comparison was invalid)
#   - GSE213621 abundance-filter diagnosis added
#   - dropped the prevalence/ComBat hypothesis (rho=0.15, p=0.63)
# ============================================================

# ---------- (a) canary genes ----------
CANARY = ["ALB","APOA1","APOB","SERPINA1","HP","TF","CYP2E1","CPS1","FGA","FGB",
          "COL1A1","COL1A2","COL3A1","TGFB1","ACTA2","TIMP1","SERPINE1"]
missing_by = Counter()
print("CANARY GENE PRESENCE PER COHORT")
print(f"{'gene':10s} " + " ".join(f"{g[3:9]:>7s}" for g in sorted(EXPR)))
for m in CANARY:
    row = []
    for g in sorted(EXPR):
        here = m in EXPR[g].index
        row.append("      Y" if here else "      .")
        if not here:
            missing_by[g] += 1
    print(f"{m:10s} " + " ".join(row))
print("\ncohorts by canary genes missing:", missing_by.most_common())

# ---------- (a2) why is a cohort missing abundant genes? ----------
for gse, nmiss in missing_by.items():
    if nmiss < 3:
        continue
    print(f"\nABUNDANCE DIAGNOSIS — {gse} (missing {nmiss} canary genes)")
    d = EXPR[gse]
    print(f"  matrix: {d.shape[0]} genes x {d.shape[1]} samples")
    means = d.mean(axis=1).sort_values(ascending=False)
    print(f"  top 12 by mean expression: {list(means.index[:12])}")
    print(f"  expression range: {means.min():.2f} to {means.max():.2f}")
    ref = next(g for g in EXPR if g != gse and "ALB" in EXPR[g].index)
    ref_means = EXPR[ref].mean(axis=1)
    shared = sorted(set(d.index) & set(ref_means.index))
    top500 = set(ref_means.sort_values(ascending=False).index[:500])
    kept = len(top500 & set(d.index))
    print(f"  of the 500 highest-expressed genes in {ref}, {gse} retains {kept}")
    if kept < 350:
        print(f"  >>> {gse} appears ABUNDANCE-FILTERED. Highly expressed genes")
        print(f"      have been removed. Flag in the paper; consider whether it")
        print(f"      belongs in a whole-transcriptome intersection.")
    else:
        print(f"  annotation gap rather than systematic abundance filtering")

print("\nin final BULK:", {m: (m in BULK.index) for m in CANARY})

# ---------- (b) supervised vs healthy ----------
meta["is_healthy"]    = meta.resolution == RES_HEALTHY
meta["is_supervised"] = meta[["y_T1_significant","y_T3_early_adv"]].notna().any(axis=1)
sup = meta[meta.is_supervised]

print("\nSAMPLE ACCOUNTING")
print(f"  BULK columns          : {BULK.shape[1]}")
print(f"  healthy references    : {int(meta.is_healthy.sum())}")
print(f"  supervised (any task) : {int(meta.is_supervised.sum())}")
print("\n  healthy per cohort:")
print(meta[meta.is_healthy].cohort.value_counts().to_string())

print("\nNUMBERS FOR THE PAPER")
print(f"  N_real (supervised)        : {len(sup)}")
print(f"  T1 F2+-evaluable           : "
      f"{int(sup[sup.tier=='T1'].y_T1_significant.notna().sum())}")
print(f"  T2 training-only           : {int((sup.tier=='T2').sum())}")
print(f"  healthy refs (not counted) : {int(meta.is_healthy.sum())}")
print(f"  T1 folds                   : {sup[sup.tier=='T1'].cohort.nunique()}")
print(f"  genes                      : {BULK.shape[0]}")

BULK_SUP = BULK[sup.index]
BULK_SUP.to_parquet(PATHS["interim"]/"bulk_supervised_combat.parquet")
sup.to_parquet(PATHS["interim"]/"bulk_meta_supervised.parquet")
BULK[meta.index[meta.is_healthy]].to_parquet(
    PATHS["interim"]/"bulk_healthy_reference.parquet")
print(f"\n  saved supervised matrix: {BULK_SUP.shape}")

# ---------- (c) TRUE no-ComBat arm ----------
print("\nBUILDING NO-COMBAT VERSION (ablation arm)")

def build_uncorrected(cohorts):
    """Concatenate on the same gene space with NO batch correction."""
    cnt = Counter()
    for g in cohorts:
        cnt.update(set(EXPR[g].index))
    genes = sorted([g for g, v in cnt.items() if v == len(cohorts)])
    mats, cols = [], []
    for g in cohorts:
        d = EXPR[g].loc[genes]
        d = d.loc[:, ~d.columns.duplicated()]
        mats.append(d); cols += list(d.columns)
    X = pd.concat(mats, axis=1); X.columns = cols
    return X.loc[:, ~X.columns.duplicated()]

# Use the exact corrected matrix axes, without another intersection
# that could silently discard genes or samples.
raw_parts = []

for g in EXPR:
    if g in DROP_COHORTS:
        continue

    check_expression(EXPR[g], f"{g}: no-ComBat input")
    raw_parts.append(EXPR[g])

RAW_ALL = pd.concat(raw_parts, axis=1)

if not RAW_ALL.columns.is_unique:
    raise ValueError("Duplicate sample IDs across no-ComBat cohorts")

missing_genes = BULK_SUP.index.difference(RAW_ALL.index)
missing_samples = BULK_SUP.columns.difference(RAW_ALL.columns)

if len(missing_genes) or len(missing_samples):
    raise ValueError(
        f"No-ComBat input is missing {len(missing_genes)} genes "
        f"and {len(missing_samples)} samples"
    )

BULK_RAW = RAW_ALL.loc[
    BULK_SUP.index, BULK_SUP.columns
].copy()

check_expression(BULK_RAW, "BULK_RAW")

assert BULK_RAW.index.equals(BULK_SUP.index)
assert BULK_RAW.columns.equals(BULK_SUP.columns)

delta = BULK_SUP - BULK_RAW
overall = float(np.abs(delta.to_numpy()).mean())

if overall <= 1e-6:
    raise ValueError("ComBat and no-ComBat matrices are effectively identical")

BULK_RAW.to_parquet(
    PATHS["interim"] / "bulk_supervised_nocombat.parquet"
)

# Used by the diagnostics below.
sg = BULK_SUP.index.tolist()
ss = BULK_SUP.columns.tolist()

print(f"mean |ComBat - no-ComBat| = {overall:.4f}")

print("\nPER-COHORT MEDIAN: raw vs ComBat")
print(f"{'cohort':12s} {'n':>4s} {'raw_med':>8s} {'combat_med':>11s} "
      f"{'mean|shift|':>11s} {'pos%':>6s}")
rows = []
for g in sorted(sup.cohort.unique()):
    cols = [c for c in ss if sup.loc[c, "cohort"] == g]
    if not cols:
        continue
    s = sup.loc[cols, "y_T1_significant"].dropna()
    r = dict(cohort=g, n=len(cols),
             raw_med=float(np.nanmedian(BULK_RAW.loc[sg, cols].values)),
             combat_med=float(np.nanmedian(BULK_SUP.loc[sg, cols].values)),
             shift=float(np.nanmean(np.abs(delta[cols].values))),
             pos_pct=float(100*s.mean()) if len(s) else np.nan)
    rows.append(r)
    print(f"{g:12s} {r['n']:4d} {r['raw_med']:8.2f} {r['combat_med']:11.2f} "
          f"{r['shift']:11.3f} {r['pos_pct']:6.1f}")
eff = pd.DataFrame(rows)

# ---------- what actually drives the correction ----------
from scipy.stats import spearmanr
eff = eff.rename(columns={"shift": "abs_shift"})     # 'shift' collides with DataFrame.shift
grand_med = float(np.nanmedian(BULK_RAW.loc[sg, ss].values))
grand_pos = float(sup.y_T1_significant.mean() * 100)

r1, p1 = spearmanr((eff["raw_med"] - grand_med).abs(), eff["abs_shift"])
ok = eff.dropna(subset=["pos_pct"])
r2, p2 = spearmanr((ok["pos_pct"] - grand_pos).abs(), ok["abs_shift"])

print(f"\n  grand median = {grand_med:.2f}   grand prevalence = {grand_pos:.1f}%")
print(f"  |median - grand median| vs shift : rho={r1:.3f} p={p1:.3f}")
print(f"  |prevalence - grand|     vs shift: rho={r2:.3f} p={p2:.3f}")
print("  Expect the first strong, the second weak: ComBat corrects platform")
print("  scale, which is what it is for. Keep the no-ComBat ablation, but do")
print("  not claim ComBat removes label-correlated signal on this evidence.")
eff.to_csv(PATHS["reports"]/"combat_effect_per_cohort.csv", index=False)

write_manifest("p05b_diagnostics",
               inputs={"bulk": "bulk_expression_log2.parquet"},
               params={"n_supervised": int(len(sup)),
                       "n_healthy": int(meta.is_healthy.sum()),
                       "combat_mean_abs_delta": round(overall, 5),
                       "canary_missing": dict(missing_by)},
               outputs=[PATHS["interim"]/"bulk_supervised_combat.parquet",
                        PATHS["interim"]/"bulk_supervised_nocombat.parquet",
                        PATHS["interim"]/"bulk_healthy_reference.parquet",
                        PATHS["interim"]/"bulk_meta_supervised.parquet",
                        PATHS["reports"]/"combat_effect_per_cohort.csv"])

CANARY GENE PRESENCE PER COHORT
gene        130970  135251  162694  193066  207310  213621  240729  276114   31803   48452   84044   89632
ALB              Y       Y       Y       Y       Y       .       Y       Y       Y       Y       Y       Y
APOA1            Y       Y       Y       Y       Y       .       Y       Y       Y       Y       Y       Y
APOB             Y       Y       Y       Y       Y       .       Y       Y       Y       Y       Y       Y
SERPINA1         Y       Y       Y       Y       Y       Y       Y       Y       Y       Y       Y       Y
HP               Y       Y       Y       Y       Y       .       Y       Y       Y       Y       Y       Y
TF               Y       Y       Y       Y       Y       Y       Y       Y       Y       Y       Y       Y
CYP2E1           Y       Y       Y       Y       Y       Y       Y       Y       Y       Y       Y       Y
CPS1             Y       Y       Y       Y       Y       Y       Y       Y       Y       Y       Y       Y
FGA  

{'stage': 'p05b_diagnostics',
 'timestamp': '2026-10-08T11:13:37',
 'inputs': {'bulk': 'bulk_expression_log2.parquet'},
 'params': {'n_supervised': 1366,
  'n_healthy': 148,
  'combat_mean_abs_delta': 1.19223,
  'canary_missing': {'GSE213621': 6}},
 'outputs': {'/content/drive/MyDrive/CoDAGAT_preprocessing_fix_v1/interim/bulk_supervised_combat.parquet': 'ed6d07f63e68baea60bb32a05947cf525388c47bf95e8a5e82d49cac5840d21f',
  '/content/drive/MyDrive/CoDAGAT_preprocessing_fix_v1/interim/bulk_supervised_nocombat.parquet': '15ba6764114002745693898b426b8b91af119fec062606c14290c9efb4f54b64',
  '/content/drive/MyDrive/CoDAGAT_preprocessing_fix_v1/interim/bulk_healthy_reference.parquet': '9b9c2350ae4d0dbf87bc1cb8b8654a0b745dbfce53115a2a2e191084d8dc4c12',
  '/content/drive/MyDrive/CoDAGAT_preprocessing_fix_v1/interim/bulk_meta_supervised.parquet': '03f57d1bd1bce280f2d88e95f3d779c0758960c707037ead82e9bc198abad537',
  '/content/drive/MyDrive/CoDAGAT_preprocessing_fix_v1/reports/combat_effect_per_coh

In [53]:
# ============================================================
# CELL 27 — GTEx v8 LIVER REFERENCE   [v2]
# Fix: GTEx moved gtex_analysis_v8 -> adult-gtex. Candidate URLs
# are tried in order and the working one is recorded.
#
# Two modes:
#   full   : individual liver samples (~226) -> per-gene mean AND sd
#   median : median-TPM-by-tissue only (~10 MB) -> per-gene median only
# Rank-based deviation needs only the median, so 'median' is a valid
# fallback. It costs node feature 15 (gtex_expression_std), and the
# flag below tells the tensor cell to drop it.
# ============================================================

ATT_URLS = [
  "https://storage.googleapis.com/adult-gtex/annotations/v8/metadata-files/"
  "GTEx_Analysis_v8_Annotations_SampleAttributesDS.txt",
  "https://storage.googleapis.com/gtex_analysis_v8/annotations/"
  "GTEx_Analysis_v8_Annotations_SampleAttributesDS.txt",
]
TPM_URLS = [
  "https://storage.googleapis.com/adult-gtex/bulk-gex/v8/rna-seq/"
  "GTEx_Analysis_2017-06-05_v8_RNASeQCv1.1.9_gene_tpm.gct.gz",
  "https://storage.googleapis.com/gtex_analysis_v8/rna_seq_data/"
  "GTEx_Analysis_2017-06-05_v8_RNASeQCv1.1.9_gene_tpm.gct.gz",
]
MEDIAN_URLS = [
  "https://storage.googleapis.com/adult-gtex/bulk-gex/v8/rna-seq/"
  "GTEx_Analysis_2017-06-05_v8_RNASeQCv1.1.9_gene_median_tpm.gct.gz",
  "https://storage.googleapis.com/gtex_analysis_v8/rna_seq_data/"
  "GTEx_Analysis_2017-06-05_v8_RNASeQCv1.1.9_gene_median_tpm.gct.gz",
]

def try_urls(urls, dest, label, timeout=2400):
    for u in urls:
        p, st = download_file(u, dest, retries=2, timeout=timeout)
        if p is not None:
            print(f"  {label}: {st} via {u.split('/')[3]}"
                  f" ({dest.stat().st_size/1e6:.1f} MB)")
            return u
        print(f"  {label}: failed {u}")
    return None

liver_f = PATHS["interim"] / "gtex_liver_log2tpm.parquet"
mode_f  = PATHS["interim"] / "gtex_mode.json"

if liver_f.exists() and mode_f.exists():
    GTEX = pd.read_parquet(liver_f)
    GTEX_MODE = json.loads(mode_f.read_text())["mode"]
    print(f"[cache] GTEx liver {GTEX.shape}, mode={GTEX_MODE}")
else:
    att_f = PATHS["raw_gtex"] / "gtex_v8_sample_attributes.txt"
    tpm_f = PATHS["raw_gtex"] / "gtex_v8_gene_tpm.gct.gz"
    med_f = PATHS["raw_gtex"] / "gtex_v8_gene_median_tpm.gct.gz"
    GTEX_MODE, GTEX = None, None

    att_url = try_urls(ATT_URLS, att_f, "sample attributes", timeout=300)
    if att_url:
        att = pd.read_csv(att_f, sep="\t", low_memory=False)
        liver_ids = set(att.loc[att.SMTSD == "Liver", "SAMPID"].astype(str))
        print(f"  liver samples in annotation: {len(liver_ids)}")
        if try_urls(TPM_URLS, tpm_f, "full TPM matrix"):
            if verify_gzip(tpm_f):
                print("  streaming GCT (several minutes)...")
                chunks, keep = [], None
                for i, ch in enumerate(pd.read_csv(tpm_f, sep="\t", skiprows=2,
                                                   chunksize=4000, low_memory=False)):
                    if keep is None:
                        keep = [c for c in ch.columns if c in liver_ids]
                        print(f"  matched {len(keep)} liver columns")
                    chunks.append(ch[["Name","Description"] + keep])
                    if (i+1) % 5 == 0:
                        print(f"    {(i+1)*4000} genes...")
                g = pd.concat(chunks, ignore_index=True)
                g["symbol"] = g["Description"].astype(str).str.strip()
                g = g[g.symbol.notna() & ~g.symbol.isin(["", "nan"])]
                GTEX = np.log2(g.drop(columns=["Name","Description"])
                                .groupby("symbol").median().clip(lower=0) + 1)
                GTEX_MODE = "full"
            else:
                print("  !! TPM gzip truncated — deleting")
                tpm_f.unlink(missing_ok=True)

    if GTEX is None:                                   # fallback
        print("  falling back to median-TPM-by-tissue")
        if not try_urls(MEDIAN_URLS, med_f, "median TPM", timeout=600):
            raise RuntimeError(
                "All GTEx URLs failed. Check https://gtexportal.org/home/downloads/"
                "adult-gtex/bulk_tissue_expression for the current paths.")
        m = pd.read_csv(med_f, sep="\t", skiprows=2, low_memory=False)
        col = next((c for c in m.columns if c.strip().lower() == "liver"), None)
        assert col, f"no Liver column; available: {list(m.columns)[:12]}"
        m["symbol"] = m["Description"].astype(str).str.strip()
        m = m[m.symbol.notna() & ~m.symbol.isin(["", "nan"])]
        GTEX = np.log2(m.groupby("symbol")[col].median().clip(lower=0) + 1).to_frame("Liver")
        GTEX_MODE = "median"

    GTEX.to_parquet(liver_f)
    mode_f.write_text(json.dumps({"mode": GTEX_MODE,
                                  "n_samples": int(GTEX.shape[1])}))
    print(f"  GTEx liver: {GTEX.shape[0]} genes x {GTEX.shape[1]} cols, mode={GTEX_MODE}")

GTEX_HAS_INDIVIDUALS = (GTEX_MODE == "full")
if not GTEX_HAS_INDIVIDUALS:
    print("  NOTE: median-only mode. Node feature 15 (gtex_expression_std) is")
    print("        unavailable; the tensor will carry 25 features, not 26.")

overlap = sorted(set(GTEX.index) & set(BULK.index))
print(f"\nGTEx x BULK overlap: {len(overlap)} / {BULK.shape[0]} genes")
print(f"  GTEx median {float(np.nanmedian(GTEX.values)):.2f}   "
      f"BULK median {float(np.nanmedian(BULK.values)):.2f}")
print("  Different scales by design — hence rank-based deviation.")
for m_ in ("ALB","COL1A1","TGFB1","ACTA2","TIMP1"):
    if m_ in GTEX.index:
        print(f"  {m_:8s} GTEx mean log2TPM = {float(GTEX.loc[m_].mean()):6.2f}")
assert len(overlap) > 5000, f"only {len(overlap)} shared genes with GTEx"

write_manifest("p06a_gtex",
               inputs={"mode": GTEX_MODE},
               params={"tissue": "Liver", "n_cols": int(GTEX.shape[1]),
                       "has_individuals": GTEX_HAS_INDIVIDUALS,
                       "overlap_genes": len(overlap)},
               outputs=[liver_f])

[cache] GTEx liver (54592, 226), mode=full

GTEx x BULK overlap: 10731 / 10732 genes
  GTEx median 0.01   BULK median 7.73
  Different scales by design — hence rank-based deviation.
  ALB      GTEx mean log2TPM =  14.55
  COL1A1   GTEx mean log2TPM =   4.14
  TGFB1    GTEx mean log2TPM =   3.39
  ACTA2    GTEx mean log2TPM =   4.93
  TIMP1    GTEx mean log2TPM =   6.52
[manifest] wrote manifest_p06a_gtex.json


{'stage': 'p06a_gtex',
 'timestamp': '2026-10-08T11:13:41',
 'inputs': {'mode': 'full'},
 'params': {'tissue': 'Liver',
  'n_cols': 226,
  'has_individuals': True,
  'overlap_genes': 10731},
 'outputs': {'/content/drive/MyDrive/CoDAGAT_preprocessing_fix_v1/interim/gtex_liver_log2tpm.parquet': 'f1c28d2115d6db78e93094e1575a8434a23fbaad38495484b7ac778337647245'}}

In [54]:
# ============================================================
# CELL 27b — DEVIATION PREREQUISITES
# Cell 28 (v3) assumes disp / dev_raw / info / evaluate / passes /
# std_within / z_in_sample already exist from the v2 run. On a clean
# rerun they don't. This rebuilds them from saved artifacts.
# ============================================================
from scipy.stats import kruskal, spearmanr, mannwhitneyu

BULK = pd.read_parquet(PATHS["interim"]/"bulk_expression_log2.parquet")
meta = pd.read_parquet(PATHS["interim"]/"bulk_meta.parquet")
if "is_healthy" not in meta.columns:
    meta["is_healthy"] = meta.resolution == RES_HEALTHY
if "is_supervised" not in meta.columns:
    meta["is_supervised"] = meta[["y_T1_significant","y_T3_early_adv"]].notna().any(axis=1)

DEV_GENES = sorted(set(GTEX.index) & set(BULK.index))
print(f"deviation gene space: {len(DEV_GENES)}")

# ---- rank displacement (identical to Cell 28 v2) ----
gtex_rank = GTEX.loc[DEV_GENES].median(axis=1).rank(pct=True)
disp = BULK.loc[DEV_GENES].rank(axis=0, pct=True).sub(gtex_rank, axis=0)
dev_raw = disp.abs().mean(axis=0).rename("dev_raw")

info = meta.loc[dev_raw.index,
                ["cohort","platform","stage","resolution","tier","is_healthy"]].copy()
info["dev_raw"] = dev_raw

# ---- helpers ----
def std_within(s, key):
    g = info[key]
    return s.groupby(g).transform(lambda v: (v - v.mean()) / (v.std() + 1e-12))

def z_in_sample(d):
    return d.sub(d.mean(axis=0), axis=1) \
            .div(d.std(axis=0).replace(0, np.nan), axis=1).fillna(0.0)

def evaluate(name, s):
    d = info.assign(dev=s)
    ex = d[d.resolution.isin([RES_EXACT, "exact_ishak_mapped"]) & d.stage.notna()]
    num = ex.stage.str[1].astype(float)
    rho, prho = spearmanr(num, ex.dev)
    grp = [ex.loc[ex.stage == st, "dev"].values for st in ["F0","F1","F2","F3","F4"]
           if (ex.stage == st).sum() > 3]
    H, pk = kruskal(*grp)
    m = {st: float(ex.loc[ex.stage == st, "dev"].mean())
         for st in ["F0","F1","F2","F3","F4"]}
    pooled = ex.dev.std() + 1e-12
    hlth = float(d.loc[d.is_healthy, "dev"].mean())
    bw = d.groupby("cohort").dev.mean().std()
    wi = d.groupby("cohort").dev.std().mean()
    return dict(variant=name, rho=rho, p_rho=prho, kruskal_p=pk,
                **{f"mean_{k}": v for k, v in m.items()},
                healthy=hlth,
                f4_minus_f0_sd=(m["F4"] - m["F0"]) / pooled,
                f0_minus_healthy_sd=(m["F0"] - hlth) / pooled,
                cohort_ratio=bw / wi,
                monotone=all(b >= a - 1e-9 for a, b in
                             zip([m[f"F{i}"] for i in range(5)],
                                 [m[f"F{i}"] for i in range(1, 5)])))

def passes(r):
    return (r.rho > 0.15 and r.p_rho < 0.01
            and r.f4_minus_f0_sd > 0.30
            and r.f0_minus_healthy_sd > -0.10
            and r.cohort_ratio < 0.8)

print(f"disp {disp.shape}   dev_raw {dev_raw.shape}   info {info.shape}")
print("helpers defined: std_within, z_in_sample, evaluate, passes")
print(f"platform split: {info.platform.value_counts().to_dict()}")

deviation gene space: 10731
disp (10731, 1514)   dev_raw (1514,)   info (1514, 7)
helpers defined: std_within, z_in_sample, evaluate, passes
platform split: {'rnaseq': 1185, 'microarray': 329}


In [55]:
# ============================================================
# CELL 28 — GTEx-ANCHORED DEVIATION   [v3 — FINAL ATTEMPT]
#
# v1: strict monotonicity failed on a 0.0007 dip. Gate was brittle.
# v2: all variants failed because healthy controls deviate MORE than
#     F0 patients. Cause: mean|rank displacement| over 10,734 genes
#     measures global profile distance (postmortem GTEx vs fresh
#     biopsy), not fibrosis. rho only 0.07 after platform correction.
#
# v3: DIRECTIONAL deviation on a-priori gene sets.
#     fibrosis = (ECM/collagen displacement) - (hepatocyte displacement)
#     Both sets fixed before any label is seen -> no leakage.
#
# DECISION RULE: if no variant passes, Head 3 is dropped (a2=0).
# ============================================================
from scipy.stats import kruskal, spearmanr, mannwhitneyu

# ---------- a-priori gene sets (biological prior, NOT label-selected) ----------
# ---------- a-priori gene sets (biological prior, NOT label-selected) ----------
ECM_UP = ["COL1A1","COL1A2","COL3A1","COL4A1","COL4A2","COL5A1","COL5A2",
          "COL6A1","COL6A2","COL6A3","COL15A1","COL18A1","FN1","LUM","DCN",
          "VCAN","BGN","FBN1","SPARC","THBS1","THBS2","TIMP1","TIMP2","MMP2",
          "MMP14","LOX","LOXL1","LOXL2","ELN","POSTN","TNC","SERPINE1","ACTA2",
          "TAGLN","PDGFRB","TGFB1","CCN2","SPP1","VIM","ITGB1","FBLN1","FBLN2",
          "NID1","LAMB1","LAMC1","EMILIN1","MFAP4","OLFML3","CTHRC1","SULF1"]

# expanded: several classic hepatocyte markers (ALB, APOA1, APOB, FGA, FGB)
# are absent from the intersection because GSE213621 lacks them
HEP_DOWN = [
    # cytochrome P450 / xenobiotic
    "CYP2E1","CYP3A4","CYP1A2","CYP2C9","CYP2C8","CYP2C19","CYP2A6","CYP2B6",
    "CYP2D6","CYP8B1","CYP4A11","CYP7A1","CYP27A1",
    # phase II conjugation
    "UGT1A1","UGT2B7","SULT2A1","GSTA1","NAT2","CES1",
    # urea cycle / amino acid metabolism
    "CPS1","OTC","ASS1","ASL","ARG1","TAT","HAL","SDS","CTH","AGXT","HPD","FAH",
    "GLS2","BHMT","GNMT","MAT1A","TDO2",
    # carbohydrate / lipid metabolism
    "PCK1","ALDOB","FBP1","G6PC1","SLC2A2","ADH1B","ADH1C","ADH4","ACSM2B",
    # secreted plasma proteins
    "SERPINA1","SERPINC1","SERPIND1","TTR","TF","FGG","AHSG","APOH","APOC3",
    "APOA2","APOC1","HPX","ITIH4","AMBP","F2","F7","F9","PROC","AGT","C8A","C9",
    # transporters / nuclear receptors
    "ABCB11","SLCO1B1","SLC10A1","SLC22A1","NR1H4","NR1I2","NR1I3",
    "HNF4A","HNF1A","ONECUT1","FOXA2","GLUL",
]

ecm = [g for g in ECM_UP  if g in disp.index]
hep = [g for g in HEP_DOWN if g in disp.index]
print(f"ECM set:        {len(ecm):3d}/{len(ECM_UP)} present")
print(f"hepatocyte set: {len(hep):3d}/{len(HEP_DOWN)} present")
print(f"  ECM missing: {[g for g in ECM_UP if g not in disp.index]}")
print(f"  hep missing: {[g for g in HEP_DOWN if g not in disp.index][:20]}"
      f"{' ...' if len([g for g in HEP_DOWN if g not in disp.index]) > 20 else ''}")

MIN_ECM, MIN_HEP = 25, 12
assert len(ecm) >= MIN_ECM and len(hep) >= MIN_HEP, (
    f"gene sets too sparse: ECM {len(ecm)}<{MIN_ECM} or hep {len(hep)}<{MIN_HEP}. "
    f"Both sets are fixed a priori; do not add genes to make the metric work.")

dev_dir = (disp.loc[ecm].mean(axis=0) - disp.loc[hep].mean(axis=0)).rename("dev_dir")
variants = {
    "raw":            dev_raw,
    "platform_std":   std_within(dev_raw, "platform"),
    "cohort_std":     std_within(dev_raw, "cohort"),
    "directional":    dev_dir,
    "directional_ps": std_within(dev_dir, "platform"),
    "directional_cs": std_within(dev_dir, "cohort"),
}

res = pd.DataFrame([evaluate(k, v) for k, v in variants.items()])
print("\n[COMPARISON]")
print(res[["variant","rho","p_rho","monotone","f4_minus_f0_sd",
           "f0_minus_healthy_sd","cohort_ratio"]]
      .to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("\nstage means:")
print(res[["variant"]+[f"mean_F{i}" for i in range(5)]+["healthy"]]
      .to_string(index=False, float_format=lambda x: f"{x:.4f}"))

res["passes"] = res.apply(passes, axis=1)
print("\n[GATE] rho>0.15 & p<0.01 & (F4-F0)>0.30sd & (F0-healthy)>-0.10sd & ratio<0.8")
print(res[["variant","passes"]].to_string(index=False))

ok = res[res.passes]
USE_HEAD3 = bool(len(ok))
if USE_HEAD3:
    CHOSEN = ok.sort_values("rho", ascending=False).iloc[0].variant
    print(f"\n  SELECTED: {CHOSEN}  -> Head 3 ENABLED (a2 = 0.3)")
else:
    CHOSEN = "directional"
    print("\n  >>> NO VARIANT PASSES.")
    print("      DECISION: Head 3 (deviation regression) is DROPPED, a2 = 0.")
    print("      The per-gene node feature is RETAINED — it is a within-sample")
    print("      z-score, so the platform offset cancels and it carries")
    print("      gene-level information the scalar target cannot.")
    print("      Report as a negative result: GTEx-anchored sample-level")
    print("      deviation is dominated by technical distance (postmortem")
    print("      reference vs fresh biopsy) and does not track fibrosis stage.")

dev_final = variants[CHOSEN].rename("deviation_score")
dev_per_gene = z_in_sample(disp)

CONFIG_HEAD3 = dict(enabled=USE_HEAD3, variant=CHOSEN,
                    a2=0.3 if USE_HEAD3 else 0.0)
(PATHS["config"]/"head3.json").write_text(json.dumps(CONFIG_HEAD3, indent=2))

pd.DataFrame({"deviation_score": dev_final, "dev_raw": dev_raw,
              "dev_directional": dev_dir}) \
  .to_parquet(PATHS["interim"]/"deviation_sample.parquet")
dev_per_gene.to_parquet(PATHS["interim"]/"deviation_per_gene_all.parquet")

# ---------- report on the chosen variant ----------
d = info.assign(dev=dev_final)
ex = d[d.resolution.isin([RES_EXACT,"exact_ishak_mapped"]) & d.stage.notna()]
print(f"\n[FINAL: {CHOSEN}]  Head3={'ON' if USE_HEAD3 else 'OFF'}")
print(f"{'stage':8s} {'n':>5s} {'mean':>9s} {'sd':>8s}")
for st in ["F0","F1","F2","F3","F4"]:
    v = ex.loc[ex.stage == st, "dev"]
    if len(v):
        print(f"{st:8s} {len(v):5d} {v.mean():9.4f} {v.std():8.4f}")
hv = d.loc[d.is_healthy, "dev"]
print(f"{'healthy':8s} {len(hv):5d} {hv.mean():9.4f} {hv.std():8.4f}")
u, pu = mannwhitneyu(ex.loc[ex.stage.isin(["F0","F1"]), "dev"],
                     ex.loc[ex.stage.isin(["F2","F3","F4"]), "dev"])
print(f"\n  F0/F1 vs F2+ Mann-Whitney p = {pu:.3e}  (primary task boundary)")

# ---------- node feature sanity (retained regardless of Head 3) ----------
print("\n[NODE FEATURE CHECK] per-gene deviation on marker genes")
for g_ in ["COL1A1","COL3A1","TIMP1","ACTA2","CYP2E1","CPS1"]:
    if g_ not in dev_per_gene.index:
        continue
    s = dev_per_gene.loc[g_, ex.index]
    r, p_ = spearmanr(ex.stage.str[1].astype(float), s)
    arrow = "up" if r > 0 else "down"
    print(f"  {g_:8s} rho={r:+.3f} p={p_:.1e}  ({arrow} with stage)")
print("  ECM genes should be positive, hepatocyte genes negative.")

write_manifest("p06_deviation",
               inputs={"gtex":"gtex_liver_log2tpm.parquet",
                       "bulk":"bulk_expression_log2.parquet"},
               params={"method":"rank_displacement","variant":CHOSEN,
                       "head3_enabled":USE_HEAD3,"n_genes":len(DEV_GENES),
                       "n_ecm":len(ecm),"n_hep":len(hep),
                       "comparison":res.drop(columns=["passes"]).to_dict("records")},
               outputs=[PATHS["interim"]/"deviation_sample.parquet",
                        PATHS["interim"]/"deviation_per_gene_all.parquet",
                        PATHS["config"]/"head3.json"])

ECM set:         49/50 present
hepatocyte set:  38/78 present
  ECM missing: ['CCN2']
  hep missing: ['CYP3A4', 'CYP2C8', 'CYP2C19', 'CYP2B6', 'CYP8B1', 'CYP4A11', 'CYP7A1', 'UGT1A1', 'UGT2B7', 'SULT2A1', 'GSTA1', 'OTC', 'TAT', 'AGXT', 'BHMT', 'PCK1', 'FBP1', 'G6PC1', 'SLC2A2', 'ADH1B'] ...

[COMPARISON]
       variant    rho  p_rho  monotone  f4_minus_f0_sd  f0_minus_healthy_sd  cohort_ratio
           raw 0.2029 0.0000     False          0.7339              -0.6917        1.0532
  platform_std 0.0713 0.0200      True          0.3014              -0.5292        0.0921
    cohort_std 0.1033 0.0007      True          0.4157              -0.4557        0.0000
   directional 0.4783 0.0000      True          1.3875              -0.1463        0.5527
directional_ps 0.4144 0.0000      True          1.2019              -0.0634        0.0129
directional_cs 0.4127 0.0000      True          1.1906              -0.0923        0.0000

stage means:
       variant  mean_F0  mean_F1  mean_F2  mean_F3

{'stage': 'p06_deviation',
 'timestamp': '2026-10-08T11:13:51',
 'inputs': {'gtex': 'gtex_liver_log2tpm.parquet',
  'bulk': 'bulk_expression_log2.parquet'},
 'params': {'method': 'rank_displacement',
  'variant': 'directional_ps',
  'head3_enabled': True,
  'n_genes': 10731,
  'n_ecm': 49,
  'n_hep': 38,
  'comparison': [{'variant': 'raw',
    'rho': 0.20290875109465686,
    'p_rho': 2.4327127181526334e-11,
    'kruskal_p': 3.743792173611449e-14,
    'mean_F0': 0.14824722524588255,
    'mean_F1': 0.1560907546133193,
    'mean_F2': 0.15847220387366254,
    'mean_F3': 0.1577260557374,
    'mean_F4': 0.1592928055869617,
    'healthy': 0.15865837893228588,
    'f4_minus_f0_sd': 0.7338625261093421,
    'f0_minus_healthy_sd': -0.6917115541318914,
    'cohort_ratio': 1.0532476584909547,
    'monotone': False},
   {'variant': 'platform_std',
    'rho': 0.07134452378312967,
    'p_rho': 0.02000136794110593,
    'kruskal_p': 0.10677635918209998,
    'mean_F0': -0.2341562673649271,
    'mean_F1':

In [56]:
# ============================================================
# CELL 29 — DEVIATION AUDIT + STRING GRAPH
# Part A: audit the a-priori sets. DIAGNOSTIC ONLY — genes are
#         never removed on the basis of stage correlation, or the
#         prior becomes label-selected.
# Part B: 72-gene panel + STRING PPI subgraph + topology features.
# ============================================================
from scipy.stats import spearmanr

# ---------------- Part A ----------------
ex_idx = ex.index
stg = ex.stage.str[1].astype(float)

rows = []
for g in ecm + hep:
    r, p_ = spearmanr(stg, disp.loc[g, ex_idx])
    rows.append(dict(gene=g, set="ECM" if g in ecm else "HEP",
                     rho=r, p=p_,
                     expected="+" if g in ecm else "-",
                     agrees=(r > 0) if g in ecm else (r < 0)))
au = pd.DataFrame(rows)
print("A-PRIORI SET AUDIT (diagnostic — no genes are removed)")
print(au.groupby("set").agrees.agg(["sum", "count", "mean"])
        .rename(columns={"sum": "agree", "count": "n", "mean": "rate"})
        .to_string(float_format=lambda x: f"{x:.2f}"))
print("\ngenes contradicting their prior direction:")
print(au[~au.agrees].sort_values("p").head(12)
        .to_string(index=False, float_format=lambda x: f"{x:.3f}"))
au.to_csv(PATHS["reports"]/"deviation_geneset_audit.csv", index=False)
print("\nBoth sets were fixed before any label was seen. Contradicting genes")
print("are retained. State this in the methods — removing them would make")
print("the deviation feature label-selected and therefore leaky.")

# ---------------- Part B: STRING ----------------
STRING_LINKS = ("https://stringdb-downloads.org/download/protein.links.v12.0/"
                "9606.protein.links.v12.0.txt.gz")
STRING_INFO  = ("https://stringdb-downloads.org/download/protein.info.v12.0/"
                "9606.protein.info.v12.0.txt.gz")
STRING_ALT   = [("https://stringdb-static.org/download/protein.links.v11.5/"
                 "9606.protein.links.v11.5.txt.gz"),
                ("https://stringdb-static.org/download/protein.info.v11.5/"
                 "9606.protein.info.v11.5.txt.gz")]

links_f = PATHS["raw_string"]/"string_links.txt.gz"
info_f  = PATHS["raw_string"]/"string_info.txt.gz"
u1 = try_urls([STRING_LINKS, STRING_ALT[0]], links_f, "STRING links", timeout=900)
u2 = try_urls([STRING_INFO,  STRING_ALT[1]], info_f,  "STRING info",  timeout=300)
assert u1 and u2, ("STRING download failed. Check current paths at "
                   "https://string-db.org/cgi/download")

sinfo = pd.read_csv(info_f, sep="\t", compression="gzip")
pid_col = next(c for c in sinfo.columns if "protein_external_id" in c or c == "#string_protein_id")
name_col = next(c for c in sinfo.columns if "preferred_name" in c)
prot2sym = dict(zip(sinfo[pid_col].astype(str), sinfo[name_col].astype(str)))
print(f"\nSTRING proteins: {len(prot2sym)}")

# ---------------- 72-gene panel ----------------
PATHWAYS = {
    "ECM_remodeling": ["COL1A1","COL1A2","COL3A1","COL4A1","COL5A1","COL6A1",
                       "FN1","LUM","DCN","VCAN","BGN","SPARC","MMP2","MMP14",
                       "TIMP1","TIMP2","LOX","LOXL2","POSTN","THBS2"],
    "TGFB_signaling": ["TGFB1","TGFB2","TGFBR1","TGFBR2","SMAD2","SMAD3","SMAD4",
                       "SMAD7","SERPINE1","CCN2","THBS1","LTBP1"],
    "HSC_activation": ["ACTA2","TAGLN","PDGFRB","PDGFB","DES","VIM","COL1A1",
                       "TIMP1","MYH11","CALD1","NOTCH3","RGS5"],
    "Inflammation":   ["CD68","CCL2","CCL5","CXCL9","CXCL10","IL6","TNF","IL1B",
                       "CD163","MARCO","ITGAX","TLR4","NFKB1","SPP1","CSF1R"],
    "Apoptosis_stress":["CASP3","CASP8","BAX","BCL2","TP53","FAS","XBP1","DDIT3",
                       "ATF4","HSPA5","GADD45A","CDKN1A"],
    "Hepatocyte_function":["CYP2E1","CYP1A2","CYP2C9","CPS1","ASS1","ARG1",
                       "SERPINA1","TTR","TF","HNF4A","ALDOB","GLUL","APOH",
                       "AHSG","ASGR1","NR1H4"],
}
avail = set(BULK.index)
panel = []
for p, gs in PATHWAYS.items():
    got = [g for g in gs if g in avail]
    print(f"  {p:22s} {len(got):2d}/{len(gs):2d}")
    panel += got
panel = sorted(set(panel))
print(f"\npathway-member genes available: {len(panel)}")

# top up to 72 with high-variance fibrosis-relevant genes, ranked by
# cross-cohort expression variance — NO label information used
if len(panel) < N_PANEL_GENES:
    extra_pool = [g for g in (ECM_UP + HEP_DOWN) if g in avail and g not in panel]
    var = BULK.loc[extra_pool].var(axis=1).sort_values(ascending=False)
    add = list(var.index[:N_PANEL_GENES - len(panel)])
    print(f"topping up with {len(add)} high-variance genes (label-free): {add}")
    panel = sorted(panel + add)
panel = panel[:N_PANEL_GENES]
print(f"FINAL PANEL: {len(panel)} genes")

pd.DataFrame({"gene": panel,
              "pathways": [";".join(p for p, gs in PATHWAYS.items() if g in gs) or "extra"
                           for g in panel]}) \
  .to_csv(PATHS["interim"]/"panel_genes.csv", index=False)

# ---------------- STRING subgraph ----------------
panel_set = set(panel)
keep_pids = {p for p, s in prot2sym.items() if s in panel_set}
print(f"\npanel genes with a STRING protein id: "
      f"{len({prot2sym[p] for p in keep_pids})}/{len(panel)}")

edges = []
for ch in pd.read_csv(links_f, sep=" ", compression="gzip", chunksize=2_000_000):
    ch = ch[ch.combined_score >= STRING_THRESHOLD]
    ch = ch[ch.protein1.isin(keep_pids) & ch.protein2.isin(keep_pids)]
    if len(ch):
        edges.append(ch)
E = pd.concat(edges, ignore_index=True) if edges else pd.DataFrame()
E["g1"] = E.protein1.map(prot2sym); E["g2"] = E.protein2.map(prot2sym)
E = E[E.g1 != E.g2]

# STRING lists both directions; collapse to undirected
E["key"] = E.apply(lambda r: tuple(sorted((r.g1, r.g2))), axis=1)
und = E.groupby("key").combined_score.max().reset_index()
und["g1"] = und.key.map(lambda t: t[0]); und["g2"] = und.key.map(lambda t: t[1])
print(f"\nSTRING edges at score>={STRING_THRESHOLD}: "
      f"{len(und)} undirected ({2*len(und)} directed rows)")
print(f"  Phase 2 reported 405 undirected / 810 directed on a 72-gene panel.")

G = nx.Graph()
G.add_nodes_from(panel)
G.add_edges_from([(r.g1, r.g2, {"score": r.combined_score}) for r in und.itertuples()])
iso = [n for n in G.nodes if G.degree(n) == 0]
print(f"  isolated nodes: {len(iso)} -> {iso}")
print("  Isolated nodes are RETAINED: they carry patient-specific features and")
print("  contribute through self-loops and global pooling.")

gi = {g: i for i, g in enumerate(panel)}
edge_df = pd.DataFrame([
    dict(src_idx=gi[a], dst_idx=gi[b], src=a, dst=b,
         string_score=int(s), weight=1.0)
    for a, b, s in [(r.g1, r.g2, r.combined_score) for r in und.itertuples()]
    for a, b, s in [(a, b, s), (b, a, s)]      # both directions for PyG
])
edge_df.to_csv(PATHS["interim"]/"graph_edges.csv", index=False)

topo = pd.DataFrame({
    "gene": panel,
    "degree_centrality":      [nx.degree_centrality(G)[g] for g in panel],
    "betweenness_centrality": [nx.betweenness_centrality(G)[g] for g in panel],
    "closeness_centrality":   [nx.closeness_centrality(G)[g] for g in panel],
    "clustering_coefficient": [nx.clustering(G)[g] for g in panel],
    "pathway_membership_count":[sum(g in gs for gs in PATHWAYS.values())/len(PATHWAYS)
                                for g in panel],
}).set_index("gene")
topo.to_csv(PATHS["interim"]/"graph_topology.csv")
print(f"\ntopology features: {topo.shape}")
print(topo.describe().T[["mean","std","min","max"]]
        .to_string(float_format=lambda x: f"{x:.4f}"))

write_manifest("p07_graph",
               inputs={"string_links": u1, "string_info": u2},
               params={"threshold": STRING_THRESHOLD, "n_panel": len(panel),
                       "n_edges_undirected": int(len(und)),
                       "n_isolated": len(iso)},
               outputs=[PATHS["interim"]/"panel_genes.csv",
                        PATHS["interim"]/"graph_edges.csv",
                        PATHS["interim"]/"graph_topology.csv",
                        PATHS["reports"]/"deviation_geneset_audit.csv"])

A-PRIORI SET AUDIT (diagnostic — no genes are removed)
     agree   n  rate
set                 
ECM     47  49  0.96
HEP     20  38  0.53

genes contradicting their prior direction:
    gene set    rho     p expected  agrees
SERPINA1 HEP  0.323 0.000        -   False
   POSTN ECM -0.223 0.000        +   False
    CPS1 HEP  0.200 0.000        -   False
    GLUL HEP  0.197 0.000        -   False
     AGT HEP  0.192 0.000        -   False
   ALDOB HEP  0.183 0.000        -   False
    ARG1 HEP  0.173 0.000        -   False
      TF HEP  0.172 0.000        -   False
  CYP2C9 HEP  0.167 0.000        -   False
  CYP2D6 HEP  0.166 0.000        -   False
     FAH HEP  0.141 0.000        -   False
     SDS HEP  0.136 0.000        -   False

Both sets were fixed before any label was seen. Contradicting genes
are retained. State this in the methods — removing them would make
the deviation feature label-selected and therefore leaky.
  STRING links: cached via download (83.2 MB)
  STRING info: cac

{'stage': 'p07_graph',
 'timestamp': '2026-10-08T11:14:15',
 'inputs': {'string_links': 'https://stringdb-downloads.org/download/protein.links.v12.0/9606.protein.links.v12.0.txt.gz',
  'string_info': 'https://stringdb-downloads.org/download/protein.info.v12.0/9606.protein.info.v12.0.txt.gz'},
 'params': {'threshold': 700,
  'n_panel': 72,
  'n_edges_undirected': 321,
  'n_isolated': 6},
 'outputs': {'/content/drive/MyDrive/CoDAGAT_preprocessing_fix_v1/interim/panel_genes.csv': '7d5e20c7d5ea3bd2c212a9a1a2ad792354f0bddf07d00bf3da54737ec4fadaa8',
  '/content/drive/MyDrive/CoDAGAT_preprocessing_fix_v1/interim/graph_edges.csv': '62da8a6671e387bf95843e797a035667a5131cc617826f0bdc5ea5c336c48fa6',
  '/content/drive/MyDrive/CoDAGAT_preprocessing_fix_v1/interim/graph_topology.csv': 'da5faaf3c3ade145fb2546f4746cede3ba3a75287ce129db45a2411fe27dedbd',
  '/content/drive/MyDrive/CoDAGAT_preprocessing_fix_v1/reports/deviation_geneset_audit.csv': '6c566c98e30d4069ae1bd2e0f71281217e2eff3cc514cffe67f4f64

In [57]:
# ============================================================
# CELL 30 — VARIANT STABILITY + PATHWAY ACTIVITY + NODE TENSOR (A+B)
#  A. leave-one-cohort-out check on the deviation variant choice
#  B. leave-one-out pathway activity per gene (node-aware, not broadcast)
#  C. Group A (8 dynamic) + Group B (9 static) -> (N, 72, 17)
#     Group C (9 fold-fitted) is NEVER precomputed.
# ============================================================
from scipy.stats import spearmanr

panel = pd.read_csv(PATHS["interim"]/"panel_genes.csv").gene.tolist()
topo  = pd.read_csv(PATHS["interim"]/"graph_topology.csv", index_col=0).loc[panel]
sup   = pd.read_parquet(PATHS["interim"]/"bulk_meta_supervised.parquet")
BULK_SUP = pd.read_parquet(PATHS["interim"]/"bulk_supervised_combat.parquet")

# ---------------- A. variant stability ----------------
print("VARIANT SELECTION STABILITY (leave-one-cohort-out)")
print("Addresses: was 'directional_ps' chosen using held-out label information?")
folds = sorted(sup.loc[sup.tier == "T1", "cohort"].unique())
stab = []
for f in folds:
    keep = info.index[info.cohort != f]
    sub_v = {k: v.loc[[i for i in keep if i in v.index]] for k, v in variants.items()}
    sub_info = info.loc[keep]
    saved = info
    try:
        globals()["info"] = sub_info
        r = pd.DataFrame([evaluate(k, v) for k, v in sub_v.items()])
    finally:
        globals()["info"] = saved
    r["passes"] = r.apply(passes, axis=1)
    p_ = r[r.passes]
    win = (p_.sort_values("rho", ascending=False).iloc[0].variant if len(p_)
           else "NONE_PASS")
    stab.append(dict(excluded=f, winner=win,
                     rho=float(r.set_index("variant").loc[CHOSEN, "rho"])))
stab = pd.DataFrame(stab)
print(stab.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
same = (stab.winner == CHOSEN).mean()
print(f"\n  '{CHOSEN}' wins in {int((stab.winner==CHOSEN).sum())}/{len(stab)} "
      f"leave-one-cohort-out recomputations ({same:.0%})")
print("  If 100%, the choice does not depend on any single held-out cohort")
print("  and can be reported as a fixed preprocessing decision.")
stab.to_csv(PATHS["reports"]/"deviation_variant_stability.csv", index=False)

# ---------------- B. pathway activity, leave-one-out ----------------
print("\nPATHWAY ACTIVITY (leave-one-out per gene)")
PW_NAMES = list(PATHWAYS.keys())
members = {p: [g for g in gs if g in panel] for p, gs in PATHWAYS.items()}
for p in PW_NAMES:
    print(f"  {p:22s} {len(members[p]):2d} panel members")

# within-cohort z-score of panel expression (no labels used)
panel_expr = BULK_SUP.loc[panel]
z = panel_expr.copy()
for c in sup.cohort.unique():
    cols = sup.index[sup.cohort == c]
    cols = [x for x in cols if x in z.columns]
    blk = z[cols]
    z[cols] = blk.sub(blk.mean(axis=1), axis=0).div(
        blk.std(axis=1).replace(0, np.nan), axis=0)
z = z.fillna(0.0)
print(f"  expression_z: {z.shape}, "
      f"mean={float(z.values.mean()):.3f}, sd={float(z.values.std()):.3f}")

N, P = z.shape[1], len(panel)
pw = np.zeros((N, P, len(PW_NAMES)), dtype=np.float32)
gidx = {g: i for i, g in enumerate(panel)}
for k, p in enumerate(PW_NAMES):
    mem = members[p]
    if not mem:
        continue
    M = z.loc[mem].values                       # (n_mem, N)
    tot, n_m = M.sum(axis=0), len(mem)
    for g in mem:
        i = gidx[g]
        loo = (tot - z.loc[g].values) / max(n_m - 1, 1)   # exclude self
        pw[:, i, k] = loo
print(f"  pathway tensor {pw.shape}; non-members are exactly 0 by design")
nz = (pw != 0).any(axis=0).sum(axis=0)
print("  genes with non-zero activity per pathway:", dict(zip(PW_NAMES, nz)))

# ---------------- C. assemble Group A + B ----------------
dev_pg = pd.read_parquet(PATHS["interim"]/"deviation_per_gene_all.parquet")
dev_panel = dev_pg.loc[panel, z.columns]

FEATURES_A = ["expression_z", "deviation_rank_z"] + [f"pw_{p}" for p in PW_NAMES]
FEATURES_B = ["pathway_membership_count", "degree_centrality",
              "betweenness_centrality", "closeness_centrality",
              "clustering_coefficient", "gtex_expression_level",
              "gtex_expression_std", "expression_variability",
              "cross_platform_consistency"]

gl = GTEX.loc[[g for g in panel if g in GTEX.index]]
gtex_level = pd.Series({g: float(gl.loc[g].mean()) if g in gl.index else np.nan
                        for g in panel})
if GTEX_HAS_INDIVIDUALS:
    gtex_std = pd.Series({g: float(GTEX.loc[g].std()) if g in GTEX.index else np.nan
                          for g in panel})
else:
    gtex_std = pd.Series(0.0, index=panel)
    FEATURES_B = [f for f in FEATURES_B if f != "gtex_expression_std"]

norm = lambda s: (s - s.min()) / (s.max() - s.min() + 1e-12)
gtex_level = norm(gtex_level.fillna(gtex_level.median()))
gtex_std   = norm(gtex_std.fillna(gtex_std.median())) if GTEX_HAS_INDIVIDUALS else gtex_std

expr_var = norm((panel_expr.std(axis=1) / (panel_expr.mean(axis=1).abs() + 1e-9)))

rna_c = [c for c in z.columns if sup.loc[c, "platform"] == "rnaseq"]
mic_c = [c for c in z.columns if sup.loc[c, "platform"] == "microarray"]
rmean = panel_expr[rna_c].mean(axis=1).rank()
mmean = panel_expr[mic_c].mean(axis=1).rank()
xplat = pd.Series(1.0 - (rmean - mmean).abs() / len(panel), index=panel)
print(f"\n  cross_platform_consistency: mean={xplat.mean():.4f} "
      f"sd={xplat.std():.4f}  (near-constant means a dead feature)")
if xplat.std() < 0.01:
    print("  >>> near-constant. Consider reporting 25 features, not 26.")

static = pd.DataFrame({
    "pathway_membership_count": topo.pathway_membership_count.values,
    "degree_centrality":        topo.degree_centrality.values,
    "betweenness_centrality":   topo.betweenness_centrality.values,
    "closeness_centrality":     topo.closeness_centrality.values,
    "clustering_coefficient":   topo.clustering_coefficient.values,
    "gtex_expression_level":    gtex_level.loc[panel].values,
    "gtex_expression_std":      gtex_std.loc[panel].values,
    "expression_variability":   expr_var.loc[panel].values,
    "cross_platform_consistency": xplat.loc[panel].values,
}, index=panel)[FEATURES_B]

A = np.zeros((N, P, len(FEATURES_A)), dtype=np.float32)
A[:, :, 0] = z.values.T
A[:, :, 1] = dev_panel.values.T
A[:, :, 2:] = pw
B = np.broadcast_to(static.values.astype(np.float32)[None, :, :],
                    (N, P, len(FEATURES_B)))
TENSOR_AB = np.concatenate([A, B], axis=2)

assert_tensor_integrity(TENSOR_AB, "TENSOR_AB")
np.save(PATHS["processed"]/"node_features_AB.npy", TENSOR_AB)
pd.Series(list(z.columns), name="sample_id").to_csv(
    PATHS["processed"]/"tensor_sample_order.csv", index=False)
pd.DataFrame({"idx": range(len(FEATURES_A+FEATURES_B)),
              "feature": FEATURES_A + FEATURES_B,
              "group": ["A"]*len(FEATURES_A) + ["B"]*len(FEATURES_B),
              "leakage_class": ["dynamic_global"]*len(FEATURES_A) +
                               ["static_global"]*len(FEATURES_B)}) \
  .to_csv(PATHS["processed"]/"feature_schema_AB.csv", index=False)

print(f"\nTENSOR A+B: {TENSOR_AB.shape}  "
      f"({len(FEATURES_A)} dynamic + {len(FEATURES_B)} static)")
print(f"  Group C adds 9 fold-fitted features at runtime -> "
      f"{len(FEATURES_A)+len(FEATURES_B)+9} total")
print(f"  NaN={int(np.isnan(TENSOR_AB).sum())} Inf={int(np.isinf(TENSOR_AB).sum())}")
print("\nper-feature stats:")
for i, f in enumerate(FEATURES_A + FEATURES_B):
    v = TENSOR_AB[:, :, i]
    print(f"  {i:2d} {f:28s} mean={v.mean():8.4f} sd={v.std():7.4f} "
          f"min={v.min():8.3f} max={v.max():8.3f}")

write_manifest("p08_tensor_AB",
               inputs={"panel": "panel_genes.csv", "topology": "graph_topology.csv",
                       "deviation": "deviation_per_gene_all.parquet"},
               params={"n_samples": int(N), "n_nodes": int(P),
                       "n_features_AB": int(TENSOR_AB.shape[2]),
                       "variant_stability": float(same),
                       "gtex_individuals": bool(GTEX_HAS_INDIVIDUALS)},
               outputs=[PATHS["processed"]/"node_features_AB.npy",
                        PATHS["processed"]/"feature_schema_AB.csv",
                        PATHS["processed"]/"tensor_sample_order.csv",
                        PATHS["reports"]/"deviation_variant_stability.csv"])

VARIANT SELECTION STABILITY (leave-one-cohort-out)
Addresses: was 'directional_ps' chosen using held-out label information?
 excluded         winner    rho
GSE130970 directional_ps 0.4147
GSE135251 directional_ps 0.3988
GSE162694      NONE_PASS 0.4258
GSE193066 directional_ps 0.4168
GSE213621      NONE_PASS 0.4264
GSE240729 directional_ps 0.3986
 GSE31803 directional_ps 0.4144
 GSE84044    directional 0.3824

  'directional_ps' wins in 5/8 leave-one-cohort-out recomputations (62%)
  If 100%, the choice does not depend on any single held-out cohort
  and can be reported as a fixed preprocessing decision.

PATHWAY ACTIVITY (leave-one-out per gene)
  ECM_remodeling         20 panel members
  TGFB_signaling         10 panel members
  HSC_activation         11 panel members
  Inflammation           10 panel members
  Apoptosis_stress       11 panel members
  Hepatocyte_function    12 panel members
  expression_z: (72, 1366), mean=-0.000, sd=0.996
  pathway tensor (1366, 72, 6); non-members 

{'stage': 'p08_tensor_AB',
 'timestamp': '2026-10-08T11:14:18',
 'inputs': {'panel': 'panel_genes.csv',
  'topology': 'graph_topology.csv',
  'deviation': 'deviation_per_gene_all.parquet'},
 'params': {'n_samples': 1366,
  'n_nodes': 72,
  'n_features_AB': 17,
  'variant_stability': 0.625,
  'gtex_individuals': True},
 'outputs': {'/content/drive/MyDrive/CoDAGAT_preprocessing_fix_v1/processed/node_features_AB.npy': 'a35abef4b70082f4dee5f937151f7c8d2cc2e594e06af4f7431309b23aefac55',
  '/content/drive/MyDrive/CoDAGAT_preprocessing_fix_v1/processed/feature_schema_AB.csv': 'b838f24141fd6ee58d26bf90b596173609c092f26ce193bc58b8ddd4aafd33af',
  '/content/drive/MyDrive/CoDAGAT_preprocessing_fix_v1/processed/tensor_sample_order.csv': '523ae8e206434257350b98b021f30db94eed91167da4f30f3f01939bdc4f01be',
  '/content/drive/MyDrive/CoDAGAT_preprocessing_fix_v1/reports/deviation_variant_stability.csv': 'febd5d39f7d4d3d16b0fbafe8d8769ef36551f9004c2b1e50e10132617c3dd05'}}

In [58]:
# ============================================================
# CELL 30b — SELECTION-SET HELPERS
# Cell 31 (v2) assumes evaluate_subset / t1_idx / t2_idx exist
# from the v1 run. On a clean rerun they don't.
# ============================================================
t1_idx = info.index[info.tier == "T1"]
t2_idx = info.index[info.tier == "T2"]

def evaluate_subset(name, s, idx):
    """Run evaluate() restricted to a subset of samples."""
    saved = info
    try:
        globals()["info"] = info.loc[idx]
        return evaluate(name, s.loc[[i for i in idx if i in s.index]])
    finally:
        globals()["info"] = saved

print(f"T1: {len(t1_idx)} samples / {info.loc[t1_idx,'cohort'].nunique()} cohorts")
print(f"T2: {len(t2_idx)} samples / {info.loc[t2_idx,'cohort'].nunique()} cohorts")
print(f"variants available: {list(variants)}")
assert "directional" in variants and "directional_ps" in variants, \
    "run Cell 28 first — the directional variants are missing"

T1: 1174 samples / 8 cohorts
T2: 340 samples / 4 cohorts
variants available: ['raw', 'platform_std', 'cohort_std', 'directional', 'directional_ps', 'directional_cs']


In [59]:
# ============================================================
# CELL 31 (commit block) — DEFER, DON'T DECIDE
# The gate's healthy criterion is unsound for this data: the
# "controls" are bariatric-surgery, obesity-matched, and
# normal-histology-NAFLD patients, not healthy donor liver.
# So Head 3 becomes a 3-arm ablation rather than a preprocessing
# guess. All targets are saved; the model config picks.
# ============================================================

HEAD3_ARMS = {
    "off":            dict(enabled=False, variant=None,             a2=0.0),
    "directional":    dict(enabled=True,  variant="directional",    a2=0.3),
    "directional_ps": dict(enabled=True,  variant="directional_ps", a2=0.3),
}

dev_out = pd.DataFrame({k: variants[k] for k in
                        ["raw","platform_std","cohort_std",
                         "directional","directional_ps","directional_cs"]})
dev_out["deviation_score"] = dev_out["directional"]     # default arm target
dev_out.to_parquet(PATHS["interim"]/"deviation_sample.parquet")

readout = {}
for name, cfg in HEAD3_ARMS.items():
    if cfg["variant"]:
        r = evaluate_subset(cfg["variant"], variants[cfg["variant"]], t1_idx)
        readout[name] = {k: float(v) for k, v in r.items()
                         if k in ("rho","p_rho","f4_minus_f0_sd","cohort_ratio")}

(PATHS["config"]/"head3.json").write_text(json.dumps(dict(
    arms=HEAD3_ARMS,
    default_arm="directional",
    decision="deferred_to_ablation_row_B",
    rationale=("Gate criterion f0_vs_healthy is unsound: control samples in "
               "GSE48452 (bariatric surgery), GSE89632 (obesity-matched) and "
               "GSE162694 (normal-histology NAFLD) are not healthy donor liver. "
               "Head 3 is therefore selected empirically via held-out coverage, "
               "not by a preprocessing gate."),
    t1_readout=readout), indent=2))

print("HEAD 3 -> 3-ARM ABLATION (frozen-spec row B)")
for name, cfg in HEAD3_ARMS.items():
    r = readout.get(name, {})
    print(f"  {name:16s} a2={cfg['a2']:.1f}  "
          + (f"rho={r['rho']:+.3f}  F4-F0={r['f4_minus_f0_sd']:.2f}sd  "
             f"cohort_ratio={r['cohort_ratio']:.3f}" if r else "(no target)"))

print("\nUNCHANGED BY THIS DECISION")
print(f"  cohorts registered : {len(COHORTS)}")
print(f"  cohorts loaded     : {len(EXPR)}")
print(f"  supervised samples : {len(sup)}")
print(f"  T1 LODO folds      : {sup[sup.tier=='T1'].cohort.nunique()}")
print(f"  node feature 1     : deviation_rank_z, RETAINED")
print(f"  GTEx reference     : {GTEX.shape[1]} healthy livers, RETAINED")

print("\nMETHODS SENTENCE")
print("  'Three configurations of the deviation regression head were compared")
print("   as an ablation: disabled, and enabled with each of two GTEx-anchored")
print("   directional targets. Selection was made on held-out coverage rather")
print("   than on a preprocessing criterion, because the available control")
print("   samples are not healthy donor liver and cannot support one.'")

write_manifest("p06c_variant_selection",
               inputs={"deviation": "deviation_sample.parquet"},
               params={"decision": "deferred_to_ablation", "arms": HEAD3_ARMS,
                       "t1_readout": readout},
               outputs=[PATHS["config"]/"head3.json",
                        PATHS["interim"]/"deviation_sample.parquet",
                        PATHS["reports"]/"deviation_selection_T2.csv",
                        PATHS["reports"]/"deviation_readout_T1.csv"])

HEAD 3 -> 3-ARM ABLATION (frozen-spec row B)
  off              a2=0.0  (no target)
  directional      a2=0.3  rho=+0.442  F4-F0=1.81sd  cohort_ratio=0.503
  directional_ps   a2=0.3  rho=+0.464  F4-F0=2.00sd  cohort_ratio=0.010

UNCHANGED BY THIS DECISION
  cohorts registered : 28
  cohorts loaded     : 12
  supervised samples : 1366
  T1 LODO folds      : 8
  node feature 1     : deviation_rank_z, RETAINED
  GTEx reference     : 226 healthy livers, RETAINED

METHODS SENTENCE
  'Three configurations of the deviation regression head were compared
   as an ablation: disabled, and enabled with each of two GTEx-anchored
   directional targets. Selection was made on held-out coverage rather
   than on a preprocessing criterion, because the available control
   samples are not healthy donor liver and cannot support one.'
[manifest] wrote manifest_p06c_variant_selection.json


{'stage': 'p06c_variant_selection',
 'timestamp': '2026-10-08T11:14:18',
 'inputs': {'deviation': 'deviation_sample.parquet'},
 'params': {'decision': 'deferred_to_ablation',
  'arms': {'off': {'enabled': False, 'variant': None, 'a2': 0.0},
   'directional': {'enabled': True, 'variant': 'directional', 'a2': 0.3},
   'directional_ps': {'enabled': True,
    'variant': 'directional_ps',
    'a2': 0.3}},
  't1_readout': {'directional': {'rho': 0.44238295535832406,
    'p_rho': 1.1588658379808996e-39,
    'f4_minus_f0_sd': 1.8138772927944966,
    'cohort_ratio': 0.5033141874963118},
   'directional_ps': {'rho': 0.4644732452214842,
    'p_rho': 4.704067513671556e-44,
    'f4_minus_f0_sd': 2.003542620073791,
    'cohort_ratio': 0.010244874812619402}}},
 'outputs': {'/content/drive/MyDrive/CoDAGAT_preprocessing_fix_v1/config/head3.json': 'f20720345962d2c1c72557a04088168b0582b7823bed5081d89d57a2e8e50110',
  '/content/drive/MyDrive/CoDAGAT_preprocessing_fix_v1/interim/deviation_sample.parquet': 

In [60]:
# ============================================================
# CELL R — RESTORE STATE FOR THE EXPORT (fresh runtime)
# Rebuilds, from files already on Drive, every object Cells 32-34
# need. Recomputes NOTHING upstream: no download, no expression
# loading, no ComBat, no feature building.
#
# Fresh-runtime order:
#   Cell 2, Cell 3, Cell 4, Cell 7, THIS CELL, Cell 32,
#   Cell 33, (Cell F1), Cell 33b, Cell 34, Cell 34-VERIFY
# Cell 4 must run BEFORE this cell: its self-test expects the
# pre-promotion registry.
# ============================================================
from scipy.stats import spearmanr, ttest_ind

_I, _P = PATHS["interim"], PATHS["processed"]

# ---- 0. keep the previous handoff so 34-VERIFY can compare against it ----
_old_pkg, _bak_pkg = PATHS["root"]/"handoff", PATHS["root"]/"handoff_v1_backup"
_old_zip = PATHS["root"]/"CoDAGAT_preprocessed_handoff.zip"
_bak_zip = PATHS["root"]/"CoDAGAT_preprocessed_handoff_v1_backup.zip"
if _bak_pkg.exists():
    print(f"backup already present, left as is: {_bak_pkg.name}/")
elif _old_pkg.exists():
    shutil.move(str(_old_pkg), str(_bak_pkg))
    print(f"moved handoff/ -> {_bak_pkg.name}/")
else:
    print("!! no previous handoff/ on Drive — 34-VERIFY will have nothing to compare to")
if _old_zip.exists() and not _bak_zip.exists():
    shutil.move(str(_old_zip), str(_bak_zip))
    print(f"moved {_old_zip.name} -> {_bak_zip.name}")

# ---- 1. tier promotions, exactly as Cell 13 applied them ----
PROMOTIONS = {
    "GSE213621": ("T3", "T1", "192 samples recovered (F0F1/F3F4); 97 neg / 202 pos"),
    "GSE193066": ("AUDIT","T1", "106 baseline biopsies; 43 neg / 63 pos. NOTE: PLS high-risk enriched"),
    "GSE240729": ("T2", "T1", "26 neg / 41 pos — binary-evaluable"),
    "GSE207310": ("AUDIT","T2", "only 8 positives"),
    "GSE48452":  ("T1", "T2", "3 positives under F2+ — not binary-evaluable"),
    "GSE89632":  ("T1", "T2", "10 positives under F2+ — below threshold"),
    "GSE159088": ("AUDIT","REJECTED", "no fibrosis staging in metadata (80/80 unknown)"),
}
for gse, (old, new, why) in PROMOTIONS.items():
    if COHORTS[gse]["tier"] != new:
        assert COHORTS[gse]["tier"] == old, \
            f"{gse}: expected {old}, found {COHORTS[gse]['tier']} — run Cell 3 first"
        COHORTS[gse]["tier"] = new
    COHORTS[gse]["retier_reason"] = why
(PATHS["config"]/"cohorts.json").write_text(json.dumps(COHORTS, indent=2))

_sup_disk = pd.read_parquet(_I/"bulk_meta_supervised.parquet")
assert _sup_disk.groupby("cohort").tier.nunique().max() == 1
_tier_bad = {c: (t, tier_of(c)) for c, t in _sup_disk.groupby("cohort").tier.first().items()
             if tier_of(c) != t}
assert not _tier_bad, f"registry tiers differ from the exported data: {_tier_bad}"
print("tiers: registry matches bulk_meta_supervised.parquet")

# ---- 2. matrices and references (cached outputs of Cells 25-27b) ----
BULK = pd.read_parquet(_I/"bulk_expression_log2.parquet")
GTEX = pd.read_parquet(_I/"gtex_liver_log2tpm.parquet")
GTEX_MODE = json.loads((_I/"gtex_mode.json").read_text())["mode"]
GTEX_HAS_INDIVIDUALS = (GTEX_MODE == "full")
DEV_GENES = sorted(set(GTEX.index) & set(BULK.index))
ATT_URLS = [   # Cell 27 literals (recorded as provenance only; nothing is downloaded)
  "https://storage.googleapis.com/adult-gtex/annotations/v8/metadata-files/"
  "GTEx_Analysis_v8_Annotations_SampleAttributesDS.txt",
  "https://storage.googleapis.com/gtex_analysis_v8/annotations/"
  "GTEx_Analysis_v8_Annotations_SampleAttributesDS.txt",
]
TPM_URLS = [
  "https://storage.googleapis.com/adult-gtex/bulk-gex/v8/rna-seq/"
  "GTEx_Analysis_2017-06-05_v8_RNASeQCv1.1.9_gene_tpm.gct.gz",
  "https://storage.googleapis.com/gtex_analysis_v8/rna_seq_data/"
  "GTEx_Analysis_2017-06-05_v8_RNASeQCv1.1.9_gene_tpm.gct.gz",
]
CANARY = ["ALB","APOA1","APOB","SERPINA1","HP","TF","CYP2E1","CPS1","FGA","FGB",
          "COL1A1","COL1A2","COL3A1","TGFB1","ACTA2","TIMP1","SERPINE1"]   # Cell 26
print(f"BULK {BULK.shape}  GTEx {GTEX.shape} mode={GTEX_MODE}  overlap {len(DEV_GENES)}")

# ---- 3. per-cohort matrices: only if interim/expr_cache is complete ----
#      (Cell 14b deletes this cache; restore it from Drive Trash if you can.
#       Without it, only the GSE213621 README note falls back to the old text.)
_expr_cohorts = sorted(pd.read_parquet(_I/"bulk_meta.parquet", columns=["cohort"]).cohort.unique())
_cache = _I/"expr_cache"
_have = {f.stem for f in _cache.glob("*.parquet")} if _cache.exists() else set()
if set(_expr_cohorts) <= _have:
    EXPR = {g: pd.read_parquet(_cache/f"{g}.parquet") for g in _expr_cohorts}
    print(f"EXPR restored from expr_cache: {len(EXPR)} cohorts")
else:
    EXPR = {}
    print(f"EXPR not restored: expr_cache has {len(_have)}/{len(_expr_cohorts)} cohorts")

# ---- 4. label provenance file: was it rewritten by a partial re-run? ----
_lr = pd.read_parquet(_I/"labels_raw.parquet")
_bm_ids = set(pd.read_parquet(_I/"bulk_meta.parquet", columns=["cohort"]).index)
LABELS_RAW_COMPLETE = (_bm_ids <= set(_lr.sample_id)
                       and bool((_lr.resolution == "duplicate_biopsy").any()))
print(f"labels_raw.parquet: {_lr.cohort.nunique()} cohorts, complete = {LABELS_RAW_COMPLETE}")
if not LABELS_RAW_COMPLETE:
    print("  !! It no longer covers every exported sample. Restore the previous version")
    print("     (Drive > CoDAGAT/interim/labels_raw.parquet > Manage versions), then rerun")
    print("     this cell. Without it, Cell 33 exports no source_label for the controls")
    print("     and cites the duplicate-biopsy count from the saved Cell 11 output.")

# ---- 5. objects Cell 32 needs: panel, PATHWAYS (Cell 29), z (Cell 30) ----
PATHWAYS = {   # Cell 29, verbatim
    "ECM_remodeling": ["COL1A1","COL1A2","COL3A1","COL4A1","COL5A1","COL6A1",
                       "FN1","LUM","DCN","VCAN","BGN","SPARC","MMP2","MMP14",
                       "TIMP1","TIMP2","LOX","LOXL2","POSTN","THBS2"],
    "TGFB_signaling": ["TGFB1","TGFB2","TGFBR1","TGFBR2","SMAD2","SMAD3","SMAD4",
                       "SMAD7","SERPINE1","CCN2","THBS1","LTBP1"],
    "HSC_activation": ["ACTA2","TAGLN","PDGFRB","PDGFB","DES","VIM","COL1A1",
                       "TIMP1","MYH11","CALD1","NOTCH3","RGS5"],
    "Inflammation":   ["CD68","CCL2","CCL5","CXCL9","CXCL10","IL6","TNF","IL1B",
                       "CD163","MARCO","ITGAX","TLR4","NFKB1","SPP1","CSF1R"],
    "Apoptosis_stress":["CASP3","CASP8","BAX","BCL2","TP53","FAS","XBP1","DDIT3",
                       "ATF4","HSPA5","GADD45A","CDKN1A"],
    "Hepatocyte_function":["CYP2E1","CYP1A2","CYP2C9","CPS1","ASS1","ARG1",
                       "SERPINA1","TTR","TF","HNF4A","ALDOB","GLUL","APOH",
                       "AHSG","ASGR1","NR1H4"],
}
panel = pd.read_csv(_I/"panel_genes.csv").gene.tolist()
sup = pd.read_parquet(_I/"bulk_meta_supervised.parquet")
BULK_SUP = pd.read_parquet(_I/"bulk_supervised_combat.parquet")

# Cell 30, within-cohort z-score of panel expression (verbatim; no labels used)
panel_expr = BULK_SUP.loc[panel]
z = panel_expr.copy()
for c in sup.cohort.unique():
    cols = sup.index[sup.cohort == c]
    cols = [x for x in cols if x in z.columns]
    blk = z[cols]
    z[cols] = blk.sub(blk.mean(axis=1), axis=0).div(
        blk.std(axis=1).replace(0, np.nan), axis=0)
z = z.fillna(0.0)

# z must reproduce feature 0 (expression_z) of the saved tensor exactly
_order = pd.read_csv(_P/"tensor_sample_order.csv").sample_id.tolist()
assert list(z.columns) == _order, "z column order differs from tensor_sample_order.csv"
_T = np.load(_P/"node_features_AB.npy", mmap_mode="r")
_d = float(np.abs(_T[:, :, 0] - z.values.T.astype(np.float32)).max())
assert _d == 0.0, f"rebuilt expression_z differs from the saved tensor (max |diff| {_d})"
print("z: identical to feature 0 of node_features_AB.npy")
print("\nRESTORE OK. Next: Cell 32, then Cell 33.")

!! no previous handoff/ on Drive — 34-VERIFY will have nothing to compare to
tiers: registry matches bulk_meta_supervised.parquet
BULK (10732, 1514)  GTEx (54592, 226) mode=full  overlap 10731
EXPR restored from expr_cache: 12 cohorts
labels_raw.parquet: 14 cohorts, complete = True
z: identical to feature 0 of node_features_AB.npy

RESTORE OK. Next: Cell 32, then Cell 33.


In [61]:
# ============================================================
# CELL 32 — LODO FOLDS + GROUP C (fold-fitted, 9 features)
# Group C is NEVER precomputed. These functions are called inside
# the fold loop, on the proper-train split only.
# ============================================================
from scipy.stats import ttest_ind
from sklearn.metrics import roc_auc_score

order = pd.read_csv(PATHS["processed"]/"tensor_sample_order.csv").sample_id.tolist()
TENSOR_AB = np.load(PATHS["processed"]/"node_features_AB.npy")
sup = pd.read_parquet(PATHS["interim"]/"bulk_meta_supervised.parquet").loc[order]
panel = pd.read_csv(PATHS["interim"]/"panel_genes.csv").gene.tolist()
assert TENSOR_AB.shape[0] == len(sup) == len(order)
print(f"tensor {TENSOR_AB.shape}, meta {sup.shape}")

TASK = "y_T1_significant"
CAL_MODE = "V1"        # V1 = one whole training cohort as calibration
                       # V2 = stratified random 20% of pooled training
CAL_FRAC = 0.20
rng = np.random.RandomState(RANDOM_SEED)

t1_cohorts = sorted(sup.loc[sup.tier == "T1", "cohort"].unique())
print(f"\n{len(t1_cohorts)} LODO folds: {t1_cohorts}")

folds = []
for k, test_c in enumerate(t1_cohorts):
    test_idx = sup.index[(sup.cohort == test_c) & sup[TASK].notna()].tolist()
    pool = sup.index[(sup.cohort != test_c) & sup[TASK].notna()].tolist()

    if CAL_MODE == "V1":
        # calibration must come from T1 cohorts only: T2 is training-only
        # (frozen spec Rule 1), and the guard below enforces it.
        cands = []
        for c in t1_cohorts:
            if c == test_c:
                continue
            ci = [i for i in pool if sup.loc[i, "cohort"] == c]
            if len(ci) >= 30 and sup.loc[ci, TASK].nunique() > 1:
                cands.append(c)
        assert cands, f"fold {k}: no eligible T1 calibration cohort"
        # deterministic rotation, so no cohort is over-used as calibrator
        cal_c = cands[k % len(cands)]
        cal_idx = [i for i in pool if sup.loc[i, "cohort"] == cal_c]
        tr_idx  = [i for i in pool if sup.loc[i, "cohort"] != cal_c]
        cal_desc = f"cohort:{cal_c}"
    else:
        cal_idx, tr_idx = [], []
        for c in sup.loc[pool, "cohort"].unique():
            if tier_of(c) != "T1":          # T2 goes wholly to proper-train
                tr_idx += [i for i in pool if sup.loc[i, "cohort"] == c]
                continue
            ci = [i for i in pool if sup.loc[i, "cohort"] == c]
            for y in sup.loc[ci, TASK].unique():
                yi = [i for i in ci if sup.loc[i, TASK] == y]
                n = max(1, int(round(CAL_FRAC * len(yi))))
                pick = list(rng.choice(yi, size=min(n, len(yi)), replace=False))
                cal_idx += pick
                tr_idx  += [i for i in yi if i not in pick]
        cal_desc = f"random{int(CAL_FRAC*100)}%_T1only"

    assert_cohort_disjoint(sup.loc[tr_idx], sup.loc[cal_idx], sup.loc[test_idx], k)
    assert_tier_usage(sup.loc[test_idx], "test", k)
    assert_tier_usage(sup.loc[cal_idx], "calibration", k)
    assert_no_synthetic(sup.loc[cal_idx], "calibration", k)
    assert_no_synthetic(sup.loc[test_idx], "test", k)

    folds.append(dict(fold=k, test_cohort=test_c, calibration=cal_desc,
                      n_train=len(tr_idx), n_cal=len(cal_idx), n_test=len(test_idx),
                      test_pos=float(sup.loc[test_idx, TASK].mean()),
                      train_pos=float(sup.loc[tr_idx, TASK].mean()),
                      cal_pos=float(sup.loc[cal_idx, TASK].mean()),
                      train_idx=tr_idx, cal_idx=cal_idx, test_idx=test_idx))

fd = pd.DataFrame([{k: v for k, v in f.items() if not k.endswith("_idx")}
                   for f in folds])
print("\nFOLD TABLE")
print(fd.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

def js_div(p, q):
    p, q = np.asarray(p, float), np.asarray(q, float); m = (p+q)/2
    kl = lambda a, b: np.sum([x*np.log2(x/y) for x, y in zip(a, b) if x > 0])
    return 0.5*kl(p, m) + 0.5*kl(q, m)
fd["JS"] = [js_div([1-r.test_pos, r.test_pos], [1-r.train_pos, r.train_pos])
            for r in fd.itertuples()]
print("\nLABEL SHIFT (independent variable of the dose-response experiment)")
print(fd[["fold","test_cohort","n_test","test_pos","train_pos","JS"]]
      .sort_values("JS", ascending=False)
      .to_string(index=False, float_format=lambda x: f"{x:.4f}"))
fd.to_csv(PATHS["processed"]/"fold_assignment_lodo.csv", index=False)

rows = []
for f in folds:
    for split, ids in [("proper_train", f["train_idx"]), ("calibration", f["cal_idx"]),
                       ("test", f["test_idx"])]:
        rows += [dict(fold=f["fold"], sample_id=i, split=split,
                      cohort=sup.loc[i, "cohort"]) for i in ids]
pd.DataFrame(rows).to_csv(PATHS["processed"]/"fold_sample_splits.csv", index=False)

# ---------------- Group C ----------------
FEATURES_C = ["disease_specificity","pathway_coherence","cross_etiology_consistency",
              "fibrosis_progression","cohens_d","t_statistic","log_fold_change",
              "per_gene_auc","rank_score_composite"]

def compute_group_C(train_ids, expr_z, meta_df, panel, pathways):
    """Compute Group C using this fold's proper-train samples."""
    X = expr_z.loc[panel, train_ids].to_numpy(dtype=float)
    y = meta_df.loc[train_ids, TASK].to_numpy(dtype=float)

    if not np.isfinite(X).all():
        raise ValueError("Group C input contains NaN or infinite values")

    if not np.isfinite(y).all() or not np.isin(y, [0, 1]).all():
        raise ValueError("Group C requires finite binary training labels")

    pos = X[:, y == 1]
    neg = X[:, y == 0]
    npos, nneg = pos.shape[1], neg.shape[1]

    if npos < 2 or nneg < 2:
        raise ValueError(
            "Group C requires at least two proper-train samples per class"
        )

    mp = pos.mean(axis=1)
    mn = neg.mean(axis=1)
    sp = pos.std(axis=1, ddof=1)
    sn = neg.std(axis=1, ddof=1)

    pooled = np.sqrt(
        ((npos - 1) * sp**2 + (nneg - 1) * sn**2)
        / (npos + nneg - 2)
    )

    invalid = ~np.isfinite(pooled) | (pooled <= 0)
    if invalid.any():
        bad = np.asarray(panel)[invalid]
        raise ValueError(
            f"Cohen's d is undefined for these genes: {bad.tolist()}"
        )

    cohens_d = (mp - mn) / pooled
    tstat = ttest_ind(pos, neg, axis=1, equal_var=False).statistic
    lfc = mp - mn
    auc = np.array([roc_auc_score(y, X[i]) if len(np.unique(y)) > 1 else 0.5
                    for i in range(X.shape[0])])

    gi = {g: i for i, g in enumerate(panel)}
    coh = np.zeros(len(panel))
    for p, gs in pathways.items():
        mem = [gi[g] for g in gs if g in gi]
        if len(mem) > 1:
            C = np.corrcoef(X[mem])
            np.fill_diagonal(C, np.nan)
            for j, m in enumerate(mem):
                coh[m] = max(coh[m], np.nanmean(C[j]))

    et = meta_df.loc[train_ids, "etiology"].values
    xe = np.zeros(len(panel))
    groups = [e for e in np.unique(et) if (et == e).sum() >= 20]
    if len(groups) > 1:
        signs = []
        for e in groups:
            m = et == e
            if len(np.unique(y[m])) > 1:
                signs.append(np.sign(X[:, m & (y == 1)].mean(1) -
                                     X[:, m & (y == 0)].mean(1)))
        if signs:
            xe = np.abs(np.mean(signs, axis=0))

    stg = meta_df.loc[train_ids, "y_T2_ordinal"].values.astype(float)
    ok = np.isfinite(stg)
    prog = (np.array([spearmanr(stg[ok], X[i, ok]).statistic
                      for i in range(X.shape[0])]) if ok.sum() >= 30
            else np.zeros(len(panel)))

    spec = np.abs(cohens_d) / (np.abs(cohens_d).max() + 1e-9)
    comp = np.mean([np.argsort(np.argsort(np.abs(v)))/len(v)
                    for v in (cohens_d, tstat, lfc, auc - 0.5)], axis=0)

    C = np.column_stack([spec, coh, xe, np.nan_to_num(prog), cohens_d,
                         np.nan_to_num(tstat), lfc, auc, comp]).astype(np.float32)
    return np.nan_to_num(C, nan=0.0, posinf=0.0, neginf=0.0)

# demo on fold 0
f0 = folds[0]
C0 = compute_group_C(f0["train_idx"], z, sup, panel, PATHWAYS)
print(f"\nGroup C demo (fold 0, held out {f0['test_cohort']}): {C0.shape}")
for i, n in enumerate(FEATURES_C):
    print(f"  {i:2d} {n:26s} mean={C0[:,i].mean():8.4f} sd={C0[:,i].std():7.4f} "
          f"min={C0[:,i].min():8.3f} max={C0[:,i].max():8.3f}")
print("\ntop 8 genes by |Cohen's d| on fold 0:")
for i in np.argsort(-np.abs(C0[:, 4]))[:8]:
    print(f"  {panel[i]:10s} d={C0[i,4]:+.3f}  auc={C0[i,7]:.3f}")

def build_fold_tensor(fold, tensor_AB, expr_z, meta_df, panel, pathways):
    """(n_split, 72, 26) for each split. Group C fitted on proper-train only."""
    C = compute_group_C(fold["train_idx"], expr_z, meta_df, panel, pathways)
    pos = {s: i for i, s in enumerate(order)}
    out = {}
    for split, ids in [("proper_train", fold["train_idx"]),
                       ("calibration", fold["cal_idx"]),
                       ("test", fold["test_idx"])]:
        rows_ = [pos[i] for i in ids]
        ab = tensor_AB[rows_]
        cb = np.broadcast_to(C[None, :, :], (len(rows_), C.shape[0], C.shape[1]))
        out[split] = np.concatenate([ab, cb], axis=2).astype(np.float32)
    return out

demo = build_fold_tensor(f0, TENSOR_AB, z, sup, panel, PATHWAYS)
for k_, v_ in demo.items():
    print(f"  {k_:14s} {v_.shape}")
assert_tensor_integrity(demo["proper_train"], "fold0_train")

pd.DataFrame({"idx": range(17, 26), "feature": FEATURES_C, "group": "C",
              "leakage_class": "fold_fitted"}) \
  .to_csv(PATHS["processed"]/"feature_schema_C.csv", index=False)

write_manifest("p09_folds",
               inputs={"tensor_AB": "node_features_AB.npy"},
               params={"task": TASK, "cal_mode": CAL_MODE,
                       "n_folds": len(folds), "n_features_total": 26},
               outputs=[PATHS["processed"]/"fold_assignment_lodo.csv",
                        PATHS["processed"]/"fold_sample_splits.csv",
                        PATHS["processed"]/"feature_schema_C.csv"])
fd["JS_test_vs_train"] = [js_div([1-r.test_pos, r.test_pos],
                                 [1-r.train_pos, r.train_pos]) for r in fd.itertuples()]
fd["JS_test_vs_cal"]   = [js_div([1-r.test_pos, r.test_pos],
                                 [1-r.cal_pos,   r.cal_pos])   for r in fd.itertuples()]
print("\nLABEL SHIFT")
print(fd[["fold","test_cohort","n_test","n_cal","test_pos","cal_pos","train_pos",
          "JS_test_vs_train","JS_test_vs_cal"]]
      .sort_values("JS_test_vs_cal", ascending=False)
      .to_string(index=False, float_format=lambda x: f"{x:.4f}"))

tensor (1366, 72, 17), meta (1366, 12)

8 LODO folds: ['GSE130970', 'GSE135251', 'GSE162694', 'GSE193066', 'GSE213621', 'GSE240729', 'GSE31803', 'GSE84044']

FOLD TABLE
 fold test_cohort      calibration  n_train  n_cal  n_test  test_pos  train_pos  cal_pos
    0   GSE130970 cohort:GSE135251     1043    206      78     0.321      0.580    0.587
    1   GSE135251 cohort:GSE162694     1009    112     206     0.587      0.578    0.420
    2   GSE162694 cohort:GSE193066     1109    106     112     0.420      0.578    0.594
    3   GSE193066 cohort:GSE213621      922    299     106     0.594      0.527    0.676
    4   GSE213621 cohort:GSE240729      961     67     299     0.676      0.529    0.612
    5   GSE240729  cohort:GSE31803     1188     72      67     0.612      0.571    0.444
    6    GSE31803  cohort:GSE84044     1131    124      72     0.444      0.582    0.492
    7    GSE84044 cohort:GSE130970     1125     78     124     0.492      0.591    0.321

LABEL SHIFT (independent vari

In [66]:
# ============================================================
# CELL 33 — FINAL VERIFICATION + EXPORT   [v2]
# Every claim that will appear in the paper, checked here.
#
# v2 changes the EXPORT and its DESCRIPTION only. No expression
# value, feature, label or fold assignment is recomputed.
#   - labels.parquet: + exact_stage (null unless resolution=="exact")
#     + eligible_T1 / eligible_T2 / eligible_T3 masks. Targets are
#     never filled: a missing target stays missing.
#   - preservation: missing values exported as an explicit "unknown"
#     (never inferred from another field).
#   - healthy_reference.parquet -> geo_control_reference.parquet,
#     with per-sample metadata and a description. It is NOT GTEx.
#   - GTEx reference provenance (input files + SHA-256, samples,
#     gene overlap) exported so the GTEx features can be reproduced.
#   - Table I: F0-F4 count exact stages only; grouped ranges are
#     separate columns; F0/F1 counts only y_T1 == 0.
#   - handoff.json numbers computed from the outputs.
# ============================================================

PRESERVATION_UNKNOWN = "unknown"
TARGET_COLS       = ["y_T1_significant", "y_T2_ordinal", "y_T3_early_adv"]
EXPORT_LABEL_COLS = ["exact_stage", "eligible_T1", "eligible_T2", "eligible_T3"]

# Verified against the exported data. A mismatch means upstream data
# changed: find out why before editing a number here.
EXPECTED = dict(n_records=1366, eligible_T1=1327, eligible_T2=1063,
                preservation_unknown=136, geo_controls=148,
                gtex_samples=226, gtex_overlap_genes=10731)


def add_export_label_columns(df):
    """Export-only columns. Never modifies stage, resolution or a y_* target.
    Idempotent: safe to call again on an already-enriched frame."""
    out = df.copy()
    out["exact_stage"] = out["stage"].where(out["resolution"].eq(RES_EXACT))
    out["eligible_T1"] = out["y_T1_significant"].notna()
    out["eligible_T2"] = out["y_T2_ordinal"].notna()
    out["eligible_T3"] = out["y_T3_early_adv"].notna()
    out["preservation"] = out["preservation"].fillna(PRESERVATION_UNKNOWN)
    return out


def check_export_labels(df, name):
    is_exact = df["resolution"].eq(RES_EXACT)
    assert df.loc[is_exact, "exact_stage"].notna().all(), \
        f"{name}: a resolution=='exact' record has no stage"
    assert df.loc[~is_exact, "exact_stage"].isna().all(), \
        f"{name}: exact_stage is set on a non-exact record"
    ex_ = df[is_exact]
    assert (ex_["y_T2_ordinal"] == ex_["exact_stage"].str[1].astype(int)).all(), \
        f"{name}: y_T2_ordinal disagrees with exact_stage"
    for t, m in [("y_T1_significant", "eligible_T1"), ("y_T2_ordinal", "eligible_T2"),
                 ("y_T3_early_adv", "eligible_T3")]:
        assert df[m].dtype == bool, f"{name}: {m} is not boolean"
        assert df[m].equals(df[t].notna()), f"{name}: {m} != {t}.notna()"
    assert df["preservation"].notna().all(), f"{name}: preservation still has nulls"
    return True


def _json_default(o):
    if isinstance(o, np.integer):  return int(o)
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.bool_):    return bool(o)
    if isinstance(o, (np.ndarray, pd.Index)): return list(o)
    if isinstance(o, Path):        return str(o)
    raise TypeError(f"not JSON serialisable: {type(o)}")


def _counts(s):
    return {str(k): int(v) for k, v in s.value_counts().sort_index().items()}


_HASH_CACHE_F = PATHS["reports"] / "input_sha256_cache.json"

def sha256_cached(path):
    """sha256 of a (possibly multi-GB) file, cached by path + size + mtime."""
    path = Path(path)
    if not path.exists():
        return None
    st = path.stat()
    key = f"{path}|{st.st_size}|{int(st.st_mtime)}"
    cache = json.loads(_HASH_CACHE_F.read_text()) if _HASH_CACHE_F.exists() else {}
    if key not in cache:
        print(f"  hashing {path.name} ({st.st_size/1e6:,.0f} MB) ...")
        cache[key] = sha256_file(path)
        _HASH_CACHE_F.write_text(json.dumps(cache, indent=2))
    return cache[key]


print("="*72)
print("DATASET SUMMARY")
print("="*72)

# ---- label enrichment (export-only) ----
_src_labels = pd.read_parquet(PATHS["interim"]/"bulk_meta_supervised.parquet")
_pres_missing = _src_labels[_src_labels["preservation"].isna()]
LABELS_EXPORT = add_export_label_columns(_src_labels)
check_export_labels(LABELS_EXPORT, "labels.parquet")
for t in TARGET_COLS:
    assert LABELS_EXPORT[t].isna().sum() == _src_labels[t].isna().sum(), \
        f"{t}: missing values changed during export"

sup = add_export_label_columns(sup)          # same columns for the fold y.csv files
check_export_labels(sup, "sup")
assert set(sup.index) == set(LABELS_EXPORT.index), "sup and labels.parquet differ"
_cmp = LABELS_EXPORT.columns.tolist()
pd.testing.assert_frame_equal(LABELS_EXPORT.sort_index(), sup[_cmp].sort_index())

N_EL = {t: int(LABELS_EXPORT[f"eligible_{t}"].sum()) for t in ("T1", "T2", "T3")}
assert len(LABELS_EXPORT) == EXPECTED["n_records"], len(LABELS_EXPORT)
assert N_EL["T1"] == EXPECTED["eligible_T1"], N_EL
assert N_EL["T2"] == EXPECTED["eligible_T2"], N_EL
assert N_EL["T3"] == len(LABELS_EXPORT), N_EL      # every supervised record has y_T3
assert len(_pres_missing) == EXPECTED["preservation_unknown"], len(_pres_missing)

_inel = (LABELS_EXPORT[~LABELS_EXPORT.eligible_T1]
         .groupby(["tier", "cohort", "resolution"]).size())
T1_INELIGIBLE = [dict(tier=t, cohort=c, resolution=r, n=int(n))
                 for (t, c, r), n in _inel.items()]
PRES_UNKNOWN_BY_COHORT = _counts(_pres_missing["cohort"])

loaded = sorted(pd.read_parquet(PATHS["interim"]/"bulk_meta.parquet",
                                columns=["cohort"]).cohort.unique())
_lr_all = pd.read_parquet(PATHS["interim"]/"labels_raw.parquet")
t1c = sorted(sup.loc[sup.tier=="T1","cohort"].unique())
t2c = sorted(sup.loc[sup.tier=="T2","cohort"].unique())

_edges = pd.read_csv(PATHS["interim"]/"graph_edges.csv")
assert len(_edges) % 2 == 0, "graph_edges.csv should hold both directions of each edge"
N_EDGES_UND = len(_edges) // 2
_schema = pd.concat([pd.read_csv(PATHS["processed"]/"feature_schema_AB.csv"),
                     pd.read_csv(PATHS["processed"]/"feature_schema_C.csv")])
N_FEAT = {g: int((_schema.group == g).sum()) for g in ("A", "B", "C")}
N_FEAT["total"] = int(len(_schema))

print(f"  cohorts with expression : {len(loaded)}")
print(f"  T1 (LODO folds)         : {len(t1c)}  {t1c}")
print(f"  T2 (training-only)      : {len(t2c)}  {t2c}")
print(f"  T4 (external)           : ['GSE163211']  NanoString; processed separately,"
      f" not in this handoff")
print(f"  supervised records      : {len(LABELS_EXPORT)}")
print(f"  eligible T1 / T2 / T3   : {N_EL['T1']} / {N_EL['T2']} / {N_EL['T3']}")
print(f"  T1-ineligible records   : {T1_INELIGIBLE}")
print(f"  preservation unknown    : {len(_pres_missing)}  {PRES_UNKNOWN_BY_COHORT}")
print(f"  genes (common space)    : {BULK.shape[0]}")
print(f"  panel genes             : {len(panel)}")
print(f"  STRING edges            : {N_EDGES_UND} undirected / {len(_edges)} directed,"
      f" score>={STRING_THRESHOLD}")
print(f"  node features           : {N_FEAT['A']+N_FEAT['B']} precomputed + "
      f"{N_FEAT['C']} fold-fitted = {N_FEAT['total']}")
print(f"  platforms               : RNA-seq, microarray (NanoString external, separate)")
print(f"  etiologies              : {sorted(sup.etiology.dropna().unique())}")

print("\nTABLE I (for the paper)  F0-F4 = EXACT biopsy stages only")
GROUPED_RES = [RES_G_F0F1, RES_G_F0F2, RES_G_F3F4]
tab = []
for c in t1c + t2c:
    s = sup[sup.cohort == c]
    row = dict(cohort=c, tier=tier_of(c),
               platform=COHORTS[c].get("platform"),
               etiology=COHORTS[c].get("etiology"),
               preservation=COHORTS[c].get("preservation") or PRESERVATION_UNKNOWN,
               n=len(s))
    for st in ["F0","F1","F2","F3","F4"]:
        row[st] = int((s.exact_stage == st).sum())
    for r_ in GROUPED_RES:
        row[r_] = int((s.resolution == r_).sum())
    row["exact_ishak_mapped"] = int((s.resolution == "exact_ishak_mapped").sum())
    row["grouped"] = int((~s.resolution.isin([RES_EXACT,"exact_ishak_mapped"])).sum())
    row["F2plus"] = int((s.y_T1_significant == 1).sum())
    row["F0F1"] = int((s.y_T1_significant == 0).sum())
    row["eligible_T1"] = int(s.eligible_T1.sum())
    row["eligible_T2"] = int(s.eligible_T2.sum())
    assert row["n"] == (sum(row[f"F{i}"] for i in range(5)) + row["exact_ishak_mapped"]
                        + sum(row[r_] for r_ in GROUPED_RES)), f"{c}: Table I does not add up"
    assert row["F2plus"] + row["F0F1"] == row["eligible_T1"], f"{c}: F2+ / F0-F1 mismatch"
    tab.append(row)
tab = pd.DataFrame(tab)
print(tab.to_string(index=False))
print(f"\n  TOTAL n={tab.n.sum()}  eligible_T1={tab.eligible_T1.sum()}  "
      f"F2+={tab.F2plus.sum()}  F0/F1={tab.F0F1.sum()}  "
      f"T1-ineligible={tab.n.sum()-tab.eligible_T1.sum()}  "
      f"grouped-label={tab.grouped.sum()}")
tab.to_csv(PATHS["reports"]/"table1_cohorts.csv", index=False)

print("\n" + "="*72)
print("LEAKAGE VERIFICATION")
print("="*72)
splits = pd.read_csv(PATHS["processed"]/"fold_sample_splits.csv")
checks = []
for k in sorted(splits.fold.unique()):
    f = splits[splits.fold == k]
    tr = set(f[f.split=="proper_train"].sample_id)
    ca = set(f[f.split=="calibration"].sample_id)
    te = set(f[f.split=="test"].sample_id)
    trc = set(f[f.split=="proper_train"].cohort)
    cac = set(f[f.split=="calibration"].cohort)
    tec = set(f[f.split=="test"].cohort)
    checks.append(dict(fold=k,
        sample_overlap=len(tr&ca)+len(tr&te)+len(ca&te),
        test_is_one_cohort=len(tec)==1,
        test_cohort_leaks=len(tec&trc)+len(tec&cac),
        cal_cohort_leaks=len(cac&trc),
        cal_tier_ok=all(TIER_RULES[tier_of(c)][1] for c in cac),
        test_tier_ok=all(TIER_RULES[tier_of(c)][2] for c in tec),
        all_eligible_T1=bool(sup.loc[list(tr|ca|te), "eligible_T1"].all())))
ck = pd.DataFrame(checks)
print(ck.to_string(index=False))
allok = (ck.sample_overlap.sum()==0 and ck.test_is_one_cohort.all()
         and ck.test_cohort_leaks.sum()==0 and ck.cal_cohort_leaks.sum()==0
         and ck.cal_tier_ok.all() and ck.test_tier_ok.all()
         and ck.all_eligible_T1.all())
print(f"\n  ALL LEAKAGE CHECKS PASS: {allok}")
assert allok, "leakage check failed — do not proceed to modelling"
_in_folds = set(splits.sample_id)
assert _in_folds == set(sup.index[sup.eligible_T1]), \
    "fold splits do not cover exactly the eligible_T1 records"

print("\nGROUP C IS FOLD-FITTED (verify it differs across folds)")
c_by_fold = {}
for f in folds[:4]:
    C = compute_group_C(f["train_idx"], z, sup, panel, PATHWAYS)
    c_by_fold[f["test_cohort"]] = C
ks = list(c_by_fold)
for i in range(len(ks)-1):
    d_ = np.abs(c_by_fold[ks[i]] - c_by_fold[ks[i+1]]).mean()
    print(f"  mean |C({ks[i]}) - C({ks[i+1]})| = {d_:.4f}")
print("  Non-zero differences confirm Group C is refit per fold.")

print("\nDEAD / NEAR-CONSTANT FEATURES")
demo_all = build_fold_tensor(folds[0], TENSOR_AB, z, sup, panel, PATHWAYS)["proper_train"]
for i, nm in enumerate(_schema.feature):
    v = demo_all[:,:,i]
    if v.std() < 0.01:
        print(f"  {i:2d} {nm:28s} sd={v.std():.5f}  <<< near-constant")
print("  (near-constant features add a dead dimension; report or drop)")

print("\n" + "="*72)
print("REFERENCE POPULATIONS")
print("="*72)
EXPORT = PATHS["processed"]

# ---- (1) in-study GEO controls (formerly "healthy_reference") ----
_geo_src = PATHS["interim"]/"bulk_healthy_reference.parquet"
_geo = pd.read_parquet(_geo_src)
_bm = pd.read_parquet(PATHS["interim"]/"bulk_meta.parquet")
assert set(_geo.columns) <= set(_bm.index), "control samples missing from bulk_meta"
geo_meta = _bm.loc[list(_geo.columns),
                   ["cohort","resolution","tier","etiology","platform","preservation"]].copy()
assert geo_meta.resolution.eq(RES_HEALTHY).all(), "non-control sample in the control file"
assert not (set(geo_meta.index) & set(LABELS_EXPORT.index)), "control sample is also supervised"
# labels_raw.parquet is only trusted if it still covers every exported sample
LR_COMPLETE = (set(_bm.index) <= set(_lr_all.sample_id)
               and bool((_lr_all.resolution == "duplicate_biopsy").any()))
if LR_COMPLETE:
    _lr = _lr_all[_lr_all.sample_id.isin(geo_meta.index)]
    assert _lr.sample_id.is_unique, "duplicate control sample_id in labels_raw"
    geo_meta["source_label"] = _lr.set_index("sample_id").raw_string.reindex(geo_meta.index)
else:
    print("!! interim/labels_raw.parquet does not cover every exported sample "
          "(partial re-run?) — source_label not exported")
geo_meta["preservation"] = geo_meta["preservation"].fillna(PRESERVATION_UNKNOWN)
geo_meta.index.name = "sample_id"
assert len(geo_meta) == EXPECTED["geo_controls"], len(geo_meta)
assert _geo.index.equals(pd.read_parquet(PATHS["interim"]/"bulk_supervised_combat.parquet",
                                         columns=[]).index), \
    "control matrix gene index differs from bulk_expression"

GEO_DESC = dict(
    file="geo_control_reference.parquet",
    previous_name="healthy_reference.parquet",
    description=("In-study GEO control / normal-histology liver biopsies from the same "
                 "series as the supervised samples. NOT healthy donor liver and NOT the "
                 "GTEx reference (see gtex_reference_provenance.json)."),
    n_samples=int(_geo.shape[1]), n_genes=int(_geo.shape[0]),
    by_cohort=_counts(geo_meta.cohort),
       source_labels=(_counts(geo_meta.source_label.fillna("NA")) if LR_COMPLETE
                   else "not available: labels_raw.parquet incomplete"),
    matrix=("genes x samples; same genes and same integrated log2 scale as "
            "bulk_expression.parquet (columns of the same ComBat-integrated matrix)"),
    label_status=("resolution == 'healthy': no fibrosis stage. Excluded from N, from "
                  "labels.parquet and from every fold split."),
    per_sample_metadata="geo_control_reference_samples.csv",
    used_by_this_pipeline=[
        "ComBat fit (unlabelled, batch = cohort), together with all other loaded samples",
        "the platform/cohort standardisation statistics of the platform_std, cohort_std, "
        "directional_ps and directional_cs columns of deviation_targets.parquet, which "
        "also contains rows for these samples",
        "the Cell 28 deviation gate diagnostic (superseded by the 3-arm ablation; see "
        "head3.json rationale)"],
    not_used_for=("a reference or normaliser of any node feature in X.npy; they affect "
                  "X.npy only through the shared ComBat fit"),
)

# ---- (2) GTEx v8 liver (the reference behind the GTEx features) ----
_gtex_f = PATHS["interim"]/"gtex_liver_log2tpm.parquet"
_gtex_mode = json.loads((PATHS["interim"]/"gtex_mode.json").read_text())
GTEX_REF = pd.read_parquet(_gtex_f)
_bulk_genes = pd.read_parquet(PATHS["interim"]/"bulk_expression_log2.parquet",
                              columns=[]).index
GTEX_OVERLAP = sorted(set(GTEX_REF.index) & set(_bulk_genes))
if "DEV_GENES" in globals():
    assert list(DEV_GENES) == GTEX_OVERLAP, "overlap differs from the one used in Cell 27b"
assert len(_bulk_genes) == 10732, (
    f"Current run expected 10732 bulk genes; found {len(_bulk_genes)}"
)

assert len(GTEX_OVERLAP) == EXPECTED["gtex_overlap_genes"], (
    f"Expected 10731 GTEx-overlap genes; found {len(GTEX_OVERLAP)}"
)

assert set(GTEX_OVERLAP) == (
    set(_bulk_genes) & set(GTEX_REF.index)
), "GTEx overlap does not match the current expression matrix"

print(
    "Current regenerated dataset: 10732 bulk genes; "
    "10731 overlap with GTEx."
)
assert _gtex_mode["mode"] == "full", f"GTEx mode is {_gtex_mode['mode']!r}, not 'full'"
_gtex_samples = [str(c) for c in GTEX_REF.columns]
_gtex_donor = {s: "-".join(s.split("-")[:2]) for s in _gtex_samples}
assert len(_gtex_samples) == EXPECTED["gtex_samples"], len(_gtex_samples)

_att_f = PATHS["raw_gtex"]/"gtex_v8_sample_attributes.txt"
_tpm_f = PATHS["raw_gtex"]/"gtex_v8_gene_tpm.gct.gz"
_att_check = None
if _att_f.exists():
    _att = pd.read_csv(_att_f, sep="\t", low_memory=False, usecols=["SAMPID", "SMTSD"])
    _liver_ids = set(_att.loc[_att.SMTSD == "Liver", "SAMPID"].astype(str))
    _att_check = dict(liver_ids_in_annotation=len(_liver_ids),
                      all_reference_samples_annotated_liver=set(_gtex_samples) <= _liver_ids)
    assert _att_check["all_reference_samples_annotated_liver"], \
        "a GTEx reference column is not annotated SMTSD == 'Liver'"

_fi = lambda p, urls: dict(local_path=str(p), exists=p.exists(),
                           bytes=(p.stat().st_size if p.exists() else None),
                           sha256=sha256_cached(p),
                           canonical_name=(Path(urls[0]).name if urls else None),
                           url_candidates=urls)
GTEX_PROV = dict(
    description=("GTEx v8 adult liver bulk RNA-seq. The ONLY reference for the GTEx "
                 "features. Not shipped in this package; reproducible from the "
                 "public files below."),
    release="GTEx Analysis v8 (RNASeQC v1.1.9 gene TPM)",
    mode=_gtex_mode["mode"],
    n_samples=len(_gtex_samples),
    n_donors=len(set(_gtex_donor.values())),
    inputs=dict(
        gene_tpm=_fi(_tpm_f, list(globals().get("TPM_URLS", []))),
        sample_attributes=_fi(_att_f, list(globals().get("ATT_URLS", []))),
        note=("The URL actually used was not logged by Cell 27; sha256 identifies the "
              "exact file. A null sha256 means the raw file is no longer on Drive."),
    ),
    annotation_check=_att_check,
    processing=[
        "liver samples = SAMPID with SMTSD == 'Liver' that are columns of the gene-TPM GCT",
        "gene symbol = GCT 'Description'; duplicate symbols collapsed by per-sample median TPM",
        "value = log2(max(TPM, 0) + 1)",
    ],
    derived_file=dict(path=str(_gtex_f), sha256=sha256_cached(_gtex_f),
                      genes=int(GTEX_REF.shape[0]), samples=int(GTEX_REF.shape[1])),
    overlap_with_bulk_expression=dict(
        n_genes=len(GTEX_OVERLAP), of_bulk_genes=int(len(_bulk_genes)),
        bulk_genes_absent_from_gtex=sorted(set(_bulk_genes) - set(GTEX_REF.index)),
        gene_list="gtex_overlap_genes.csv"),
    used_for={
        "deviation_rank_z (node feature 1)":
            ("per sample: percentile rank of each overlap gene minus the percentile rank "
             "of its GTEx across-sample median, z-scored within the sample across the "
             "overlap genes; panel genes kept"),
        "gtex_expression_level (node feature 13)":
            "mean log2TPM across the GTEx samples, min-max scaled across the panel genes",
        "gtex_expression_std (node feature 14)":
            "SD across the GTEx samples, min-max scaled across the panel genes",
        "deviation_targets.parquet":
            "every column is built from the same rank displacement over the overlap genes",
    },
    sample_list="gtex_reference_samples.csv",
)

print(f"  GEO controls : {GEO_DESC['n_samples']} samples, {GEO_DESC['by_cohort']}")
print(f"  GTEx v8 liver: {GTEX_PROV['n_samples']} samples / {GTEX_PROV['n_donors']} donors, "
      f"overlap {len(GTEX_OVERLAP)}/{len(_bulk_genes)} genes "
      f"(absent from GTEx: {GTEX_PROV['overlap_with_bulk_expression']['bulk_genes_absent_from_gtex']})")

print("\n" + "="*72)
print("EXPORT")
print("="*72)
for src, dst in [(PATHS["interim"]/"bulk_supervised_combat.parquet",
                  EXPORT/"bulk_expression.parquet"),
                 (PATHS["interim"]/"bulk_supervised_nocombat.parquet",
                  EXPORT/"bulk_expression_nocombat.parquet"),
                 (PATHS["interim"]/"deviation_sample.parquet",
                  EXPORT/"deviation_targets.parquet"),
                 (PATHS["interim"]/"panel_genes.csv", EXPORT/"panel_genes.csv"),
                 (PATHS["interim"]/"graph_edges.csv", EXPORT/"graph_edges.csv"),
                 (PATHS["interim"]/"graph_topology.csv", EXPORT/"graph_topology.csv"),
                 (_geo_src, EXPORT/"geo_control_reference.parquet")]:
    if not Path(src).is_file():
        raise FileNotFoundError(
            f"Required export source is missing: {src}"
        )

    shutil.copy(src, dst)

if LABELS_EXPORT.index.hasnans or not LABELS_EXPORT.index.is_unique:
    raise ValueError("Export labels contain missing or duplicate sample IDs")

if not LABELS_EXPORT.index.equals(
    pd.Index(BULK_SUP.columns)
):
    raise ValueError(
        "Label sample IDs/order do not match the supervised expression matrix"
    )

LABELS_EXPORT.index.name = "sample_id"
LABELS_EXPORT.to_parquet(EXPORT / "labels.parquet", index=True)
geo_meta.to_csv(EXPORT/"geo_control_reference_samples.csv")
(EXPORT/"geo_control_reference.json").write_text(json.dumps(GEO_DESC, indent=2))
(EXPORT/"gtex_reference_provenance.json").write_text(
    json.dumps(GTEX_PROV, indent=2, default=_json_default))
pd.Series(GTEX_OVERLAP, name="gene").to_csv(EXPORT/"gtex_overlap_genes.csv", index=False)
pd.DataFrame({"sample_id": _gtex_samples,
              "donor_id": [_gtex_donor[s] for s in _gtex_samples]}) \
  .to_csv(EXPORT/"gtex_reference_samples.csv", index=False)

# the old ambiguous name: remove it only if it is a byte-identical copy
_old = EXPORT/"healthy_reference.parquet"
if _old.exists():
    if sha256_file(_old) == sha256_file(EXPORT/"geo_control_reference.parquet"):
        _old.unlink()
        print("removed processed/healthy_reference.parquet "
              "(identical copy; now geo_control_reference.parquet)")
    else:
        print("!! processed/healthy_reference.parquet differs from the new file — "
              "left in place; inspect it")

REQUIRED = ["node_features_AB.npy","tensor_sample_order.csv","feature_schema_AB.csv",
            "feature_schema_C.csv","fold_assignment_lodo.csv","fold_sample_splits.csv",
            "bulk_expression.parquet","bulk_expression_nocombat.parquet",
            "labels.parquet","deviation_targets.parquet","panel_genes.csv",
            "graph_edges.csv","graph_topology.csv",
            "geo_control_reference.parquet","geo_control_reference_samples.csv",
            "geo_control_reference.json","gtex_reference_provenance.json",
            "gtex_overlap_genes.csv","gtex_reference_samples.csv"]
print(f"{'file':38s} {'MB':>8s}  status")
missing = []
for f in REQUIRED:
    p = EXPORT/f
    if p.exists():
        print(f"{f:38s} {p.stat().st_size/1e6:8.2f}  OK")
    else:
        print(f"{f:38s} {'—':>8s}  MISSING"); missing.append(f)
assert not missing, f"missing exports: {missing}"

_rb = pd.read_parquet(EXPORT / "labels.parquet")
check_export_labels(_rb, "labels.parquet (re-read)")

assert _rb.index.name == "sample_id", (
    f"Unexpected index name: {_rb.index.name!r}"
)

assert len(_rb) == len(LABELS_EXPORT), (
    f"Row count changed: {len(LABELS_EXPORT)} -> {len(_rb)}"
)

pd.testing.assert_frame_equal(_rb, LABELS_EXPORT)

print("Label export verified: sample IDs, order, values and row count preserved.")

# ---- facts for the README (computed, not typed) ----
N_DUP_BIOPSY = (int((_lr_all.resolution == "duplicate_biopsy").sum()) if LR_COMPLETE
                else None)
G213 = None
if isinstance(globals().get("EXPR"), dict) and "GSE213621" in EXPR:
    _canary = globals().get("CANARY", ["ALB","APOA1","APOB","FGA","FGB"])
    _d = EXPR["GSE213621"]
    _ref = next(g for g in EXPR if g != "GSE213621" and "ALB" in EXPR[g].index)
    _top = set(EXPR[_ref].mean(axis=1).sort_values(ascending=False).index[:500])
    G213 = dict(missing_canary=[m for m in _canary if m not in _d.index],
                top500_reference=_ref, top500_retained=len(_top & set(_d.index)))

handoff = dict(
    generated=time.strftime("%Y-%m-%dT%H:%M:%S"),
    n_supervised=int(len(LABELS_EXPORT)),
    n_eligible=N_EL,
    t1_ineligible=T1_INELIGIBLE,
    n_by_tier=_counts(LABELS_EXPORT.tier),
    n_by_resolution=_counts(LABELS_EXPORT.resolution),
    n_exact_stage=int(LABELS_EXPORT.exact_stage.notna().sum()),
    preservation=_counts(LABELS_EXPORT.preservation),
    preservation_unknown_by_cohort=PRES_UNKNOWN_BY_COHORT,
    n_geo_control_reference=GEO_DESC["n_samples"],
    geo_control_reference_by_cohort=GEO_DESC["by_cohort"],
    gtex_reference=dict(n_samples=GTEX_PROV["n_samples"], n_donors=GTEX_PROV["n_donors"],
                        overlap_genes=len(GTEX_OVERLAP), mode=GTEX_PROV["mode"]),
    n_genes=int(BULK.shape[0]), n_panel=len(panel),
    n_edges_undirected=N_EDGES_UND, n_folds=len(folds),
    t1_cohorts=t1c, t2_cohorts=t2c, external=["GSE163211"],
    task_primary=TASK, calibration_mode=CAL_MODE,
    features=N_FEAT,
    duplicate_biopsies_dropped=N_DUP_BIOPSY,
    gse213621_annotation=G213,
    head3=json.loads((PATHS["config"]/"head3.json").read_text()),
    fold_label_shift=fd[["fold","test_cohort","JS_test_vs_cal",
                         "JS_test_vs_train"]].to_dict("records"),
    notes=[
        "Group C must be recomputed inside each fold; never precompute.",
        "Augmentation: proper_train only. Never calibration, never test.",
        "Select task rows with eligible_T1 / eligible_T2 / eligible_T3; "
        "never turn a missing target into class 0.",
        "stage holds a representative value for grouped records; use exact_stage "
        "for biopsy-confirmed stages.",
        "geo_control_reference = in-study GEO controls; the GTEx features use GTEx v8 "
        "liver (gtex_reference_provenance.json). They are different populations.",
        "GSE163211 (NanoString) was processed separately; it is not model-ready data "
        "in this handoff.",
        "Head 3 is a 3-arm ablation, not a fixed setting.",
    ])
(EXPORT/"handoff.json").write_text(json.dumps(handoff, indent=2, default=_json_default))
print(f"\nhandoff.json written")
print(f"\nPREPROCESSING COMPLETE — {EXPORT}")

DATASET SUMMARY
  cohorts with expression : 12
  T1 (LODO folds)         : 8  ['GSE130970', 'GSE135251', 'GSE162694', 'GSE193066', 'GSE213621', 'GSE240729', 'GSE31803', 'GSE84044']
  T2 (training-only)      : 4  ['GSE207310', 'GSE276114', 'GSE48452', 'GSE89632']
  T4 (external)           : ['GSE163211']  NanoString; processed separately, not in this handoff
  supervised records      : 1366
  eligible T1 / T2 / T3   : 1327 / 1063 / 1366
  T1-ineligible records   : [{'tier': 'T2', 'cohort': 'GSE276114', 'resolution': 'grouped_F0_F2', 'n': 39}]
  preservation unknown    : 136  {'GSE193066': 106, 'GSE207310': 30}
  genes (common space)    : 10732
  panel genes             : 72
  STRING edges            : 321 undirected / 642 directed, score>=700
  node features           : 17 precomputed + 9 fold-fitted = 26
  platforms               : RNA-seq, microarray (NanoString external, separate)
  etiologies              : ['HBV', 'MASLD', 'MASLD_mixed', 'NAFLD']

TABLE I (for the paper)  F0-F4 = E

In [67]:
# ============================================================
# CELL F1 — FOOTPRINT-BASED PATHWAY ACTIVITY
#
# Problem this solves: TGF-beta is controlled by activation of a
# latent complex, not by transcription, so TGFB1 mRNA lags its
# activity. Verified on this dataset: TGFB1 mRNA first differs
# from F0 at F2, while its 11 canonical targets differ at F1.
#
# PROGENy (Schubert et al., Nat Commun 2018) infers pathway
# activity from the downstream transcriptional footprint.
# This is SAMPLE-LEVEL. It is NOT added as a node feature —
# broadcasting one scalar across 72 nodes was the Phase 2 bug.
# ============================================================
!pip install -q decoupler

import decoupler as dc
import pandas as pd, numpy as np
from scipy.stats import mannwhitneyu, spearmanr

B   = pd.read_parquet(PATHS["processed"]/"bulk_expression.parquet")
lab = pd.read_parquet(PATHS["processed"]/"labels.parquet")
print(f"expression {B.shape}   labels {len(lab)}")

# ---------- 1. get the PROGENy model ----------
FALLBACK = {   # canonical direct targets; used ONLY if download fails
 "TGFb": ["SERPINE1","SMAD7","SKIL","JUNB","PMEPA1","TGFBI","ID1","JUN",
          "KLF10","THBS1","CDKN1A"],
 "NFkB": ["NFKBIA","TNFAIP3","CCL2","CXCL8","ICAM1","BIRC3","SOD2","IER3"],
 "JAK-STAT": ["STAT1","IRF1","SOCS1","SOCS3","CXCL10","GBP1","MX1","OAS1"],
 "TNFa": ["TNFAIP3","NFKBIA","CXCL2","ICAM1","CCL20","SOD2"],
 "p53": ["CDKN1A","MDM2","GADD45A","BAX","SESN1","RRM2B","TP53I3"],
 "Hypoxia": ["VEGFA","SLC2A1","PGK1","LDHA","CA9","ADM","BNIP3"],
}

net = dc.op.progeny(organism="human", top=500)
source = "PROGENy (decoupler)"

if not {"source", "target", "weight"}.issubset(net.columns):
    raise ValueError("PROGENy network lacks required columns")
# coverage against your gene space
cov = net.groupby("source").target.apply(lambda s: s.isin(B.index).mean())
n_in = net.groupby("source").target.apply(lambda s: s.isin(B.index).sum())
print("\nfootprint coverage in the 10,735-gene matrix:")
for p in cov.index:
    print(f"  {p:14s} {int(n_in[p]):4d} genes  {cov[p]:5.1%}")
USABLE_PATHWAYS = [p for p in cov.index if n_in[p] >= 10]
print(f"\npathways with >=10 footprint genes: {USABLE_PATHWAYS}")

# ---------- 2. within-cohort z-score (no labels used) ----------
ex = lab[lab.resolution.isin(["exact","exact_ishak_mapped"])].copy()
ex["s"] = ex.stage.str[1].astype(int)
ids = [i for i in ex.index if i in B.columns]
ex = ex.loc[ids]
print(f"\nexact-stage samples: {len(ex)}")
print(ex.s.value_counts().sort_index().to_string())

Z = B[ids].copy()
for c in ex.cohort.unique():
    cc  = [i for i in ids if ex.loc[i,"cohort"] == c]
    blk = Z[cc]
    Z[cc] = blk.sub(blk.mean(axis=1), axis=0).div(
              blk.std(axis=1).replace(0, np.nan), axis=0)
Z = Z.fillna(0.0)

# ---------- 3. pathway activity, samples x pathways ----------
mat = Z.T
net_u = net[net.source.isin(USABLE_PATHWAYS)].copy()

if not USABLE_PATHWAYS or net_u.empty:
    raise ValueError("No pathways meet the footprint coverage requirement")

ACT, ACT_PVALUES = dc.mt.mlm(
    data=mat,
    net=net_u,
    tmin=5,
    verbose=False,
)

if not isinstance(ACT, pd.DataFrame):
    raise TypeError("MLM did not return a score DataFrame")

if not ACT.index.equals(mat.index):
    raise ValueError("MLM output sample order differs from its input")

if set(ACT.columns) != set(USABLE_PATHWAYS):
    raise ValueError("MLM output pathways differ from the requested pathways")

ACT = ACT.loc[:, USABLE_PATHWAYS]

if not np.isfinite(ACT.to_numpy()).all():
    raise ValueError("MLM returned non-finite pathway scores")

method = "PROGENy MLM"
print(f"activity matrix {ACT.shape} via {method}")

# ---------- 4. onset: footprint vs ligand mRNA ----------
def onset(v, name, base_idx, ex):
    row, first = [], None
    for st in [1,2,3,4]:
        x = v[ex.index[ex.s == st]]
        d = x.mean() - v[base_idx].mean()
        p = mannwhitneyu(x, v[base_idx]).pvalue
        row.append((d, p))
        if first is None and p < 0.01 and abs(d) > 0.15:
            first = st
    rho, prho = spearmanr(ex.s, v[ex.index])
    print(f"  {name:30s} " + " ".join(f"{d:+6.2f}" for d,_ in row) +
          f"   first={'F'+str(first) if first else 'ns':3s}  rho={rho:+.3f}")
    return dict(signal=name, first_stage=first, rho=rho, p_rho=prho,
                **{f"d_F{st}": d for st,(d,_) in zip([1,2,3,4], row)},
                **{f"p_F{st}": p for st,(_,p) in zip([1,2,3,4], row)})

base = ex.index[ex.s == 0]
LIGAND = {"TGFb":"TGFB1", "TNFa":"TNF", "NFkB":"NFKB1", "JAK-STAT":"STAT1",
          "p53":"TP53", "Hypoxia":"HIF1A"}

print(f"\n{'':30s} {'F1':>6s} {'F2':>6s} {'F3':>6s} {'F4':>6s}")
rows = []
for p in USABLE_PATHWAYS:
    rows.append(onset(ACT[p], f"{p} FOOTPRINT", base, ex))
    g = LIGAND.get(p)
    if g and g in Z.index:
        rows.append(onset(Z.loc[g], f"  {g} mRNA (ligand/TF)", base, ex))

res = pd.DataFrame(rows)
res.to_csv(PATHS["reports"]/"footprint_onset.csv", index=False)
ACT.to_parquet(PATHS["processed"]/"pathway_activity.parquet")

# ---------- 5. the headline comparison ----------
print("\n" + "="*60)
print("FOOTPRINT vs LIGAND mRNA — earliest stage of significant change")
for p in USABLE_PATHWAYS:
    g = LIGAND.get(p)

    if not (g and g in Z.index):
        continue

    fp = res.loc[
        res.signal == f"{p} FOOTPRINT", "first_stage"
    ].iloc[0]

    lg = res.loc[
        res.signal == f"  {g} mRNA (ligand/TF)", "first_stage"
    ].iloc[0]

    fp_exists = pd.notna(fp)
    lg_exists = pd.notna(lg)

    verdict = (
        "footprint EARLIER"
        if fp_exists and (not lg_exists or fp < lg)
        else "no advantage"
    )

    fp_text = f"F{int(fp)}" if fp_exists else "ns"
    lg_text = f"F{int(lg)}" if lg_exists else "ns"

    print(
        f"  {p:12s} footprint={fp_text:4s} "
        f"{g}={lg_text:4s}   {verdict}"
    )
print("\nExpected from the pre-check: TGFb footprint F1 vs TGFB1 mRNA F2.")
print("If PROGENy's weighted 500-gene set does not reproduce that, report it —")
print("a curated 11-gene set beating a fitted model would itself be worth noting.")

write_manifest("f01_footprint",
               inputs={"bulk":"bulk_expression.parquet","net_source":source},
               params={"method":method, "pathways":USABLE_PATHWAYS,
                       "n_samples":int(len(ex))},
               outputs=[PATHS["processed"]/"pathway_activity.parquet",
                        PATHS["reports"]/"footprint_onset.csv"])

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.4/147.4 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 6.7 MB/s eta 0:00:00
expression (10732, 1366)   labels 1366

footprint coverage in the 10,735-gene matrix:
  Androgen        386 genes  77.4%
  EGFR            354 genes  70.8%
  Estrogen        383 genes  76.9%
  Hypoxia         387 genes  77.6%
  JAK-STAT        365 genes  73.1%
  MAPK            419 genes  84.1%
  NFkB            396 genes  79.2%
  PI3K            356 genes  71.5%
  TGFb            384 genes  76.8%
  TNFa            384 genes  76.8%
  Trail            72 genes  35.6%
  VEGF            283 genes  56.7%
  WNT             168 genes  61.5%
  p53             353 genes  70.9%

pathways with >=10 footprint genes: ['Androgen', 'EGFR', 'Estrogen', 'Hypoxia', 'JAK-STAT', 'MAPK', 'NFkB', 'PI3K', 'TGFb', 'TNFa', 'Trail', 'VEGF', 'WNT', 'p53']

exact-stage samples: 1063
s
0    221
1    218
2    300
3  

{'stage': 'f01_footprint',
 'timestamp': '2026-10-08T11:26:00',
 'inputs': {'bulk': 'bulk_expression.parquet',
  'net_source': 'PROGENy (decoupler)'},
 'params': {'method': 'PROGENy MLM',
  'pathways': ['Androgen',
   'EGFR',
   'Estrogen',
   'Hypoxia',
   'JAK-STAT',
   'MAPK',
   'NFkB',
   'PI3K',
   'TGFb',
   'TNFa',
   'Trail',
   'VEGF',
   'WNT',
   'p53'],
  'n_samples': 1063},
 'outputs': {'/content/drive/MyDrive/CoDAGAT_preprocessing_fix_v1/processed/pathway_activity.parquet': '63453448406e8dcec9922c60ea59f1707a3ca0c3504511fd34f185718932d638',
  '/content/drive/MyDrive/CoDAGAT_preprocessing_fix_v1/reports/footprint_onset.csv': 'd1cb52674bf2ba768320495691ad9e1f2039c19417ecc0e4dc9bbc5908065132'}}

In [68]:
# ============================================================
# CELL 33b — CELL 34 PREREQUISITES   [v2]
# META_COLS was lost when the shared-assets loop was edited.
# v2: y.csv also carries exact_stage + eligibility masks, and the
#     check no longer requires PKG (Cell 34 creates it).
# ============================================================
assert "add_export_label_columns" in globals(), "run Cell 33 (v2) first"

META_COLS = ["cohort","stage","resolution","tier","etiology","platform",
             "y_T1_significant","y_T2_ordinal","y_T3_early_adv",
             "exact_stage","eligible_T1","eligible_T2","eligible_T3"]

sup = add_export_label_columns(sup)      # idempotent; targets are not touched
check_export_labels(sup, "sup")

missing = [c for c in META_COLS if c not in sup.columns]
assert not missing, f"columns missing from sup: {missing}"

_need = {"folds": "Cell 32", "TENSOR_AB": "Cell 30", "z": "Cell 30",
         "sup": "Cell 32", "panel": "Cell 29", "PATHWAYS": "Cell 29",
         "build_fold_tensor": "Cell 32", "compute_group_C": "Cell 32",
         "assert_tensor_integrity": "Cell 4", "assert_no_synthetic": "Cell 4"}
for n, src in _need.items():
    print(f"  {'OK     ' if n in globals() else 'MISSING'} {n:24s} ({src})")
_m = [n for n in _need if n not in globals()]
assert not _m, f"missing {_m} — run the cells listed above first"

_ho = PATHS["processed"]/"handoff.json"
assert _ho.exists() and "n_eligible" in json.loads(_ho.read_text()), \
    "processed/handoff.json is from the old Cell 33 — run Cell 33 (v2) first"
print(f"\nMETA_COLS defined ({len(META_COLS)} cols). Cell 34 can run.")

  OK      folds                    (Cell 32)
  OK      TENSOR_AB                (Cell 30)
  OK      z                        (Cell 30)
  OK      sup                      (Cell 32)
  OK      panel                    (Cell 29)
  OK      PATHWAYS                 (Cell 29)
  OK      build_fold_tensor        (Cell 32)
  OK      compute_group_C          (Cell 32)
  OK      assert_tensor_integrity  (Cell 4)
  OK      assert_no_synthetic      (Cell 4)

META_COLS defined (13 cols). Cell 34 can run.


In [69]:
# ============================================================
# CELL 34 — HANDOFF ZIP   [v2]
# Structure: every fold has its own train/ calibration/ test/
# folders under folds/. Those eight folders are the evaluation data.
# v2: README is generated from the exported files; split folders are
#     described accurately (real + unaugmented, features transformed);
#     GEO controls and GTEx are documented as separate references;
#     external/README corrected (40/72); the top-level flat copy of
#     fold 0 is optional and OFF by default.
# ============================================================
import zipfile

WRITE_FLAT_VIEW = False   # True: also write the byte-identical fold-0 copy at top level
DEFAULT_FOLD    = 0       # which fold the optional flat copy uses

PKG = PATHS["root"] / "handoff"
if PKG.exists():
    shutil.rmtree(PKG)
PKG.mkdir(parents=True)

# ---------------- shared assets ----------------
SHARED_REQUIRED = [
    "bulk_expression.parquet", "bulk_expression_nocombat.parquet",
    "labels.parquet", "deviation_targets.parquet", "panel_genes.csv",
    "graph_edges.csv", "graph_topology.csv",
    "geo_control_reference.parquet", "geo_control_reference_samples.csv",
    "geo_control_reference.json",
    "gtex_reference_provenance.json", "gtex_overlap_genes.csv",
    "gtex_reference_samples.csv",
    "feature_schema_AB.csv", "feature_schema_C.csv",
    "fold_assignment_lodo.csv", "fold_sample_splits.csv",
    "tensor_sample_order.csv", "handoff.json"]
SHARED_OPTIONAL = ["pathway_activity.parquet"]

(PKG / "shared").mkdir()
_miss = [f for f in SHARED_REQUIRED if not (PATHS["processed"] / f).exists()]
assert not _miss, f"missing in processed/: {_miss} — run Cell 33 (v2) first"
for f in SHARED_REQUIRED + SHARED_OPTIONAL:
    p = PATHS["processed"] / f
    if p.exists():
        shutil.copy(p, PKG / "shared" / f)
    else:
        print(f"  skipped (optional, not found): {f}")

fp = PATHS["reports"] / "footprint_onset.csv"
if fp.exists():
    shutil.copy(fp, PKG / "shared" / "footprint_onset.csv")
else:
    print("  skipped: footprint_onset.csv (run Cell F1 first)")

print("shared/:", sorted(x.name for x in (PKG / "shared").iterdir()))

# ---------------- facts, read back from the exported files ----------------
H   = json.loads((PKG / "shared" / "handoff.json").read_text())
LAB = pd.read_parquet(PKG / "shared" / "labels.parquet")
GEO = json.loads((PKG / "shared" / "geo_control_reference.json").read_text())
GTX = json.loads((PKG / "shared" / "gtex_reference_provenance.json").read_text())
SCH = pd.concat([pd.read_csv(PKG / "shared" / "feature_schema_AB.csv"),
                 pd.read_csv(PKG / "shared" / "feature_schema_C.csv")], ignore_index=True)
N_EDGES_DIR = len(pd.read_csv(PKG / "shared" / "graph_edges.csv"))

check_export_labels(LAB, "shared/labels.parquet")
assert set(LAB.index) == set(sup.index), "labels.parquet and sup hold different samples"
for c in ["eligible_T1", "eligible_T2", "eligible_T3"]:
    assert (LAB[c].reindex(sup.index) == sup[c]).all(), f"{c}: labels.parquet != sup"
assert (LAB.exact_stage.reindex(sup.index).fillna("") == sup.exact_stage.fillna("")).all()

def write_split(dest, ids, arr, fold_id, split_name):
    dest.mkdir(parents=True, exist_ok=True)
    assert_tensor_integrity(arr, f"fold{fold_id}_{split_name}")
    if split_name in ("calibration", "test"):
        assert_no_synthetic(sup.loc[ids], split_name, fold_id)
    assert sup.loc[ids, "eligible_T1"].all(), f"fold{fold_id} {split_name}: ineligible row"
    np.save(dest / "X.npy", arr)
    m = sup.loc[ids, META_COLS].copy(); m.index.name = "sample_id"
    m.to_csv(dest / "y.csv")
    pd.Series(ids, name="sample_id").to_csv(dest / "sample_ids.csv", index=False)
    return arr.shape

SPLIT_RULES = [
    "augmentation applies to train/ ONLY",
    "calibration/ and test/ hold real, unaugmented samples and must stay that way",
    "X.npy holds transformed features, not raw expression (README: SPLIT CONTENTS)",
    "features 17-25 (Group C) were fitted on THIS fold's train/ labels only",
    "select task rows with eligible_T1 / eligible_T2 / eligible_T3; never fill a "
    "missing target with 0",
]

# ---------------- per-fold ----------------
print("BUILDING FOLD PACKAGES")
manifest = []
for f in folds:
    k, tc = f["fold"], f["test_cohort"]
    base = PKG / "folds" / f"fold{k}_{tc}"
    t = build_fold_tensor(f, TENSOR_AB, z, sup, panel, PATHWAYS)

    shapes = {}
    for folder, key, ids in [("train", "proper_train", f["train_idx"]),
                             ("calibration", "calibration", f["cal_idx"]),
                             ("test", "test", f["test_idx"])]:
        shapes[folder] = list(write_split(base / folder, ids, t[key], k, folder))

    (base / "fold_info.json").write_text(json.dumps(dict(
        fold=k, test_cohort=tc, calibration_source=f["calibration"],
        shapes=shapes,
        positive_rate=dict(train=f["train_pos"], calibration=f["cal_pos"],
                           test=f["test_pos"]),
        n_eligible_T2=dict(
            train=int(sup.loc[f["train_idx"], "eligible_T2"].sum()),
            calibration=int(sup.loc[f["cal_idx"], "eligible_T2"].sum()),
            test=int(sup.loc[f["test_idx"], "eligible_T2"].sum())),
        train_cohorts=sorted(sup.loc[f["train_idx"], "cohort"].unique().tolist()),
        rules=SPLIT_RULES,
    ), indent=2))

    manifest.append(dict(fold=k, test_cohort=tc, calibration=f["calibration"],
                         train=f["n_train"], cal=f["n_cal"], test=f["n_test"],
                         X_shape=tuple(shapes["train"][1:])))
    print(f"  fold{k}_{tc:12s} train={f['n_train']:5d} "
          f"cal={f['n_cal']:4d} test={f['n_test']:4d}")
MAN = pd.DataFrame(manifest)
assert MAN.X_shape.nunique() == 1, "folds disagree on (nodes, features)"
N_NODES, N_FEATS = MAN.X_shape.iloc[0]

# ---------------- optional flat top-level view ----------------
if WRITE_FLAT_VIEW:
    fd0 = folds[DEFAULT_FOLD]
    src = PKG / "folds" / f"fold{fd0['fold']}_{fd0['test_cohort']}"
    for folder in ["train", "calibration", "test"]:
        shutil.copytree(src / folder, PKG / folder)
    (PKG / "FLAT_VIEW_WARNING.txt").write_text(
        f"The top-level train/ calibration/ test/ folders are a byte-identical COPY of\n"
        f"folds/fold{fd0['fold']}_{fd0['test_cohort']}/ and exist only for quick\n"
        f"inspection.\n\n"
        f"They are NOT the evaluation protocol. Evaluation is leave-one-dataset-out\n"
        f"across all {len(folds)} folds in folds/. Reporting results from this single\n"
        f"split alone would collapse {len(folds)} independent measurements into one and\n"
        f"invalidate the label-shift analysis.\n")

# ---------------- external ----------------
(PKG / "external").mkdir()
(PKG / "external" / "README.txt").write_text(
    "GSE163211 (NanoString, 318 samples) is tier T4 (external).\n\n"
    "It was processed SEPARATELY, outside this notebook: RCC parsing, positive-control\n"
    "and housekeeping normalisation, the 40/72 panel-gene subset, and a 40-node\n"
    "STRING subgraph. (Earlier project documents stated 43/72; the correct figure\n"
    "is 40/72.)\n\n"
    "It is NOT included in this handoff as model-ready data: no X.npy, y.csv or fold\n"
    "split here contains it, and the 72-node models trained on folds/ cannot be\n"
    "applied to it without a 40-node variant.\n\n"
    "It CANNOT validate the F2+ boundary: the NASH_F1_F4 group (82 samples)\n"
    "straddles the decision boundary. Legitimate uses are a specificity probe\n"
    "on the 236 confirmed non-fibrotic samples, coarse fibrotic detection, and\n"
    "cross-platform conformal coverage.\n")

# ---------------- README (every number computed above) ----------------
N   = len(LAB)
T1L = LAB[LAB.tier == "T1"]; T2L = LAB[LAB.tier == "T2"]
INEL = LAB[~LAB.eligible_T1]
f_idx = dict(zip(SCH.feature, SCH.idx))
gtex_feats = ", ".join(f"{n} ({f_idx[n]})" for n in
                       ["deviation_rank_z", "gtex_expression_level", "gtex_expression_std"]
                       if n in f_idx)
c_lo, c_hi = int(SCH.loc[SCH.group == "C", "idx"].min()), int(SCH.loc[SCH.group == "C", "idx"].max())
ab_hi = int(SCH.loc[SCH.group.isin(["A", "B"]), "idx"].max())
pres = LAB.preservation.value_counts()
res_ = LAB.resolution.value_counts()
n_tr_lo, n_tr_hi = int(MAN.train.min()), int(MAN.train.max())
f0 = MAN.iloc[0]

def _rows(s, w=14):
    return "\n".join(f"      {str(k):{w}s} {int(v):5d}" for k, v in s.items())

inel_tab = INEL.groupby(["tier", "cohort", "resolution"]).size()
inel_lines = "\n".join(f"      {t_:3s} {c_:10s} {r_:14s} {int(n_):4d}"
                       for (t_, c_, r_), n_ in inel_tab.items())
fold_lines = "\n".join(f"  fold{r.fold}_{r.test_cohort:10s} cal={r.calibration:18s} "
                       f"train={r.train:5d}  cal={r.cal:4d}  test={r.test:4d}"
                       for r in MAN.itertuples())
_fmt = lambda d: ", ".join(f"{k} {v}" for k, v in d.items())
pres_unknown = _fmt(H["preservation_unknown_by_cohort"])
cal_text = ("one other whole T1 cohort (see FOLDS)" if H["calibration_mode"] == "V1"
            else "a stratified random subset of the other T1 cohorts (see FOLDS)")
g213 = H.get("gse213621_annotation")

L = []
L += ["CoDA-GAT PREPROCESSED DATASET",
      f"generated {time.strftime('%Y-%m-%d %H:%M')}",
      "Every number below was computed from the files in this package when it was built.",
      "",
      "LAYOUT",
      "  shared/                    assets identical across folds",
      f"  folds/foldN_<cohort>/      train/ calibration/ test/   <- the evaluation data "
      f"({len(folds)} folds)"]
if WRITE_FLAT_VIEW:
    L += [f"  train/ calibration/ test/  byte-identical copy of fold {DEFAULT_FOLD} only "
          f"(see FLAT_VIEW_WARNING.txt)"]
L += ["  external/                  GSE163211 (NanoString) notes; not model-ready here",
      "",
      "Each split folder contains:",
      f"  X.npy           (n_samples, {N_NODES}, {N_FEATS}) float32 node-feature tensor",
      "  y.csv           labels, eligibility masks and metadata, indexed by sample_id",
      "  sample_ids.csv  row order of X.npy",
      "",
      "NUMBERS",
      f"  records in shared/labels.parquet   {N}   (all real; zero synthetic)",
      f"  cohorts                            {LAB.cohort.nunique()}",
      f"  T1 cohorts (one LODO fold each)    {T1L.cohort.nunique()} cohorts, {len(T1L)} records,"
      f" {int(T1L.eligible_T1.sum())} eligible for y_T1_significant",
      f"  T2 cohorts (training-only)         {T2L.cohort.nunique()} cohorts, {len(T2L)} records"
      f" retained, {int(T2L.eligible_T1.sum())} eligible for primary training",
      f"  records in folds/ splits           {int(LAB.eligible_T1.sum())}"
      f"  (= every eligible_T1 record)",
      "  T4 external                        GSE163211 (NanoString; processed separately,"
      " not in this handoff)",
      f"  GEO control reference              {GEO['n_samples']}  (separate file; not in N)",
      f"  genes                              {H['n_genes']}",
      f"  graph                              {H['n_panel']} nodes, {N_EDGES_DIR // 2} undirected"
      f" STRING edges (score >= {STRING_THRESHOLD})",
      f"  node features                      {H['features']['total']} = {H['features']['A']}"
      f" dynamic + {H['features']['B']} static + {H['features']['C']} fold-fitted",
      "",
      "RECORDS WITHOUT A PRIMARY (F2+) TARGET",
      f"  {len(INEL)} records have y_T1_significant missing:",
      inel_lines,
      "  grouped_F0_F2 means the biopsy was reported only as the range F0-F2. That range",
      "  straddles the F2 boundary, so F2+ vs F0/F1 is undefined; the ordinal stage is",
      "  undefined too. y_T3_early_adv (F3/F4 vs F0-F2) IS defined, so these records are",
      "  kept in shared/labels.parquet and shared/bulk_expression*.parquet. They are in",
      "  no folds/ split, because every split is restricted to eligible_T1 records.",
      "",
      "TASKS AND ELIGIBILITY",
      "  task                  target            mask          eligible",
      f"  primary  F2+ vs F0/F1 y_T1_significant  eligible_T1   {int(LAB.eligible_T1.sum()):5d}",
      f"  ordinal  F0<..<F4     y_T2_ordinal      eligible_T2   {int(LAB.eligible_T2.sum()):5d}",
      f"  early vs advanced     y_T3_early_adv    eligible_T3   {int(LAB.eligible_T3.sum()):5d}",
      "  A missing target means 'not defined for this record'. Never fill it with 0;",
      "  select rows with the mask. Inside folds/, y_T2_ordinal is missing for",
      "  grouped-label records: use eligible_T2.",
      "",
      "STAGE COLUMNS",
      "  exact_stage  biopsy-reported exact stage; null whenever resolution != 'exact'.",
      "  stage        kept for backward compatibility. For grouped records it holds a",
      "               REPRESENTATIVE value (the lower bound of the reported range):",
      "               grouped_F0_F1 -> F0, grouped_F0_F2 -> F0, grouped_F3_F4 -> F3.",
      "               Never use stage as a biopsy-confirmed stage.",
      "  resolution counts:",
      _rows(res_),
      "",
      "SPLIT CONTENTS",
      "  train/        proper-train: the remaining T1 cohorts plus all eligible T2 records.",
      "                The ONLY split where synthetic augmentation may be added.",
      f"  calibration/  {cal_text}.",
      "                Real, unaugmented samples.",
      "  test/         every eligible_T1 record of the held-out T1 cohort. Real,",
      "                unaugmented samples. That cohort's control samples are not in",
      "                test/ (they are in shared/geo_control_reference.parquet).",
      "  calibration/ and test/ hold REAL, UNAUGMENTED samples, but X.npy contains",
      "  TRANSFORMED features, not raw expression:",
      f"    - features {c_lo}-{c_hi} (Group C) are fitted on train/ labels only;"
      " calibration/ and",
      "      test/ labels were excluded from fitting them.",
      f"    - features 0-{ab_hi} (Groups A and B) use no labels, but are not fitted per fold.",
      "      ComBat (batch = cohort, no label covariates) and the within-cohort scaling",
      "      behind expression_z and the pathway-activity features used UNLABELLED",
      "      expression from every cohort, including the held-out calibration and test",
      "      cohorts; expression_variability and cross_platform_consistency use the",
      f"      unlabelled expression of all {N} records. This transductive preprocessing",
      "      is disclosed in the thesis.",
      "",
      "FOLDS",
      fold_lines,
      "",
      "REFERENCE POPULATIONS (two different ones)",
      "  shared/geo_control_reference.parquet   (formerly shared/healthy_reference.parquet)",
      f"    {GEO['n_samples']} in-study GEO control / normal-histology biopsies:"
      f" {_fmt(GEO['by_cohort'])}.",
      "    NOT healthy donor liver and NOT GTEx. Same genes and integrated scale as",
      "    bulk_expression.parquet. It is not a reference for any feature in X.npy.",
      "    Metadata: geo_control_reference_samples.csv, geo_control_reference.json.",
      f"  GTEx v8 liver (not shipped): {GTX['n_samples']} samples from {GTX['n_donors']} donors.",
      f"    The reference for {gtex_feats}",
      "    and for deviation_targets.parquet.",
      f"    Gene overlap with bulk_expression: {GTX['overlap_with_bulk_expression']['n_genes']}"
      f" of {GTX['overlap_with_bulk_expression']['of_bulk_genes']}.",
      "    Exact input files + SHA-256 + processing: gtex_reference_provenance.json;",
      "    sample IDs: gtex_reference_samples.csv; genes: gtex_overlap_genes.csv.",
      "",
      "PRESERVATION",
      _rows(pres),
      f"  'unknown' = not recorded in the cohort registry ({pres_unknown}).",
      "  It is NOT inferred from any other field.",
      "",
      "LOADING",
      "  import numpy as np, pandas as pd",
      f"  X = np.load('folds/fold{f0.fold}_{f0.test_cohort}/train/X.npy')"
      f"   # ({f0.train}, {N_NODES}, {N_FEATS})",
      f"  y = pd.read_csv('folds/fold{f0.fold}_{f0.test_cohort}/train/y.csv', index_col=0)",
      "  E = pd.read_csv('shared/graph_edges.csv')            # src_idx, dst_idx, weight",
      "  lab = pd.read_parquet('shared/labels.parquet')",
      "  primary = lab[lab.eligible_T1]                        # never lab.fillna(0)",
      "",
      "BINDING RULES",
      "  1. Synthetic augmentation goes in train/ ONLY. A synthetic sample in",
      "     calibration/ destroys the conformal coverage guarantee outright.",
      "  2. calibration/ and test/ stay real and unaugmented. test/ is one held-out cohort.",
      "  3. Synthesis is within-cohort and within-class, or the cohort/stage label",
      "     of the synthetic sample is undefined.",
      f"  4. Features {c_lo}-{c_hi} are already fitted per fold. Do not refit globally.",
      "  5. Select task rows with the eligible_* masks; never turn a missing target into 0.",
      "  6. Use exact_stage, not stage, wherever a biopsy-confirmed stage is needed.",
      "  Guards are in the notebook (Cell 4). Do not add a bypass flag.",
      "",
      "AUGMENTATION MULTIPLIER",
      "  m is an ablation variable, not a setting. Sweep {0, 1, 3, 7} and report all",
      f"  four rows. train/ holds {n_tr_lo:,}-{n_tr_hi:,} real rows per fold; with m synthetic",
      f"  rows per real row, m=7 gives at most {8*n_tr_lo:,}-{8*n_tr_hi:,} rows per fold",
      "  (fewer if a validation subset is held out of train/).",
      f"  N_real stays {N} - that is the dataset size for the paper.",
      "",
      "KNOWN ISSUES FOR THE PAPER"]
if g213:
    L += [f"  - GSE213621 lacks {', '.join(g213['missing_canary'])} (annotation gap; it retains",
          f"    {g213['top500_retained']}/500 of the top-expressed genes of"
          f" {g213['top500_reference']}, so not abundance-filtered)."]
else:   # per-cohort matrices unavailable: keep the Cell 26 finding, cited as such
    L += ["  - GSE213621 lacks ALB/APOA1/APOB/FGA/FGB (annotation gap; Cell 26 diagnostic",
          "    of the original run: 373/500 top-expressed genes retained, so not",
          "    abundance-filtered)."]
L += ["  - Grouped labels: "
      + ", ".join(f"{k} {int(v)}" for k, v in res_.items() if k.startswith("grouped"))
      + ". Ordinal task: exact stages only.",
            "  - GSE193066 is PLS high-risk enriched; "
      + (f"{H['duplicate_biopsies_dropped']} follow-up biopsies dropped"
         if H["duplicate_biopsies_dropped"] is not None
         else "58 follow-up biopsies dropped (saved Cell 11 output)")
      + " to",
      "    prevent patient-level leakage.",
      "  - Head 3 is a 3-arm ablation (off / directional / directional_ps).",
      "  - Both ComBat and no-ComBat expression matrices are provided.",
      ""]
README = "\n".join(L)
assert "43/72" not in README and "untouched" not in README
(PKG / "README.txt").write_text(README)

# ---------------- zip ----------------
zip_path = PATHS["root"] / "CoDAGAT_preprocessed_handoff.zip"
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for p in sorted(PKG.rglob("*")):
        if p.is_file():
            zf.write(p, p.relative_to(PKG))

with zipfile.ZipFile(zip_path) as zf:
    names = zf.namelist()
for k in range(len(folds)):
    got = [n for n in names if n.startswith(f"folds/fold{k}_")]
    assert len(got) == 10, f"fold {k}: {len(got)} files, expected 10"
assert "README.txt" in names and "external/README.txt" in names
assert all(f"shared/{f}" in names for f in SHARED_REQUIRED), "shared file missing from zip"
assert "shared/healthy_reference.parquet" not in names
assert WRITE_FLAT_VIEW or not any(n.startswith(("train/", "calibration/", "test/"))
                                  for n in names)

print(f"\nZIP: {zip_path}")
print(f"  {zip_path.stat().st_size/1e6:.1f} MB, {len(names)} files")
print("\nPER-FOLD SUMMARY")
print(MAN.drop(columns="X_shape").to_string(index=False))
print(f"\ntotal train rows across folds : {int(MAN.train.sum())}")
print(f"total test rows across folds  : {int(MAN.test.sum())} "
      f"(each T1 sample is tested exactly once)")
print(f"\nHANDOFF READY -> {zip_path}")

shared/: ['bulk_expression.parquet', 'bulk_expression_nocombat.parquet', 'deviation_targets.parquet', 'feature_schema_AB.csv', 'feature_schema_C.csv', 'fold_assignment_lodo.csv', 'fold_sample_splits.csv', 'footprint_onset.csv', 'geo_control_reference.json', 'geo_control_reference.parquet', 'geo_control_reference_samples.csv', 'graph_edges.csv', 'graph_topology.csv', 'gtex_overlap_genes.csv', 'gtex_reference_provenance.json', 'gtex_reference_samples.csv', 'handoff.json', 'labels.parquet', 'panel_genes.csv', 'pathway_activity.parquet', 'tensor_sample_order.csv']
BUILDING FOLD PACKAGES
  fold0_GSE130970    train= 1043 cal= 206 test=  78
  fold1_GSE135251    train= 1009 cal= 112 test= 206
  fold2_GSE162694    train= 1109 cal= 106 test= 112
  fold3_GSE193066    train=  922 cal= 299 test= 106
  fold4_GSE213621    train=  961 cal=  67 test= 299
  fold5_GSE240729    train= 1188 cal=  72 test=  67
  fold6_GSE31803     train= 1131 cal= 124 test=  72
  fold7_GSE84044     train= 1125 cal=  78 test